# RAFA Paper Reproduction Notebook

This notebook is a cleaned reproduction notebook for the NoF 2026 camera-ready paper on Reconstruction-Aware Federated Aggregation (RAFA).

It is derived from the development notebook, but it removes unrelated diagnostics and preserves the experiment settings needed to reproduce the paper tables. The original development notebook should stay in `notebooks/archive/RAFA_development.ipynb`.

Important reproduction details:

1. The CICIoT2023 experiments use the default RAFA setting `alpha=15`, `beta=0`, `tau=0.2`, and `m_rec=0.01`.
2. The final CICDDoS2019 table used source-specific selected configurations. These are documented in Section 7.4 of this notebook.
3. The local training loop uses `local_epochs=5`, but training is capped by `max_local_batches=20`. This is the actual setting used by the development notebook.
4. The FedREDefense-inspired comparison is an adapted proxy for update-space reconstruction behavior, not a full reproduction of the original FedREDefense method.

In [ ]:

# ======================== 0. Setup: Imports, Paths, and Configuration ========================

import os
import time
import copy
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix

import matplotlib.pyplot as plt
from IPython.display import display

try:
    import seaborn as sns
except Exception:
    sns = None


# -------------------------
# Reproducibility
# -------------------------

SEED = 42

def set_global_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # These settings reproduce the development notebook behavior.
    # Small GPU-level variations can still occur across machines.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_global_seed(SEED)


# -------------------------
# Device and paths
# -------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"
RESULTS_DIR = NOTEBOOK_DIR / "results"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# -------------------------
# Dataset registry
# -------------------------
#
# Expected local dataset layout:
#   data/CICIoT2023.csv
#   data/CICDDoS2019/MSSQL.csv
#   data/CICDDoS2019/DrDoS_DNS.csv
#
# The datasets are not redistributed with this notebook.

DATASETS = {
    "CICIoT2023": {
        "path": str(DATA_DIR / "CICIoT2023.csv"),
        "label_col": "Label",
        "forced_drop_cols": [],
        "expected_features": 46,
    },
}

ACTIVE_DATASETS = ["CICIoT2023"]


# -------------------------
# Main experiment configuration
# -------------------------

CONFIG = {
    # Reproducibility
    "seed": SEED,

    # Federated setting
    "n_clients": 10,
    "fed_rounds": 20,
    "local_epochs": 5,
    "batch_size": 256,
    "lr": 1e-3,

    # Server-side aggregation
    "server_lr": 1.0,
    "beta_momentum": 0.9,

    # Model
    "latent_dim": 16,
    "dropout": 0.2,

    # Default RAFA configuration used for CICIoT2023 experiments
    "alpha": 15.0,
    "beta": 0.0,
    "tau": 0.2,

    # RAFA scoring safeguards used in the development notebook
    "rec_margin": 0.01,
    "rec_clip": 2.0,
    "kl_floor": 0.001,
    "kl_margin": 0.02,
    "kl_clip": 2.0,

    # Reference set
    "default_reference_size": 500,

    # Attacks
    "byzantine_fractions": [0.1, 0.2, 0.3, 0.4],
    "noise_scale": 5.0,
    "sign_flip_eta": 5.0,
    "attack_scale": 1.0,
    "recon_inflation_scale": 1.0,
    "adaptive_attack_scale": 1.0,

    # Evaluation
    "threshold_percentile": 95,

    # Important reproduction detail:
    # local_epochs remains 5, but each client training run is capped at 20 local batches.
    "max_local_batches": 20,

    # Data cleaning
    "heavy_nan_threshold": 0.50,
    "drop_non_numeric_features": True,
    "drop_rows_after_column_cleanup": True,

    # Output
    "results_dir": str(RESULTS_DIR),
}


NAN_DROP_CONFIG = {
    "heavy_nan_threshold": CONFIG["heavy_nan_threshold"],
    "drop_non_numeric_features": CONFIG["drop_non_numeric_features"],
    "drop_rows_after_column_cleanup": CONFIG["drop_rows_after_column_cleanup"],
    "global_forced_drop_cols": [],
}


AGGREGATORS = [
    "RAFA",
    "FedAvg",
    "FedAvgM",
    "Krum",
    "TrimMean",
    "DnC",
    "FLTrust-AE",
]


# -------------------------
# Basic checks
# -------------------------

dataset_files_found = {
    name: Path(meta["path"]).exists()
    for name, meta in DATASETS.items()
}

failed_checks = []

if not ACTIVE_DATASETS:
    failed_checks.append("ACTIVE_DATASETS is empty")

for name in ACTIVE_DATASETS:
    if name not in DATASETS:
        failed_checks.append(f"Active dataset is not registered: {name}")
    elif not dataset_files_found.get(name, False):
        failed_checks.append(f"Active dataset file not found: {name}")

if CONFIG["n_clients"] <= 1:
    failed_checks.append("n_clients must be > 1")

if CONFIG["fed_rounds"] <= 0:
    failed_checks.append("fed_rounds must be > 0")

if CONFIG["batch_size"] <= 0:
    failed_checks.append("batch_size must be > 0")

if CONFIG["latent_dim"] <= 0:
    failed_checks.append("latent_dim must be > 0")

cell_status = "OK" if not failed_checks else "CHECK_REQUIRED"

print(f"[Setup] Status: {cell_status}")
print(f"Device      : {DEVICE.type} | CUDA available: {torch.cuda.is_available()}")
print(f"Data dir    : {DATA_DIR}")
print(f"Results dir : {RESULTS_DIR}")

print(f"\nDatasets registered: {len(DATASETS)}")
for name, found in dataset_files_found.items():
    status = "found" if found else "missing"
    print(f"  - {name}: {status} ({DATASETS[name]['path']})")

if failed_checks:
    print("\nChecks that need attention before running the experiments:")
    for item in failed_checks:
        print(f"  - {item}")


## 1. Data Loading

Place the datasets in the local `data/` directory before running the notebook.

Required files for the full reproduction:

```text
data/CICIoT2023.csv
data/CICDDoS2019/MSSQL.csv
data/CICDDoS2019/DrDoS_DNS.csv
```

In [ ]:
# ======================== CELL 1.1: Data Loading and Preprocessing ========================

def infer_binary_label(label_series):
    label_text = label_series.astype(str).str.lower()
    benign_mask = label_text.str.contains("benign|normal", regex=True, na=False)
    return (~benign_mask).astype(int)


def map_ciciot_attack_category(label):
    label = str(label).lower()

    if "benign" in label:
        return "Benign"
    if "ddos" in label:
        return "DDoS"
    if "dos" in label:
        return "DoS"
    if "mirai" in label:
        return "Mirai"
    if "recon" in label:
        return "Reconnaissance"
    if "spoof" in label:
        return "Spoofing"
    if "brute" in label or "password" in label:
        return "Brute Force"
    if "web" in label or "upload" in label or "xss" in label or "sql" in label:
        return "Web-based"

    return "Other"


def map_attack_category(dataset_name, label):
    if dataset_name == "CICIoT2023":
        return map_ciciot_attack_category(label)

    label = str(label)
    if label.lower() in {"benign", "normal"}:
        return "Benign"

    return "Other"


def clean_and_prepare_dataset(dataset_name, meta):
    path = Path(meta["path"])
    label_col = meta["label_col"]

    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {path}")

    print(f"\nLoading {dataset_name} from {path} ...", flush=True)

    # low_memory=True is more memory-friendly for large CSVs.
    # Mixed-type warnings are acceptable because non-numeric features are removed later.
    df_raw = pd.read_csv(path, low_memory=True)

    raw_rows = len(df_raw)
    print(f"  Raw rows loaded: {raw_rows:,}", flush=True)

    if label_col not in df_raw.columns:
        raise ValueError(f"Label column '{label_col}' not found in {dataset_name}")

    forced_drop_cols = list(meta.get("forced_drop_cols", []))
    forced_drop_cols += list(NAN_DROP_CONFIG.get("global_forced_drop_cols", []))
    forced_drop_cols = [c for c in forced_drop_cols if c in df_raw.columns and c != label_col]

    candidate_feature_cols = [
        c for c in df_raw.columns
        if c != label_col and c not in forced_drop_cols
    ]

    print(f"  Candidate feature columns: {len(candidate_feature_cols)}", flush=True)

    nan_ratio = df_raw[candidate_feature_cols].isna().mean()
    heavy_nan_cols = nan_ratio[
        nan_ratio > NAN_DROP_CONFIG["heavy_nan_threshold"]
    ].index.tolist()

    feature_candidates_after_nan = [
        c for c in candidate_feature_cols
        if c not in heavy_nan_cols
    ]

    non_numeric_cols = [
        c for c in feature_candidates_after_nan
        if not pd.api.types.is_numeric_dtype(df_raw[c])
    ]

    if NAN_DROP_CONFIG.get("drop_non_numeric_features", True):
        feature_cols = [
            c for c in feature_candidates_after_nan
            if c not in non_numeric_cols
        ]
    else:
        feature_cols = feature_candidates_after_nan

    excluded_cols = sorted(set(forced_drop_cols + heavy_nan_cols + non_numeric_cols))

    print(f"  Numeric feature columns kept: {len(feature_cols)}", flush=True)
    print(f"  Heavy-NaN cols dropped: {len(heavy_nan_cols)}", flush=True)
    print(f"  Non-numeric cols dropped: {len(non_numeric_cols)}", flush=True)

    keep_cols = feature_cols + [label_col]
    df = df_raw[keep_cols].copy()

    # Free memory from raw dataframe as early as possible
    del df_raw

    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if NAN_DROP_CONFIG.get("drop_rows_after_column_cleanup", True):
        before_row_drop = len(df)
        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna(subset=feature_cols + [label_col]).reset_index(drop=True)
        row_drops = before_row_drop - len(df)
    else:
        row_drops = 0

    clean_rows = len(df)
    print(f"  Clean rows after row cleanup: {clean_rows:,}", flush=True)

    df["binary_label"] = infer_binary_label(df[label_col])
    df["attack_category"] = df[label_col].apply(lambda x: map_attack_category(dataset_name, x))

    benign_df = df[df["binary_label"] == 0].copy().reset_index(drop=True)
    attack_df = df[df["binary_label"] == 1].copy().reset_index(drop=True)

    if len(benign_df) == 0:
        raise ValueError(f"No benign samples found in {dataset_name}")

    if len(attack_df) == 0:
        raise ValueError(f"No attack samples found in {dataset_name}")

    print(f"  Benign rows: {len(benign_df):,}", flush=True)
    print(f"  Attack rows: {len(attack_df):,}", flush=True)

    benign_train_pool, benign_temp = train_test_split(
        benign_df,
        test_size=0.30,
        random_state=CONFIG["seed"],
        shuffle=True,
    )

    benign_val, benign_test = train_test_split(
        benign_temp,
        test_size=0.50,
        random_state=CONFIG["seed"],
        shuffle=True,
    )

    benign_train_pool = benign_train_pool.reset_index(drop=True)
    benign_val = benign_val.reset_index(drop=True)
    benign_test = benign_test.reset_index(drop=True)

    ref_size = min(CONFIG["default_reference_size"], len(benign_train_pool))

    reference_df = benign_train_pool.sample(
        n=ref_size,
        random_state=CONFIG["seed"],
    )

    train_df = benign_train_pool.drop(index=reference_df.index).reset_index(drop=True)
    reference_df = reference_df.reset_index(drop=True)

    test_df = pd.concat([benign_test, attack_df], axis=0).reset_index(drop=True)

    print("  Fitting scaler on benign training data ...", flush=True)

    scaler = MinMaxScaler()
    scaler.fit(train_df[feature_cols])

    def apply_scaler(split_df):
        out = split_df.copy()
        out[feature_cols] = scaler.transform(out[feature_cols])
        return out.reset_index(drop=True)

    train_df = apply_scaler(train_df)
    reference_df = apply_scaler(reference_df)
    benign_val = apply_scaler(benign_val)
    test_df = apply_scaler(test_df)

    # Dynamic diagnostics for scaled feature ranges.
    train_min = float(np.nanmin(train_df[feature_cols].to_numpy()))
    train_max = float(np.nanmax(train_df[feature_cols].to_numpy()))
    test_min = float(np.nanmin(test_df[feature_cols].to_numpy()))
    test_max = float(np.nanmax(test_df[feature_cols].to_numpy()))

    attack_counts = (
        test_df[test_df["binary_label"] == 1]["attack_category"]
        .value_counts()
        .to_dict()
    )

    print(f"  Scaled train range: [{train_min:.3f}, {train_max:.3f}]", flush=True)
    print(f"  Scaled test range : [{test_min:.3f}, {test_max:.3f}]", flush=True)
    print(f"  Finished preprocessing {dataset_name}.", flush=True)

    return {
        "name": dataset_name,
        "path": str(path),
        "label_col": label_col,
        "raw_rows": raw_rows,
        "clean_rows": clean_rows,
        "feature_cols": feature_cols,
        "n_features": len(feature_cols),
        "expected_features": meta.get("expected_features", None),
        "forced_drop_cols": forced_drop_cols,
        "heavy_nan_cols": heavy_nan_cols,
        "non_numeric_cols": non_numeric_cols,
        "excluded_cols": excluded_cols,
        "row_drops": row_drops,
        "scaler": scaler,
        "train": train_df,
        "reference": reference_df,
        "val": benign_val,
        "test": test_df,
        "attack_counts": attack_counts,
        "benign_train_count": len(train_df),
        "benign_ref_count": len(reference_df),
        "benign_val_count": len(benign_val),
        "benign_test_count": int((test_df["binary_label"] == 0).sum()),
        "attack_test_count": int((test_df["binary_label"] == 1).sum()),
        "scaled_train_min": train_min,
        "scaled_train_max": train_max,
        "scaled_test_min": test_min,
        "scaled_test_max": test_max,
    }


DATA = {}
load_errors = {}

# Load only active datasets to avoid unnecessary memory use after kernel restart.
DATASETS_TO_LOAD = ACTIVE_DATASETS

print("[Cell 1.1] Data loading started", flush=True)
print(f"Datasets requested: {DATASETS_TO_LOAD}", flush=True)

for dataset_name in DATASETS_TO_LOAD:
    try:
        DATA[dataset_name] = clean_and_prepare_dataset(dataset_name, DATASETS[dataset_name])
    except Exception as e:
        load_errors[dataset_name] = str(e)

cell_status = "OK" if not load_errors else "CHECK_REQUIRED"

print(f"\n[Cell 1.1] Status: {cell_status}", flush=True)
print(f"Datasets loaded: {len(DATA)}/{len(DATASETS_TO_LOAD)}", flush=True)

for dataset_name, obj in DATA.items():
    print(f"\n{dataset_name}")
    print(f"  Rows      : raw={obj['raw_rows']:,}, clean={obj['clean_rows']:,}")
    print(f"  Features  : {obj['n_features']} (expected {obj['expected_features']})")
    print(
        f"  Splits    : train={obj['benign_train_count']:,}, "
        f"ref={obj['benign_ref_count']:,}, "
        f"val={obj['benign_val_count']:,}, "
        f"test={len(obj['test']):,}"
    )
    print(
        f"  Test mix  : benign={obj['benign_test_count']:,}, "
        f"attack={obj['attack_test_count']:,}"
    )
    print(f"  Attacks   : {obj['attack_counts']}")
    print(
        f"  NaN drops : forced={len(obj['forced_drop_cols'])}, "
        f"heavy_nan_cols={len(obj['heavy_nan_cols'])}, "
        f"non_numeric={len(obj['non_numeric_cols'])}, "
        f"rows={obj['row_drops']}"
    )
    print(
        f"  Scaled rng: train=[{obj['scaled_train_min']:.3f}, {obj['scaled_train_max']:.3f}], "
        f"test=[{obj['scaled_test_min']:.3f}, {obj['scaled_test_max']:.3f}]"
    )

if load_errors:
    print("\nLoad errors:")
    for name, msg in load_errors.items():
        print(f"  - {name}: {msg}")


## 2. Client Partitions, Models, Aggregators, Attacks, and Training Loop

In [ ]:
# ======================== CELL 1.2: Client Data Partitioning ========================

def make_loader_from_df(df, feature_cols, batch_size, shuffle=True):
    x = torch.tensor(df[feature_cols].values, dtype=torch.float32)
    return DataLoader(TensorDataset(x), batch_size=batch_size, shuffle=shuffle, drop_last=False)


def iid_partition(df, n_clients, seed):
    rng = np.random.default_rng(seed)
    indices = np.arange(len(df))
    rng.shuffle(indices)

    splits = np.array_split(indices, n_clients)
    return {
        cid: df.iloc[idx].reset_index(drop=True)
        for cid, idx in enumerate(splits)
    }


def choose_noniid_key(df, feature_cols, preferred_names=None):
    preferred_names = preferred_names or [
        "Protocol Type", "protocol_type", "Protocol", "Proto", "sTtl", "Rate"
    ]

    lower_to_real = {c.lower(): c for c in feature_cols}

    for name in preferred_names:
        if name.lower() in lower_to_real:
            return lower_to_real[name.lower()]

    variances = df[feature_cols].var(numeric_only=True).sort_values(ascending=False)
    return variances.index[0]


def make_group_labels(values, n_bins=10):
    if values.nunique(dropna=True) <= n_bins:
        return values.astype(str)

    ranked = values.rank(method="first")
    return pd.qcut(
        ranked,
        q=n_bins,
        labels=False,
        duplicates="drop"
    ).astype(str)


def balanced_shard_noniid_partition(
    df,
    feature_cols,
    n_clients,
    seed,
    shards_per_client=3,
    n_bins=10,
):
    rng = np.random.default_rng(seed)

    key_col = choose_noniid_key(df, feature_cols)
    group_labels = make_group_labels(df[key_col], n_bins=n_bins)

    work = df.copy()
    work["_noniid_group"] = group_labels.values
    work["_rand"] = rng.random(len(work))

    work = work.sort_values(["_noniid_group", "_rand"]).reset_index(drop=True)

    n_shards = n_clients * shards_per_client
    shard_indices = np.array_split(np.arange(len(work)), n_shards)
    shard_order = np.arange(n_shards)
    rng.shuffle(shard_order)

    client_indices = {cid: [] for cid in range(n_clients)}

    for pos, shard_id in enumerate(shard_order):
        cid = pos % n_clients
        client_indices[cid].extend(shard_indices[shard_id].tolist())

    partitions = {}

    for cid in range(n_clients):
        idx = np.array(client_indices[cid], dtype=int)
        rng.shuffle(idx)

        part = work.iloc[idx].drop(columns=["_noniid_group", "_rand"]).reset_index(drop=True)
        partitions[cid] = part

    return partitions, key_col


def summarize_partitions(partitions):
    counts = np.array([len(v) for v in partitions.values()])

    return {
        "min": int(counts.min()),
        "max": int(counts.max()),
        "mean": float(counts.mean()),
        "std": float(counts.std()),
        "empty_clients": int((counts == 0).sum()),
    }


def summarize_noniid_heterogeneity(partitions, key_col, n_bins=10):
    top_shares = []

    for part in partitions.values():
        labels = make_group_labels(part[key_col], n_bins=n_bins)
        shares = labels.value_counts(normalize=True)
        top_shares.append(float(shares.max()))

    top_shares = np.array(top_shares)

    return {
        "avg_top_group_share": float(top_shares.mean()),
        "min_top_group_share": float(top_shares.min()),
        "max_top_group_share": float(top_shares.max()),
    }


partition_errors = {}

for dataset_name, obj in DATA.items():
    try:
        train_df = obj["train"]
        feature_cols = obj["feature_cols"]

        iid_parts = iid_partition(
            df=train_df,
            n_clients=CONFIG["n_clients"],
            seed=CONFIG["seed"],
        )

        noniid_parts, noniid_key_col = balanced_shard_noniid_partition(
            df=train_df,
            feature_cols=feature_cols,
            n_clients=CONFIG["n_clients"],
            seed=CONFIG["seed"],
            shards_per_client=3,
            n_bins=CONFIG["n_clients"],
        )

        DATA[dataset_name]["iid_parts"] = iid_parts
        DATA[dataset_name]["noniid_parts"] = noniid_parts

        DATA[dataset_name]["iid_loaders"] = {
            cid: make_loader_from_df(part, feature_cols, CONFIG["batch_size"], shuffle=True)
            for cid, part in iid_parts.items()
        }

        DATA[dataset_name]["noniid_loaders"] = {
            cid: make_loader_from_df(part, feature_cols, CONFIG["batch_size"], shuffle=True)
            for cid, part in noniid_parts.items()
        }

        DATA[dataset_name]["partition_summary"] = {
            "iid": summarize_partitions(iid_parts),
            "noniid": summarize_partitions(noniid_parts),
            "noniid_key_col": noniid_key_col,
            "noniid_heterogeneity": summarize_noniid_heterogeneity(
                noniid_parts,
                noniid_key_col,
                n_bins=CONFIG["n_clients"],
            ),
        }

    except Exception as e:
        partition_errors[dataset_name] = str(e)


cell_status = "OK" if not partition_errors else "CHECK_REQUIRED"

print(f"[Cell 1.2] Status: {cell_status}")
print(f"Datasets partitioned: {len(DATA) - len(partition_errors)}/{len(DATA)}")

for name, obj in DATA.items():
    if "partition_summary" not in obj:
        continue

    ps = obj["partition_summary"]
    h = ps["noniid_heterogeneity"]

    print(f"\n{name}")
    print(
        f"  IID clients    : min={ps['iid']['min']:,}, max={ps['iid']['max']:,}, "
        f"mean={ps['iid']['mean']:.1f}, std={ps['iid']['std']:.1f}, empty={ps['iid']['empty_clients']}"
    )
    print(
        f"  Non-IID clients: min={ps['noniid']['min']:,}, max={ps['noniid']['max']:,}, "
        f"mean={ps['noniid']['mean']:.1f}, std={ps['noniid']['std']:.1f}, empty={ps['noniid']['empty_clients']}"
    )
    print(f"  Non-IID key    : {ps['noniid_key_col']}")
    print(
        f"  Heterogeneity : avg_top_group_share={h['avg_top_group_share']:.3f}, "
        f"range=({h['min_top_group_share']:.3f}, {h['max_top_group_share']:.3f})"
    )

if partition_errors:
    print("\nPartition errors:")
    for name, msg in partition_errors.items():
        print(f"  - {name}: {msg}")


In [ ]:
# ======================== CELL 2.1: Model Definitions (AE and VAE) ========================

class SimpleAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16, dropout=0.2):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, latent_dim),
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat


class SimpleVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16, dropout=0.2):
        super().__init__()

        self.shared_encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.mu_layer = nn.Linear(32, latent_dim)
        self.logvar_layer = nn.Linear(32, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.shared_encoder(x)
        mu = self.mu_layer(h)
        logvar = self.logvar_layer(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar


def reconstruction_loss(x_hat, x, reduction="mean"):
    return nn.functional.mse_loss(x_hat, x, reduction=reduction)


def kl_divergence(mu, logvar, reduction="mean"):
    kl_per_sample = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)

    if reduction == "mean":
        return kl_per_sample.mean()
    if reduction == "sum":
        return kl_per_sample.sum()
    return kl_per_sample


def vae_loss(x_hat, x, mu, logvar, kl_weight=1.0):
    rec = reconstruction_loss(x_hat, x, reduction="mean")
    kl = kl_divergence(mu, logvar, reduction="mean")
    return rec + kl_weight * kl, rec, kl


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


MODEL_REGISTRY = {}

model_errors = {}

for dataset_name, obj in DATA.items():
    try:
        input_dim = len(obj["feature_cols"])

        ae = SimpleAE(
            input_dim=input_dim,
            latent_dim=CONFIG["latent_dim"],
            dropout=CONFIG["dropout"],
        ).to(DEVICE)

        vae = SimpleVAE(
            input_dim=input_dim,
            latent_dim=CONFIG["latent_dim"],
            dropout=CONFIG["dropout"],
        ).to(DEVICE)

        MODEL_REGISTRY[dataset_name] = {
            "input_dim": input_dim,
            "AE": ae,
            "VAE": vae,
            "AE_params": count_parameters(ae),
            "VAE_params": count_parameters(vae),
        }

    except Exception as e:
        model_errors[dataset_name] = str(e)


cell_status = "OK" if not model_errors else "CHECK_REQUIRED"

print(f"[Cell 2.1] Status: {cell_status}")
print(f"Model sets created: {len(MODEL_REGISTRY)}/{len(DATA)}")

for name, info in MODEL_REGISTRY.items():
    print(f"\n{name}")
    print(f"  Input dim : {info['input_dim']}")
    print(f"  AE params : {info['AE_params']:,}")
    print(f"  VAE params: {info['VAE_params']:,}")

if model_errors:
    print("\nModel errors:")
    for name, msg in model_errors.items():
        print(f"  - {name}: {msg}")


In [ ]:
# ======================== 2.2: RAFA and Baseline Aggregation Functions ========================

def clone_model(model):
    cloned = copy.deepcopy(model)
    cloned.load_state_dict(copy.deepcopy(model.state_dict()))
    return cloned.to(DEVICE)


def get_trainable_param_names(model):
    return [name for name, _ in model.named_parameters()]


def get_model_update(global_model, local_model):
    global_params = dict(global_model.named_parameters())
    local_params = dict(local_model.named_parameters())

    return {
        name: (local_params[name].detach() - global_params[name].detach()).clone()
        for name in global_params.keys()
    }


def apply_update_to_model(global_model, update):
    candidate = clone_model(global_model)

    with torch.no_grad():
        for name, param in candidate.named_parameters():
            if name in update:
                param.add_(update[name].to(param.device))

    return candidate


def update_to_vector(update):
    return torch.cat([
        tensor.detach().flatten().float().cpu()
        for tensor in update.values()
    ])


def vector_to_update(vector, template_update):
    new_update = {}
    pointer = 0

    for name, tensor in template_update.items():
        numel = tensor.numel()
        new_update[name] = vector[pointer:pointer + numel].view_as(tensor).to(tensor.device)
        pointer += numel

    return new_update


def average_updates(updates, weights=None):
    if len(updates) == 0:
        raise ValueError("Cannot average an empty update list.")

    if weights is None:
        weights = np.ones(len(updates), dtype=np.float64)

    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / (weights.sum() + 1e-12)

    avg_update = {}

    for name in updates[0]:
        stacked = torch.stack([u[name].detach().float() for u in updates], dim=0)
        w = torch.tensor(
            weights,
            dtype=stacked.dtype,
            device=stacked.device
        ).view(-1, *([1] * (stacked.ndim - 1)))

        avg_update[name] = (stacked * w).sum(dim=0)

    return avg_update


def add_update_inplace(model, update, scale=1.0):
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in update:
                param.add_(scale * update[name].to(param.device))

    return model


def get_reconstruction_and_kl(model, x_tensor, model_type):
    model.eval()

    with torch.no_grad():
        x_tensor = x_tensor.to(DEVICE)

        if model_type.upper() == "VAE":
            x_hat, mu, logvar = model(x_tensor)
            rec_per_sample = torch.mean((x_hat - x_tensor) ** 2, dim=1)
            kl_per_sample = kl_divergence(mu, logvar, reduction="none")
            return rec_per_sample.mean().item(), kl_per_sample.mean().item()

        x_hat = model(x_tensor)
        rec_per_sample = torch.mean((x_hat - x_tensor) ** 2, dim=1)
        return rec_per_sample.mean().item(), 0.0


def get_reference_tensor(reference_df, feature_cols):
    return torch.tensor(reference_df[feature_cols].values, dtype=torch.float32)


# The published AQS uses the reconstruction part. beta=0 in all reported experiments,
# so the optional KL term below is inactive for the paper results.
def compute_aqs(global_model, client_update, reference_tensor, alpha, beta, model_type):
    candidate_model = apply_update_to_model(global_model, client_update)

    rec_0, kl_0 = get_reconstruction_and_kl(global_model, reference_tensor, model_type)
    rec_i, kl_i = get_reconstruction_and_kl(candidate_model, reference_tensor, model_type)

    rec_margin = CONFIG.get("rec_margin", 0.01)
    rec_clip = CONFIG.get("rec_clip", 2.0)

    kl_floor = CONFIG.get("kl_floor", 1e-3)
    kl_margin = CONFIG.get("kl_margin", 0.02)
    kl_clip = CONFIG.get("kl_clip", 2.0)

    rec_den = max(rec_0, 1e-8)
    rec_ratio_raw = (rec_i - rec_0) / rec_den
    rec_ratio = max(0.0, rec_ratio_raw - rec_margin)
    rec_ratio = min(rec_ratio, rec_clip)

    s_rec = float(np.exp(-alpha * rec_ratio))

    if model_type.upper() == "VAE":
        kl_den = max(kl_0, kl_floor)
        kl_ratio_raw = (kl_i - kl_0) / kl_den
        kl_ratio = max(0.0, kl_ratio_raw - kl_margin)
        kl_ratio = min(kl_ratio, kl_clip)
        s_kl = float(np.exp(-beta * kl_ratio))
    else:
        kl_ratio_raw = 0.0
        kl_ratio = 0.0
        s_kl = 1.0

    aqs = float(s_rec * s_kl)

    return {
        "aqs": aqs,
        "s_rec": s_rec,
        "s_kl": s_kl,
        "rec_0": rec_0,
        "rec_i": rec_i,
        "kl_0": kl_0,
        "kl_i": kl_i,
        "rec_ratio_raw": float(rec_ratio_raw),
        "rec_ratio": float(rec_ratio),
        "kl_ratio_raw": float(kl_ratio_raw),
        "kl_ratio": float(kl_ratio),
    }


def rafa_aggregate(global_model, client_updates, reference_tensor, alpha, beta, tau, model_type):
    aqs_info = [
        compute_aqs(global_model, upd, reference_tensor, alpha, beta, model_type)
        for upd in client_updates
    ]

    aqs_scores = np.array([x["aqs"] for x in aqs_info], dtype=np.float64)
    accepted = aqs_scores >= tau

    if accepted.sum() == 0:
        return clone_model(global_model), aqs_info

    accepted_updates = [upd for upd, keep in zip(client_updates, accepted) if keep]
    accepted_weights = aqs_scores[accepted]

    avg_update = average_updates(accepted_updates, weights=accepted_weights)
    new_model = clone_model(global_model)
    add_update_inplace(new_model, avg_update)

    return new_model, aqs_info


def fedavg_aggregate(global_model, client_updates, client_sizes=None):
    weights = None if client_sizes is None else np.asarray(client_sizes, dtype=np.float64)

    avg_update = average_updates(client_updates, weights=weights)
    new_model = clone_model(global_model)
    add_update_inplace(new_model, avg_update)

    return new_model, {"selected": len(client_updates)}


def fedavgm_aggregate(
    global_model,
    client_updates,
    velocity_state=None,
    client_sizes=None,
    beta_momentum=0.9,
    server_lr=1.0,
):
    if velocity_state is None:
        velocity_state = {
            name: torch.zeros_like(tensor, dtype=torch.float32)
            for name, tensor in client_updates[0].items()
        }

    avg_update = average_updates(client_updates, weights=client_sizes)

    new_velocity = {}
    momentum_update = {}

    for name in avg_update:
        prev_v = velocity_state[name].to(avg_update[name].device)
        new_v = beta_momentum * prev_v + avg_update[name]
        new_velocity[name] = new_v.detach().clone()
        momentum_update[name] = server_lr * new_v

    new_model = clone_model(global_model)
    add_update_inplace(new_model, momentum_update)

    return new_model, {
        "velocity": new_velocity,
        "selected": len(client_updates),
    }


def multi_krum_aggregate(global_model, client_updates, n_byzantine):
    n = len(client_updates)
    vectors = torch.stack([update_to_vector(u) for u in client_updates])

    k_neighbors = max(1, n - n_byzantine - 2)
    distances = torch.cdist(vectors, vectors, p=2) ** 2

    scores = []

    for i in range(n):
        nearest = torch.topk(distances[i], k=k_neighbors + 1, largest=False).values[1:]
        scores.append(nearest.sum().item())

    selected_idx = int(np.argmin(scores))
    selected_update = client_updates[selected_idx]

    new_model = clone_model(global_model)
    add_update_inplace(new_model, selected_update)

    return new_model, {
        "selected_idx": selected_idx,
        "scores": scores,
    }


def trimmed_mean_aggregate(global_model, client_updates, n_byzantine):
    n = len(client_updates)
    trim = min(n_byzantine, (n - 1) // 2)

    avg_update = {}

    for name in client_updates[0]:
        stacked = torch.stack([u[name].detach().float() for u in client_updates], dim=0)

        if trim > 0 and n > 2 * trim:
            sorted_vals, _ = torch.sort(stacked, dim=0)
            trimmed = sorted_vals[trim:n - trim]
        else:
            trimmed = stacked

        avg_update[name] = trimmed.mean(dim=0)

    new_model = clone_model(global_model)
    add_update_inplace(new_model, avg_update)

    return new_model, {
        "trim": trim,
        "selected": n - 2 * trim,
    }


def dnc_aggregate(global_model, client_updates, n_byzantine):
    n = len(client_updates)
    vectors = torch.stack([update_to_vector(u) for u in client_updates])
    centered = vectors - vectors.mean(dim=0, keepdim=True)

    try:
        _, _, vh = torch.linalg.svd(centered, full_matrices=False)
        direction = vh[0]
        scores = torch.abs(centered @ direction)
    except Exception:
        scores = torch.norm(centered, dim=1)

    keep = max(1, n - n_byzantine)
    selected_idx = torch.topk(scores, k=keep, largest=False).indices.cpu().numpy().tolist()

    selected_updates = [client_updates[i] for i in selected_idx]
    avg_update = average_updates(selected_updates)

    new_model = clone_model(global_model)
    add_update_inplace(new_model, avg_update)

    return new_model, {
        "selected_idx": selected_idx,
        "scores": scores.cpu().numpy().tolist(),
    }


def train_server_root_update(global_model, reference_tensor, model_type, lr=1e-3, epochs=1, batch_size=256):
    server_model = clone_model(global_model)
    server_model.train()

    loader = DataLoader(TensorDataset(reference_tensor), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(server_model.parameters(), lr=lr)

    for _ in range(epochs):
        for (x,) in loader:
            x = x.to(DEVICE)
            optimizer.zero_grad()

            if model_type.upper() == "VAE":
                x_hat, mu, logvar = server_model(x)
                loss, _, _ = vae_loss(x_hat, x, mu, logvar)
            else:
                x_hat = server_model(x)
                loss = reconstruction_loss(x_hat, x)

            loss.backward()
            optimizer.step()

    return get_model_update(global_model, server_model)


def fltrust_ae_aggregate(global_model, client_updates, reference_tensor, model_type, client_sizes=None):
    root_update = train_server_root_update(
        global_model=global_model,
        reference_tensor=reference_tensor,
        model_type=model_type,
        lr=CONFIG["lr"],
        epochs=1,
        batch_size=CONFIG["batch_size"],
    )

    root_vec = update_to_vector(root_update)
    root_norm = torch.norm(root_vec) + 1e-12

    trust_scores = []

    for upd in client_updates:
        vec = update_to_vector(upd)
        cos = torch.dot(vec, root_vec) / ((torch.norm(vec) + 1e-12) * root_norm)
        trust_scores.append(max(0.0, float(cos.item())))

    if np.sum(trust_scores) <= 1e-12:
        return clone_model(global_model), {
            "trust_scores": trust_scores,
            "selected": 0,
        }

    avg_update = average_updates(client_updates, weights=trust_scores)
    new_model = clone_model(global_model)
    add_update_inplace(new_model, avg_update)

    return new_model, {
        "trust_scores": trust_scores,
        "selected": int(np.sum(np.array(trust_scores) > 0)),
    }


# ------------------------
# Smoke test
# ------------------------
agg_errors = {}

for dataset_name, obj in DATA.items():
    try:
        feature_cols = obj["feature_cols"]
        reference_tensor = get_reference_tensor(obj["reference"], feature_cols)

        base_model = clone_model(MODEL_REGISTRY[dataset_name]["VAE"])
        local_model = clone_model(base_model)

        with torch.no_grad():
            for p in local_model.parameters():
                p.add_(0.001 * torch.randn_like(p))

        test_update = get_model_update(base_model, local_model)
        test_updates = [copy.deepcopy(test_update) for _ in range(CONFIG["n_clients"])]

        aqs_test = compute_aqs(
            global_model=base_model,
            client_update=test_update,
            reference_tensor=reference_tensor,
            alpha=CONFIG["alpha"],
            beta=CONFIG["beta"],
            model_type="VAE",
        )

        _, rafa_info = rafa_aggregate(
            base_model,
            test_updates,
            reference_tensor,
            CONFIG["alpha"],
            CONFIG["beta"],
            CONFIG["tau"],
            "VAE",
        )

        _, fedavg_info = fedavg_aggregate(base_model, test_updates)
        _, fedavgm_info = fedavgm_aggregate(base_model, test_updates)
        _, krum_info = multi_krum_aggregate(base_model, test_updates, n_byzantine=1)
        _, trim_info = trimmed_mean_aggregate(base_model, test_updates, n_byzantine=1)
        _, dnc_info = dnc_aggregate(base_model, test_updates, n_byzantine=1)
        _, fltrust_info = fltrust_ae_aggregate(base_model, test_updates, reference_tensor, "VAE")

        param_names = set(get_trainable_param_names(base_model))
        update_names = set(test_update.keys())

        buffer_like_keys = [
            k for k in update_names
            if ("running_mean" in k or "running_var" in k or "num_batches_tracked" in k)
        ]

        DATA[dataset_name]["cell_2_2_smoke"] = {
            "aqs": aqs_test["aqs"],
            "s_rec": aqs_test["s_rec"],
            "s_kl": aqs_test["s_kl"],
            "rafa_accepted": int(sum(x["aqs"] >= CONFIG["tau"] for x in rafa_info)),
            "fedavg_selected": fedavg_info["selected"],
            "fedavgm_selected": fedavgm_info["selected"],
            "krum_selected_idx": krum_info["selected_idx"],
            "trim_selected": trim_info["selected"],
            "dnc_selected": len(dnc_info["selected_idx"]),
            "fltrust_selected": fltrust_info["selected"],
            "trainable_params": len(param_names),
            "update_entries": len(update_names),
            "buffer_entries": len(buffer_like_keys),
        }

    except Exception as e:
        agg_errors[dataset_name] = str(e)


cell_status = "OK" if not agg_errors else "CHECK_REQUIRED"

print(f"[Cell 2.2] Status: {cell_status}")
print(f"Aggregator smoke tests: {len(DATA) - len(agg_errors)}/{len(DATA)}")

for name, obj in DATA.items():
    if "cell_2_2_smoke" not in obj:
        continue

    s = obj["cell_2_2_smoke"]

    print(f"\n{name}")
    print(f"  AQS test      : aqs={s['aqs']:.4f}, s_rec={s['s_rec']:.4f}, s_kl={s['s_kl']:.4f}")
    print(f"  RAFA accepted : {s['rafa_accepted']}/{CONFIG['n_clients']}")
    print(f"  FedAvg/FedAvgM: selected={s['fedavg_selected']}/{s['fedavgm_selected']}")
    print(f"  Krum/DnC      : krum_idx={s['krum_selected_idx']}, dnc_selected={s['dnc_selected']}")
    print(f"  Trim/FLTrust  : trim_selected={s['trim_selected']}, fltrust_selected={s['fltrust_selected']}")
    print(f"  Update check  : trainable={s['trainable_params']}, update_entries={s['update_entries']}, buffers={s['buffer_entries']}")

if agg_errors:
    print("\nAggregator errors:")
    for name, msg in agg_errors.items():
        print(f"  - {name}: {msg}")


In [ ]:
# ======================== 2.3: Attack Implementations ========================

def scale_update_to_norm(update, target_norm):
    vec = update_to_vector(update)
    norm = torch.norm(vec).item()

    if norm <= 1e-12:
        return update

    scale = target_norm / norm
    return {name: tensor * scale for name, tensor in update.items()}


def attack_random_noise(genuine_update, scale=1.0, match_norm=True):
    noise_update = {
        name: torch.randn_like(tensor.float()) * scale
        for name, tensor in genuine_update.items()
    }

    if match_norm:
        target_norm = torch.norm(update_to_vector(genuine_update)).item()
        noise_update = scale_update_to_norm(noise_update, target_norm)

    return noise_update


def attack_sign_flip(genuine_update, eta=5.0):
    return {
        name: -eta * tensor.detach().clone()
        for name, tensor in genuine_update.items()
    }


def train_model_for_attack(
    global_model,
    data_loader,
    model_type,
    objective="minimize_re",
    local_epochs=1,
    lr=1e-3,
    max_batches=None,
):
    attack_model = clone_model(global_model)
    attack_model.train()

    optimizer = torch.optim.Adam(attack_model.parameters(), lr=lr)
    batches_seen = 0

    for _ in range(local_epochs):
        for (x,) in data_loader:
            x = x.to(DEVICE)
            optimizer.zero_grad()

            if model_type.upper() == "VAE":
                x_hat, mu, logvar = attack_model(x)
                loss, rec, kl = vae_loss(x_hat, x, mu, logvar)
                target = rec
            else:
                x_hat = attack_model(x)
                target = reconstruction_loss(x_hat, x)

            if objective == "maximize_re":
                train_loss = -target
            elif objective == "minimize_re":
                train_loss = target
            else:
                raise ValueError(f"Unknown attack objective: {objective}")

            train_loss.backward()
            optimizer.step()

            batches_seen += 1
            if max_batches is not None and batches_seen >= max_batches:
                break

        if max_batches is not None and batches_seen >= max_batches:
            break

    return get_model_update(global_model, attack_model)


def attack_recon_inflation(
    global_model,
    target_loader,
    model_type,
    local_epochs=1,
    lr=1e-3,
    max_batches=20,
    match_norm_update=None,
):
    poisoned_update = train_model_for_attack(
        global_model=global_model,
        data_loader=target_loader,
        model_type=model_type,
        objective="maximize_re",
        local_epochs=local_epochs,
        lr=lr,
        max_batches=max_batches,
    )

    if match_norm_update is not None:
        target_norm = torch.norm(update_to_vector(match_norm_update)).item()
        poisoned_update = scale_update_to_norm(poisoned_update, target_norm)

    return poisoned_update


def attack_train_on_attack_traffic(
    global_model,
    attack_loader,
    model_type,
    local_epochs=1,
    lr=1e-3,
    max_batches=20,
    match_norm_update=None,
):
    poisoned_update = train_model_for_attack(
        global_model=global_model,
        data_loader=attack_loader,
        model_type=model_type,
        objective="minimize_re",
        local_epochs=local_epochs,
        lr=lr,
        max_batches=max_batches,
    )

    if match_norm_update is not None:
        target_norm = torch.norm(update_to_vector(match_norm_update)).item()
        poisoned_update = scale_update_to_norm(poisoned_update, target_norm)

    return poisoned_update


def attack_adaptive(
    global_model,
    proxy_loader,
    reference_tensor,
    model_type,
    alpha,
    beta,
    tau,
    variant="score_aware",
    lr=1e-3,
    steps=20,
    penalty_weight=10.0,
    max_batches_per_step=1,
    match_norm_update=None,
):
    attack_model = clone_model(global_model)
    attack_model.train()

    optimizer = torch.optim.Adam(attack_model.parameters(), lr=lr)
    proxy_iter = iter(proxy_loader)

    # Baseline reference statistics from the current global model
    global_model.eval()

    with torch.no_grad():
        x_ref = reference_tensor.to(DEVICE)

        if model_type.upper() == "VAE":
            x_ref_hat, mu_ref, logvar_ref = global_model(x_ref)
            rec_0 = torch.mean((x_ref_hat - x_ref) ** 2)
            kl_0 = kl_divergence(mu_ref, logvar_ref, reduction="mean")
        else:
            x_ref_hat = global_model(x_ref)
            rec_0 = torch.mean((x_ref_hat - x_ref) ** 2)
            kl_0 = torch.tensor(0.0, device=DEVICE)

    rec_margin = CONFIG.get("rec_margin", 0.01)
    rec_clip = CONFIG.get("rec_clip", 2.0)

    kl_floor = CONFIG.get("kl_floor", 1e-3)
    kl_margin = CONFIG.get("kl_margin", 0.02)
    kl_clip = CONFIG.get("kl_clip", 2.0)

    for _ in range(steps):
        try:
            (x_proxy,) = next(proxy_iter)
        except StopIteration:
            proxy_iter = iter(proxy_loader)
            (x_proxy,) = next(proxy_iter)

        x_proxy = x_proxy.to(DEVICE)
        optimizer.zero_grad()

        # Objective: degrade reconstruction on proxy data
        if model_type.upper() == "VAE":
            x_proxy_hat, _, _ = attack_model(x_proxy)
            rec_proxy = torch.mean((x_proxy_hat - x_proxy) ** 2)
        else:
            x_proxy_hat = attack_model(x_proxy)
            rec_proxy = torch.mean((x_proxy_hat - x_proxy) ** 2)

        # Differentiable RAFA score on the reference set
        if model_type.upper() == "VAE":
            x_ref_hat_i, mu_i, logvar_i = attack_model(x_ref)
            rec_i = torch.mean((x_ref_hat_i - x_ref) ** 2)
            kl_i = kl_divergence(mu_i, logvar_i, reduction="mean")
        else:
            x_ref_hat_i = attack_model(x_ref)
            rec_i = torch.mean((x_ref_hat_i - x_ref) ** 2)
            kl_i = torch.tensor(0.0, device=DEVICE)

        rec_ratio_raw = (rec_i - rec_0) / torch.clamp(rec_0, min=1e-8)
        rec_ratio = torch.clamp(rec_ratio_raw - rec_margin, min=0.0, max=rec_clip)
        s_rec = torch.exp(-alpha * rec_ratio)

        if model_type.upper() == "VAE":
            kl_den = torch.clamp(kl_0, min=kl_floor)
            kl_ratio_raw = (kl_i - kl_0) / kl_den
            kl_ratio = torch.clamp(kl_ratio_raw - kl_margin, min=0.0, max=kl_clip)
            s_kl = torch.exp(-beta * kl_ratio)
        else:
            s_kl = torch.tensor(1.0, device=DEVICE)

        aqs = s_rec * s_kl

        # Adaptive objective:
        # - maximize reconstruction error on proxy data
        # - remain above RAFA threshold if possible
        constraint_penalty = torch.relu(torch.tensor(tau, device=DEVICE) - aqs) ** 2
        loss = -rec_proxy + penalty_weight * constraint_penalty

        loss.backward()
        optimizer.step()

        if max_batches_per_step <= 1:
            continue

    poisoned_update = get_model_update(global_model, attack_model)

    if match_norm_update is not None:
        target_norm = torch.norm(update_to_vector(match_norm_update)).item()
        poisoned_update = scale_update_to_norm(poisoned_update, target_norm)

    attack_scale = CONFIG.get("adaptive_attack_scale", 1.0)

    poisoned_update = {
        name: tensor * attack_scale
        for name, tensor in poisoned_update.items()
    }

    return poisoned_update


# ------------------------
# Smoke test
# ------------------------
attack_errors = {}

for dataset_name, obj in DATA.items():
    try:
        feature_cols = obj["feature_cols"]
        base_model = clone_model(MODEL_REGISTRY[dataset_name]["VAE"])

        client_loader = DATA[dataset_name]["iid_loaders"][0]
        reference_tensor = get_reference_tensor(obj["reference"], feature_cols)

        local_update = train_model_for_attack(
            global_model=base_model,
            data_loader=client_loader,
            model_type="VAE",
            objective="minimize_re",
            local_epochs=1,
            lr=CONFIG["lr"],
            max_batches=2,
        )

        noise_update = attack_random_noise(local_update)
        flip_update = attack_sign_flip(local_update, eta=5.0)

        recon_update = attack_recon_inflation(
            global_model=base_model,
            target_loader=client_loader,
            model_type="VAE",
            local_epochs=1,
            lr=CONFIG["lr"],
            max_batches=2,
            match_norm_update=local_update,
        )

        adaptive_update = attack_adaptive(
            global_model=base_model,
            proxy_loader=client_loader,
            reference_tensor=reference_tensor,
            model_type="VAE",
            alpha=CONFIG["alpha"],
            beta=CONFIG["beta"],
            tau=CONFIG["tau"],
            variant="score_aware",
            lr=CONFIG["lr"],
            steps=2,
            match_norm_update=local_update,
        )

        template_keys = set(local_update.keys())

        test_updates = {
            "noise": noise_update,
            "sign_flip": flip_update,
            "recon_inflation": recon_update,
            "adaptive": adaptive_update,
        }

        shape_checks = {
            name: (
                set(upd.keys()) == template_keys
                and all(upd[k].shape == local_update[k].shape for k in template_keys)
            )
            for name, upd in test_updates.items()
        }

        DATA[dataset_name]["cell_2_3_smoke"] = {
            "local_norm": torch.norm(update_to_vector(local_update)).item(),
            "noise_norm": torch.norm(update_to_vector(noise_update)).item(),
            "flip_norm": torch.norm(update_to_vector(flip_update)).item(),
            "recon_norm": torch.norm(update_to_vector(recon_update)).item(),
            "adaptive_norm": torch.norm(update_to_vector(adaptive_update)).item(),
            "shape_checks": shape_checks,
        }

    except Exception as e:
        attack_errors[dataset_name] = str(e)


cell_status = "OK" if not attack_errors else "CHECK_REQUIRED"

print(f"[Cell 2.3] Status: {cell_status}")
print(f"Attack smoke tests: {len(DATA) - len(attack_errors)}/{len(DATA)}")

for name, obj in DATA.items():
    if "cell_2_3_smoke" not in obj:
        continue

    s = obj["cell_2_3_smoke"]

    print(f"\n{name}")
    print(f"  Update norms : local={s['local_norm']:.4f}, noise={s['noise_norm']:.4f}, sign_flip={s['flip_norm']:.4f}")
    print(f"               : recon={s['recon_norm']:.4f}, adaptive={s['adaptive_norm']:.4f}")
    print(f"  Shape checks : {s['shape_checks']}")

if attack_errors:
    print("\nAttack errors:")
    for name, msg in attack_errors.items():
        print(f"  - {name}: {msg}")


In [ ]:
# ======================== 2.4: Federated Training Loop and Evaluation ========================

def make_eval_loader(df, feature_cols, batch_size=1024, shuffle=False):
    x = torch.tensor(df[feature_cols].values, dtype=torch.float32)
    return DataLoader(TensorDataset(x), batch_size=batch_size, shuffle=shuffle, drop_last=False)


def train_local_model(global_model, train_loader, model_type, local_epochs, lr, max_batches=None):
    local_model = clone_model(global_model)
    local_model.train()

    optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)
    batches_seen = 0

    for _ in range(local_epochs):
        for (x,) in train_loader:
            x = x.to(DEVICE)
            optimizer.zero_grad()

            if model_type.upper() == "VAE":
                x_hat, mu, logvar = local_model(x)
                loss, _, _ = vae_loss(x_hat, x, mu, logvar)
            else:
                x_hat = local_model(x)
                loss = reconstruction_loss(x_hat, x)

            loss.backward()
            optimizer.step()

            batches_seen += 1
            if max_batches is not None and batches_seen >= max_batches:
                break

        if max_batches is not None and batches_seen >= max_batches:
            break

    return local_model, get_model_update(global_model, local_model)


def reconstruction_errors(model, data_loader, model_type):
    model.eval()
    errors = []

    with torch.no_grad():
        for (x,) in data_loader:
            x = x.to(DEVICE)

            if model_type.upper() == "VAE":
                x_hat, _, _ = model(x)
            else:
                x_hat = model(x)

            batch_errors = torch.mean((x_hat - x) ** 2, dim=1)
            errors.append(batch_errors.detach().cpu().numpy())

    return np.concatenate(errors) if errors else np.array([])


def calibrate_threshold(model, benign_val_df, feature_cols, model_type, percentile=95):
    val_loader = make_eval_loader(benign_val_df, feature_cols, batch_size=1024, shuffle=False)
    val_errors = reconstruction_errors(model, val_loader, model_type)

    if len(val_errors) == 0:
        raise ValueError("Validation set is empty, cannot calibrate threshold.")

    finite_errors = val_errors[np.isfinite(val_errors)]

    if len(finite_errors) == 0:
        raise ValueError("Validation reconstruction errors are all non-finite.")

    return float(np.percentile(finite_errors, percentile))


def evaluate_global_model(model, test_df, feature_cols, model_type, threshold):
    test_loader = make_eval_loader(test_df, feature_cols, batch_size=2048, shuffle=False)
    errors = reconstruction_errors(model, test_loader, model_type)

    y_true = test_df["binary_label"].values.astype(int)

    finite_mask = np.isfinite(errors)
    nonfinite_count = int((~finite_mask).sum())

    if len(errors) != len(y_true):
        raise ValueError(f"Error/label length mismatch: {len(errors)} vs {len(y_true)}")

    if nonfinite_count > 0:
        finite_errors = errors[finite_mask]
        replacement = float(np.nanmax(finite_errors)) if len(finite_errors) else threshold
        errors = np.where(finite_mask, errors, replacement)

    y_pred = (errors > threshold).astype(int)

    f1 = f1_score(y_true, y_pred, average="binary", zero_division=0)

    try:
        auc = roc_auc_score(y_true, errors)
    except ValueError:
        auc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn + 1e-12)
    asr = fn / (fn + tp + 1e-12)

    return {
        "f1": float(f1),
        "auc": float(auc),
        "fpr": float(fpr),
        "asr": float(asr),
        "threshold": float(threshold),
        "mean_re_benign": float(errors[y_true == 0].mean()) if np.any(y_true == 0) else np.nan,
        "mean_re_attack": float(errors[y_true == 1].mean()) if np.any(y_true == 1) else np.nan,
        "nonfinite_re_count": nonfinite_count,
    }


def select_byzantine_clients(n_clients, n_byzantine, seed):
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(np.arange(n_clients), size=n_byzantine, replace=False).tolist())


def make_attack_loader(dataset_obj, feature_cols, batch_size=256, max_samples=5000):
    attack_df = dataset_obj["test"].loc[dataset_obj["test"]["binary_label"] == 1].copy()

    if len(attack_df) > max_samples:
        attack_df = attack_df.sample(n=max_samples, random_state=CONFIG["seed"]).reset_index(drop=True)

    return make_eval_loader(attack_df, feature_cols, batch_size=batch_size, shuffle=True)


def scale_update(update, scale):
    return {
        name: tensor * scale
        for name, tensor in update.items()
    }


def apply_attack_to_update(
    attack_name,
    genuine_update,
    global_model,
    model_type,
    client_loader,
    attack_loader,
    reference_tensor,
):
    if attack_name in {None, "none"}:
        return genuine_update

    if attack_name == "noise":
        return attack_random_noise(genuine_update, scale=1.0, match_norm=True)

    if attack_name == "sign_flip":
        return attack_sign_flip(genuine_update, eta=5.0)

    if attack_name == "recon_inflation":
        attack_update = attack_recon_inflation(
            global_model=global_model,
            target_loader=client_loader,
            model_type=model_type,
            local_epochs=1,
            lr=CONFIG["lr"],
            max_batches=20,
            match_norm_update=genuine_update,
        )

        attack_scale = CONFIG.get("recon_inflation_scale", 1.0)
        return scale_update(attack_update, attack_scale)

    if attack_name == "train_on_attack":
        attack_update = attack_train_on_attack_traffic(
            global_model=global_model,
            attack_loader=attack_loader,
            model_type=model_type,
            local_epochs=1,
            lr=CONFIG["lr"],
            max_batches=20,
            match_norm_update=genuine_update,
        )

        attack_scale = CONFIG.get("train_on_attack_scale", 1.0)
        return scale_update(attack_update, attack_scale)

    if attack_name in {"adaptive_score", "adaptive_reference"}:
        variant = "score_aware" if attack_name == "adaptive_score" else "reference_aware"
        proxy_loader = attack_loader if variant == "score_aware" else client_loader

        attack_update = attack_adaptive(
            global_model=global_model,
            proxy_loader=proxy_loader,
            reference_tensor=reference_tensor,
            model_type=model_type,
            alpha=CONFIG["alpha"],
            beta=CONFIG["beta"],
            tau=CONFIG["tau"],
            variant=variant,
            lr=CONFIG["lr"],
            steps=20,
            match_norm_update=genuine_update,
        )

        attack_scale = CONFIG.get("adaptive_attack_scale", 1.0)
        return scale_update(attack_update, attack_scale)

    raise ValueError(f"Unknown attack name: {attack_name}")


def compute_screening_metrics(aqs_history, byzantine_ids, tau):
    if not aqs_history:
        return {"mdr": np.nan, "bfrr": np.nan}

    malicious_flags = []
    benign_flags = []

    byz = set(byzantine_ids)

    for round_scores in aqs_history:
        for cid, score in enumerate(round_scores):
            rejected = score < tau
            if cid in byz:
                malicious_flags.append(rejected)
            else:
                benign_flags.append(rejected)

    mdr = np.mean(malicious_flags) if malicious_flags else np.nan
    bfrr = np.mean(benign_flags) if benign_flags else np.nan

    return {
        "mdr": float(mdr) if not np.isnan(mdr) else np.nan,
        "bfrr": float(bfrr) if not np.isnan(bfrr) else np.nan,
    }


def summarize_aqs_history(aqs_history, byzantine_ids):
    if not aqs_history:
        return {
            "mean_aqs_benign": np.nan,
            "mean_aqs_malicious": np.nan,
            "aqs_gap": np.nan,
        }

    byz = set(byzantine_ids)
    benign_scores = []
    malicious_scores = []

    for round_scores in aqs_history:
        for cid, score in enumerate(round_scores):
            if cid in byz:
                malicious_scores.append(score)
            else:
                benign_scores.append(score)

    benign_scores = np.array(benign_scores, dtype=float)
    malicious_scores = np.array(malicious_scores, dtype=float)

    mean_benign = float(np.nanmean(benign_scores)) if len(benign_scores) else np.nan
    mean_malicious = float(np.nanmean(malicious_scores)) if len(malicious_scores) else np.nan

    return {
        "mean_aqs_benign": mean_benign,
        "mean_aqs_malicious": mean_malicious,
        "aqs_gap": mean_benign - mean_malicious if np.isfinite(mean_benign) and np.isfinite(mean_malicious) else np.nan,
    }


def aggregate_by_name(
    aggregator,
    global_model,
    client_updates,
    client_sizes,
    reference_tensor,
    model_type,
    n_byzantine,
    state,
):
    if aggregator == "RAFA":
        new_model, info = rafa_aggregate(
            global_model,
            client_updates,
            reference_tensor,
            CONFIG["alpha"],
            CONFIG["beta"],
            CONFIG["tau"],
            model_type,
        )
        return new_model, info, state

    if aggregator == "FedAvg":
        new_model, info = fedavg_aggregate(global_model, client_updates, client_sizes=client_sizes)
        return new_model, info, state

    if aggregator == "FedAvgM":
        new_model, info = fedavgm_aggregate(
            global_model,
            client_updates,
            velocity_state=state.get("velocity"),
            client_sizes=client_sizes,
            beta_momentum=CONFIG["beta_momentum"],
            server_lr=CONFIG["server_lr"],
        )
        state["velocity"] = info["velocity"]
        return new_model, info, state

    if aggregator == "Krum":
        new_model, info = multi_krum_aggregate(global_model, client_updates, n_byzantine=n_byzantine)
        return new_model, info, state

    if aggregator == "TrimMean":
        new_model, info = trimmed_mean_aggregate(global_model, client_updates, n_byzantine=n_byzantine)
        return new_model, info, state

    if aggregator == "DnC":
        new_model, info = dnc_aggregate(global_model, client_updates, n_byzantine=n_byzantine)
        return new_model, info, state

    if aggregator == "FLTrust-AE":
        new_model, info = fltrust_ae_aggregate(
            global_model,
            client_updates,
            reference_tensor,
            model_type,
            client_sizes=client_sizes,
        )
        return new_model, info, state

    raise ValueError(f"Unknown aggregator: {aggregator}")


def run_federation(config):
    dataset_name = config["dataset_name"]
    aggregator = config["aggregator"]
    model_type = config.get("model_type", "VAE")
    partition_name = config.get("partition", "iid")
    n_byzantine = int(config.get("n_byzantine", 0))
    attack_name = config.get("attack", "none")
    fed_rounds = int(config.get("fed_rounds", CONFIG["fed_rounds"]))
    max_local_batches = config.get("max_local_batches", None)

    dataset_obj = DATA[dataset_name]
    feature_cols = dataset_obj["feature_cols"]

    loaders_key = "iid_loaders" if partition_name == "iid" else "noniid_loaders"
    client_loaders = dataset_obj[loaders_key]
    client_parts = dataset_obj["iid_parts"] if partition_name == "iid" else dataset_obj["noniid_parts"]
    client_sizes = [len(client_parts[cid]) for cid in range(CONFIG["n_clients"])]

    global_model = clone_model(MODEL_REGISTRY[dataset_name][model_type.upper()])
    reference_tensor = get_reference_tensor(dataset_obj["reference"], feature_cols)
    attack_loader = make_attack_loader(dataset_obj, feature_cols, batch_size=CONFIG["batch_size"])

    byzantine_ids = select_byzantine_clients(
        n_clients=CONFIG["n_clients"],
        n_byzantine=n_byzantine,
        seed=CONFIG["seed"] + n_byzantine,
    )

    aqs_history = []
    round_re_ref = []
    round_re_test_benign = []
    aggregator_state = {}

    start_time = time.time()

    for round_idx in range(fed_rounds):
        client_updates = []

        for cid in range(CONFIG["n_clients"]):
            _, genuine_update = train_local_model(
                global_model=global_model,
                train_loader=client_loaders[cid],
                model_type=model_type,
                local_epochs=CONFIG["local_epochs"],
                lr=CONFIG["lr"],
                max_batches=max_local_batches,
            )

            if cid in byzantine_ids:
                final_update = apply_attack_to_update(
                    attack_name=attack_name,
                    genuine_update=genuine_update,
                    global_model=global_model,
                    model_type=model_type,
                    client_loader=client_loaders[cid],
                    attack_loader=attack_loader,
                    reference_tensor=reference_tensor,
                )
            else:
                final_update = genuine_update

            client_updates.append(final_update)

        global_model, agg_info, aggregator_state = aggregate_by_name(
            aggregator=aggregator,
            global_model=global_model,
            client_updates=client_updates,
            client_sizes=client_sizes,
            reference_tensor=reference_tensor,
            model_type=model_type,
            n_byzantine=n_byzantine,
            state=aggregator_state,
        )

        if aggregator == "RAFA":
            round_aqs = [x["aqs"] for x in agg_info]
            aqs_history.append(round_aqs)

        ref_loader = make_eval_loader(dataset_obj["reference"], feature_cols, batch_size=1024)
        test_benign_df = dataset_obj["test"].loc[dataset_obj["test"]["binary_label"] == 0]
        test_benign_loader = make_eval_loader(test_benign_df, feature_cols, batch_size=1024)

        ref_errors = reconstruction_errors(global_model, ref_loader, model_type)
        test_benign_errors = reconstruction_errors(global_model, test_benign_loader, model_type)

        round_re_ref.append(float(np.nanmean(ref_errors)))
        round_re_test_benign.append(float(np.nanmean(test_benign_errors)))

    threshold = calibrate_threshold(
        model=global_model,
        benign_val_df=dataset_obj["val"],
        feature_cols=feature_cols,
        model_type=model_type,
        percentile=CONFIG["threshold_percentile"],
    )

    final_metrics = evaluate_global_model(
        model=global_model,
        test_df=dataset_obj["test"],
        feature_cols=feature_cols,
        model_type=model_type,
        threshold=threshold,
    )

    screening = compute_screening_metrics(aqs_history, byzantine_ids, CONFIG["tau"])
    aqs_summary = summarize_aqs_history(aqs_history, byzantine_ids)

    return {
        "config": copy.deepcopy(config),
        "dataset": dataset_name,
        "aggregator": aggregator,
        "model_type": model_type,
        "partition": partition_name,
        "attack": attack_name,
        "n_byzantine": n_byzantine,
        "byzantine_ids": byzantine_ids,
        "final_metrics": final_metrics,
        "screening": screening,
        "aqs_summary": aqs_summary,
        "aqs_history": aqs_history,
        "round_re_ref": round_re_ref,
        "round_re_test_benign": round_re_test_benign,
        "runtime_sec": float(time.time() - start_time),
    }


# ------------------------
# Smoke test with reduced rounds/batches
# ------------------------
train_loop_errors = {}

for dataset_name in DATA.keys():
    try:
        smoke_result = run_federation({
            "dataset_name": dataset_name,
            "aggregator": "RAFA",
            "model_type": "VAE",
            "partition": "iid",
            "n_byzantine": 1,
            "attack": "sign_flip",
            "fed_rounds": 1,
            "max_local_batches": 1,
        })

        DATA[dataset_name]["cell_2_4_smoke"] = {
            "f1": smoke_result["final_metrics"]["f1"],
            "auc": smoke_result["final_metrics"]["auc"],
            "fpr": smoke_result["final_metrics"]["fpr"],
            "threshold": smoke_result["final_metrics"]["threshold"],
            "mdr": smoke_result["screening"]["mdr"],
            "bfrr": smoke_result["screening"]["bfrr"],
            "runtime_sec": smoke_result["runtime_sec"],
            "byzantine_ids": smoke_result["byzantine_ids"],
            "aqs_rounds": len(smoke_result["aqs_history"]),
            "nonfinite_re_count": smoke_result["final_metrics"].get("nonfinite_re_count", 0),
        }

    except Exception as e:
        train_loop_errors[dataset_name] = str(e)


cell_status = "OK" if not train_loop_errors else "CHECK_REQUIRED"

print(f"[Cell 2.4] Status: {cell_status}")
print(f"Training loop smoke tests: {len(DATA) - len(train_loop_errors)}/{len(DATA)}")

for name, obj in DATA.items():
    if "cell_2_4_smoke" not in obj:
        continue

    s = obj["cell_2_4_smoke"]

    print(f"\n{name}")
    print(f"  Metrics   : F1={s['f1']:.4f}, AUC={s['auc']:.4f}, FPR={s['fpr']:.4f}")
    print(f"  Threshold : {s['threshold']:.6f}")
    print(f"  Screening : MDR={s['mdr']:.4f}, BFRR={s['bfrr']:.4f}, AQS rounds={s['aqs_rounds']}")
    print(f"  Byzantine : {s['byzantine_ids']}")
    print(f"  Nonfinite : reconstruction_errors={s['nonfinite_re_count']}")
    print(f"  Runtime   : {s['runtime_sec']:.2f}s")

if train_loop_errors:
    print("\nTraining loop errors:")
    for name, msg in train_loop_errors.items():
        print(f"  - {name}: {msg}")


## 3. Main CICIoT2023 Experiments

The final paper numbering is used here:

- S1: reconstruction-targeted attacks
- S2: untargeted attacks
- S3: adaptive attacks
- S4: non-IID fidelity without attack

In [ ]:
# ======================== Paper S1 — Reconstruction-Targeted Attack ========================

import pickle
import json

if "ACTIVE_DATASETS" not in globals():
    ACTIVE_DATASETS = ["CICIoT2023"]

S1_AGGREGATORS = ["RAFA", "FedAvg", "FedAvgM", "Krum", "TrimMean", "DnC", "FLTrust-AE"]
S1_ATTACK = "recon_inflation"
S1_FRACTIONS = [0.2, 0.4]

# Controlled attack strength.
# lambda=1: mild semantic poisoning
# lambda=2: moderate semantic poisoning
# lambda=5: strong stress-test
S1_ATTACK_SCALES = [1.0, 2.0, 5.0]

S1_MAX_LOCAL_BATCHES = 20

S1_RESULTS = {}
S1_ROWS = []
S1_ERRORS = {}

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

s1_pickle_path = results_dir / "s1_results.pkl"
s1_csv_path = results_dir / "s1_summary.csv"
s1_errors_path = results_dir / "s1_errors.json"

original_recon_scale = CONFIG.get("recon_inflation_scale", 1.0)

total_runs = (
    len(ACTIVE_DATASETS)
    * len(S1_AGGREGATORS)
    * len(S1_FRACTIONS)
    * len(S1_ATTACK_SCALES)
)

run_id = 0
start_s1 = time.time()

print("[Paper S1] S1 reconstruction-targeted experiment started")
print(f"Active datasets   : {ACTIVE_DATASETS}")
print(f"Aggregators       : {S1_AGGREGATORS}")
print(f"Attack            : {S1_ATTACK}")
print(f"Attack scales     : {S1_ATTACK_SCALES}")
print(f"Fractions         : {S1_FRACTIONS}")
print(f"Fed rounds        : {CONFIG['fed_rounds']}")
print(f"Max local batches : {S1_MAX_LOCAL_BATCHES}")
print(f"Total runs        : {total_runs}")
print(f"Results dir       : {results_dir.resolve()}")

for dataset_name in ACTIVE_DATASETS:
    for attack_scale in S1_ATTACK_SCALES:
        CONFIG["recon_inflation_scale"] = attack_scale

        for frac in S1_FRACTIONS:
            n_byzantine = int(round(CONFIG["n_clients"] * frac))

            for aggregator in S1_AGGREGATORS:
                run_id += 1
                key = (dataset_name, aggregator, S1_ATTACK, attack_scale, frac)

                try:
                    run_start = time.time()

                    result = run_federation({
                        "dataset_name": dataset_name,
                        "aggregator": aggregator,
                        "model_type": "VAE",
                        "partition": "iid",
                        "n_byzantine": n_byzantine,
                        "attack": S1_ATTACK,
                        "fed_rounds": CONFIG["fed_rounds"],
                        "max_local_batches": S1_MAX_LOCAL_BATCHES,
                    })

                    metrics = result["final_metrics"]
                    screening = result["screening"]
                    aqs_summary = result.get("aqs_summary", {})

                    finite_metrics = all(
                        np.isfinite(metrics[k])
                        for k in ["f1", "auc", "fpr", "asr"]
                    )

                    S1_RESULTS[key] = result

                    row = {
                        "dataset": dataset_name,
                        "aggregator": aggregator,
                        "attack": S1_ATTACK,
                        "attack_scale": attack_scale,
                        "byzantine_fraction": frac,
                        "n_byzantine": n_byzantine,
                        "f1": metrics["f1"],
                        "auc": metrics["auc"],
                        "fpr": metrics["fpr"],
                        "asr": metrics["asr"],
                        "threshold": metrics["threshold"],
                        "nonfinite_re_count": metrics.get("nonfinite_re_count", 0),
                        "mdr": screening["mdr"],
                        "bfrr": screening["bfrr"],
                        "mean_aqs_benign": aqs_summary.get("mean_aqs_benign", np.nan),
                        "mean_aqs_malicious": aqs_summary.get("mean_aqs_malicious", np.nan),
                        "aqs_gap": aqs_summary.get("aqs_gap", np.nan),
                        "runtime_sec": result["runtime_sec"],
                        "finite_metrics": finite_metrics,
                        "fed_rounds": CONFIG["fed_rounds"],
                        "max_local_batches": S1_MAX_LOCAL_BATCHES,
                    }

                    S1_ROWS.append(row)

                    pd.DataFrame(S1_ROWS).to_csv(s1_csv_path, index=False)
                    with open(s1_pickle_path, "wb") as f:
                        pickle.dump(S1_RESULTS, f)

                    metric_flag = "finite" if finite_metrics else "NONFINITE"

                    print(
                        f"[{run_id:03d}/{total_runs}] {dataset_name} | {aggregator} | "
                        f"{S1_ATTACK} | scale={attack_scale:.1f} | f={frac:.1f} -> "
                        f"F1={metrics['f1']:.4f}, AUC={metrics['auc']:.4f}, "
                        f"FPR={metrics['fpr']:.4f}, ASR={metrics['asr']:.4f}, "
                        f"MDR={screening['mdr']:.4f}, BFRR={screening['bfrr']:.4f}, "
                        f"AQS_gap={row['aqs_gap']:.4f}, {metric_flag}, "
                        f"time={time.time() - run_start:.1f}s"
                    )

                except Exception as e:
                    S1_ERRORS[str(key)] = str(e)

                    with open(s1_errors_path, "w") as f:
                        json.dump(S1_ERRORS, f, indent=2)

                    print(
                        f"[{run_id:03d}/{total_runs}] ERROR | {dataset_name} | "
                        f"{aggregator} | {S1_ATTACK} | scale={attack_scale:.1f} | "
                        f"f={frac:.1f}: {e}"
                    )

CONFIG["recon_inflation_scale"] = original_recon_scale

s1_df = pd.DataFrame(S1_ROWS)

nonfinite_count = 0
if len(s1_df) > 0 and "finite_metrics" in s1_df.columns:
    nonfinite_count = int((~s1_df["finite_metrics"]).sum())

cell_status = "OK" if not S1_ERRORS and nonfinite_count == 0 else "CHECK_REQUIRED"

print(f"\n[Paper S1] Status: {cell_status}")
print(f"Completed runs    : {len(S1_ROWS)}/{total_runs}")
print(f"Nonfinite results : {nonfinite_count}")
print(f"Saved summary     : {s1_csv_path}")
print(f"Saved results     : {s1_pickle_path}")
print(f"Saved errors      : {s1_errors_path if S1_ERRORS else 'none'}")
print(f"Restored scale    : {CONFIG.get('recon_inflation_scale', None)}")
print(f"Total runtime     : {(time.time() - start_s1) / 60:.2f} min")

if len(s1_df) > 0:
    summary_cols = [
        "dataset", "aggregator", "attack", "attack_scale", "byzantine_fraction",
        "f1", "auc", "fpr", "asr",
        "mdr", "bfrr",
        "mean_aqs_benign", "mean_aqs_malicious", "aqs_gap",
        "nonfinite_re_count",
        "finite_metrics", "fed_rounds", "max_local_batches"
    ]

    display(
        s1_df[summary_cols]
        .sort_values(["dataset", "attack_scale", "byzantine_fraction", "aggregator"])
        .reset_index(drop=True)
    )

    print("\nBest F1 by attack scale and Byzantine fraction:")
    display(
        s1_df.sort_values(["attack_scale", "byzantine_fraction", "f1"], ascending=[True, True, False])
        .groupby(["attack_scale", "byzantine_fraction"])
        .head(3)
        [["attack_scale", "byzantine_fraction", "aggregator", "f1", "auc", "fpr", "asr"]]
        .reset_index(drop=True)
    )

    rafa_rows = s1_df[s1_df["aggregator"] == "RAFA"].copy()

    if len(rafa_rows) > 0:
        print("\nRAFA screening summary:")
        display(
            rafa_rows[
                [
                    "attack_scale", "byzantine_fraction",
                    "mdr", "bfrr",
                    "mean_aqs_benign", "mean_aqs_malicious", "aqs_gap",
                    "f1", "auc", "fpr", "asr"
                ]
            ].sort_values(["attack_scale", "byzantine_fraction"]).reset_index(drop=True)
        )

if S1_ERRORS:
    print("\nS1 errors:")
    for key, msg in S1_ERRORS.items():
        print(f"  - {key}: {msg}")


In [ ]:
# ======================== Paper S2 — Untargeted Attacks ========================

import pickle
import json

if "ACTIVE_DATASETS" not in globals():
    ACTIVE_DATASETS = ["CICIoT2023"]

S2_AGGREGATORS = ["RAFA", "FedAvg", "FedAvgM", "Krum", "TrimMean", "DnC", "FLTrust-AE"]

# S2 focuses on untargeted attacks only.
# Reconstruction-targeted attack is handled separately in S2.
S2_ATTACKS = ["noise", "sign_flip"]

S2_FRACTIONS = CONFIG["byzantine_fractions"]

# Runtime control: each client sees at most this many batches per round.
S2_MAX_LOCAL_BATCHES = 20

S2_RESULTS = {}
S2_ROWS = []
S2_ERRORS = {}

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

s2_pickle_path = results_dir / "s2_results.pkl"
s2_csv_path = results_dir / "s2_summary.csv"
s2_errors_path = results_dir / "s2_errors.json"

total_runs = len(ACTIVE_DATASETS) * len(S2_AGGREGATORS) * len(S2_ATTACKS) * len(S2_FRACTIONS)
run_id = 0
start_s2 = time.time()

print("[Paper S2] S2 experiment started")
print(f"Active datasets   : {ACTIVE_DATASETS}")
print(f"Aggregators       : {S2_AGGREGATORS}")
print(f"Attacks           : {S2_ATTACKS}")
print(f"Fractions         : {S2_FRACTIONS}")
print(f"Fed rounds        : {CONFIG['fed_rounds']}")
print(f"Max local batches : {S2_MAX_LOCAL_BATCHES}")
print(f"Total runs        : {total_runs}")
print(f"Results dir       : {results_dir.resolve()}")

for dataset_name in ACTIVE_DATASETS:
    for attack_name in S2_ATTACKS:
        for frac in S2_FRACTIONS:
            n_byzantine = int(round(CONFIG["n_clients"] * frac))

            for aggregator in S2_AGGREGATORS:
                run_id += 1
                key = (dataset_name, aggregator, attack_name, frac)

                try:
                    run_start = time.time()

                    result = run_federation({
                        "dataset_name": dataset_name,
                        "aggregator": aggregator,
                        "model_type": "VAE",
                        "partition": "iid",
                        "n_byzantine": n_byzantine,
                        "attack": attack_name,
                        "fed_rounds": CONFIG["fed_rounds"],
                        "max_local_batches": S2_MAX_LOCAL_BATCHES,
                    })

                    metrics = result["final_metrics"]
                    screening = result["screening"]

                    finite_metrics = all(
                        np.isfinite(metrics[k])
                        for k in ["f1", "auc", "fpr", "asr"]
                    )

                    S2_RESULTS[key] = result

                    row = {
                        "dataset": dataset_name,
                        "aggregator": aggregator,
                        "attack": attack_name,
                        "byzantine_fraction": frac,
                        "n_byzantine": n_byzantine,
                        "f1": metrics["f1"],
                        "auc": metrics["auc"],
                        "fpr": metrics["fpr"],
                        "asr": metrics["asr"],
                        "threshold": metrics["threshold"],
                        "mdr": screening["mdr"],
                        "bfrr": screening["bfrr"],
                        "runtime_sec": result["runtime_sec"],
                        "finite_metrics": finite_metrics,
                        "fed_rounds": CONFIG["fed_rounds"],
                        "max_local_batches": S2_MAX_LOCAL_BATCHES,
                    }

                    S2_ROWS.append(row)

                    pd.DataFrame(S2_ROWS).to_csv(s2_csv_path, index=False)
                    with open(s2_pickle_path, "wb") as f:
                        pickle.dump(S2_RESULTS, f)

                    metric_flag = "finite" if finite_metrics else "NONFINITE"

                    print(
                        f"[{run_id:03d}/{total_runs}] {dataset_name} | {aggregator} | "
                        f"{attack_name} | f={frac:.1f} -> "
                        f"F1={metrics['f1']:.4f}, AUC={metrics['auc']:.4f}, "
                        f"FPR={metrics['fpr']:.4f}, ASR={metrics['asr']:.4f}, "
                        f"MDR={screening['mdr']:.4f}, BFRR={screening['bfrr']:.4f}, "
                        f"{metric_flag}, time={time.time() - run_start:.1f}s"
                    )

                except Exception as e:
                    S2_ERRORS[str(key)] = str(e)

                    with open(s2_errors_path, "w") as f:
                        json.dump(S2_ERRORS, f, indent=2)

                    print(
                        f"[{run_id:03d}/{total_runs}] ERROR | {dataset_name} | "
                        f"{aggregator} | {attack_name} | f={frac:.1f}: {e}"
                    )

s2_df = pd.DataFrame(S2_ROWS)

nonfinite_count = 0
if len(s2_df) > 0 and "finite_metrics" in s2_df.columns:
    nonfinite_count = int((~s2_df["finite_metrics"]).sum())

cell_status = "OK" if not S2_ERRORS and nonfinite_count == 0 else "CHECK_REQUIRED"

print(f"\n[Paper S2] Status: {cell_status}")
print(f"Completed runs    : {len(S2_ROWS)}/{total_runs}")
print(f"Nonfinite results : {nonfinite_count}")
print(f"Saved summary     : {s2_csv_path}")
print(f"Saved results     : {s2_pickle_path}")
print(f"Saved errors      : {s2_errors_path if S2_ERRORS else 'none'}")
print(f"Total runtime     : {(time.time() - start_s2) / 60:.2f} min")

if len(s2_df) > 0:
    summary_cols = [
        "dataset", "aggregator", "attack", "byzantine_fraction",
        "f1", "auc", "fpr", "asr", "mdr", "bfrr", "finite_metrics",
        "fed_rounds", "max_local_batches"
    ]

    display(
        s2_df[summary_cols]
        .sort_values(["dataset", "attack", "byzantine_fraction", "aggregator"])
        .reset_index(drop=True)
    )

if S2_ERRORS:
    print("\nS2 errors:")
    for key, msg in S2_ERRORS.items():
        print(f"  - {key}: {msg}")


In [ ]:
# ======================== CELL 3.1b: Record S2 Diverged Rows ========================

import json
import ast
from pathlib import Path
import pandas as pd
import numpy as np

results_dir = Path(CONFIG.get("results_dir", "results"))
s2_csv_path = results_dir / "s2_summary.csv"
s2_errors_path = results_dir / "s2_errors.json"

if not s2_csv_path.exists():
    raise FileNotFoundError(f"S2 summary not found: {s2_csv_path}")

s2_df = pd.read_csv(s2_csv_path)

added_rows = []

if s2_errors_path.exists():
    with open(s2_errors_path, "r") as f:
        s2_errors = json.load(f)
else:
    s2_errors = {}

for key_text, msg in s2_errors.items():
    # key_text looks like "('CICIoT2023', 'FedAvgM', 'sign_flip', 0.2)"
    try:
        parsed = ast.literal_eval(key_text)
        dataset_name, aggregator, attack, frac = parsed
    except Exception:
        continue

    exists = (
        (s2_df["dataset"] == dataset_name)
        & (s2_df["aggregator"] == aggregator)
        & (s2_df["attack"] == attack)
        & (s2_df["byzantine_fraction"] == float(frac))
    ).any()

    if exists:
        continue

    added_rows.append({
        "dataset": dataset_name,
        "aggregator": aggregator,
        "attack": attack,
        "byzantine_fraction": float(frac),
        "n_byzantine": int(round(CONFIG["n_clients"] * float(frac))),
        "f1": np.nan,
        "auc": np.nan,
        "fpr": np.nan,
        "asr": np.nan,
        "threshold": np.nan,
        "mdr": np.nan,
        "bfrr": np.nan,
        "runtime_sec": np.nan,
        "finite_metrics": False,
        "fed_rounds": CONFIG["fed_rounds"],
        "max_local_batches": 20,
        "status": "Diverged",
        "error_message": msg,
    })

if added_rows:
    s2_df = pd.concat([s2_df, pd.DataFrame(added_rows)], ignore_index=True)
else:
    if "status" not in s2_df.columns:
        s2_df["status"] = np.where(s2_df.get("finite_metrics", True), "OK", "Diverged")
    if "error_message" not in s2_df.columns:
        s2_df["error_message"] = ""

# Ensure all existing finite rows have status
if "status" not in s2_df.columns:
    s2_df["status"] = "OK"
else:
    s2_df["status"] = s2_df["status"].fillna("OK")

if "error_message" not in s2_df.columns:
    s2_df["error_message"] = ""
else:
    s2_df["error_message"] = s2_df["error_message"].fillna("")

s2_df.to_csv(s2_csv_path, index=False)

print("[Cell 3.1b] S2 divergence bookkeeping completed")
print(f"Rows added as Diverged : {len(added_rows)}")
print(f"Total S2 rows          : {len(s2_df)}")
print(f"Saved summary          : {s2_csv_path}")

display(
    s2_df[
        [
            "dataset", "aggregator", "attack", "byzantine_fraction",
            "f1", "auc", "fpr", "asr", "mdr", "bfrr",
            "finite_metrics", "status"
        ]
    ]
    .sort_values(["attack", "byzantine_fraction", "aggregator"])
    .reset_index(drop=True)
)


In [ ]:
# ======================== 3.3: S3 — Adaptive Attack ========================

import pickle
import json

if "ACTIVE_DATASETS" not in globals():
    ACTIVE_DATASETS = ["CICIoT2023"]

S3_AGGREGATORS = ["RAFA", "FedAvg", "FedAvgM", "Krum", "TrimMean", "DnC", "FLTrust-AE"]
S3_VARIANTS = ["adaptive_score", "adaptive_reference"]
S3_FRACTIONS = [0.2, 0.4]

# Controlled adaptive strength.
# Keep this modest first. Increase later only if the attack is too weak.
S3_ATTACK_SCALES = [1.0, 2.0]

S3_MAX_LOCAL_BATCHES = 20

S3_RESULTS = {}
S3_ROWS = []
S3_ERRORS = {}

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

s3_pickle_path = results_dir / "s3_results.pkl"
s3_csv_path = results_dir / "s3_summary.csv"
s3_errors_path = results_dir / "s3_errors.json"

original_adaptive_scale = CONFIG.get("adaptive_attack_scale", 1.0)

total_runs = (
    len(ACTIVE_DATASETS)
    * len(S3_AGGREGATORS)
    * len(S3_VARIANTS)
    * len(S3_FRACTIONS)
    * len(S3_ATTACK_SCALES)
)

run_id = 0
start_s3 = time.time()

print("[Cell 3.3] S3 adaptive attack experiment started")
print(f"Active datasets   : {ACTIVE_DATASETS}")
print(f"Aggregators       : {S3_AGGREGATORS}")
print(f"Adaptive variants : {S3_VARIANTS}")
print(f"Attack scales     : {S3_ATTACK_SCALES}")
print(f"Fractions         : {S3_FRACTIONS}")
print(f"Fed rounds        : {CONFIG['fed_rounds']}")
print(f"Max local batches : {S3_MAX_LOCAL_BATCHES}")
print(f"Total runs        : {total_runs}")
print(f"Results dir       : {results_dir.resolve()}")

for dataset_name in ACTIVE_DATASETS:
    for variant in S3_VARIANTS:
        for attack_scale in S3_ATTACK_SCALES:
            CONFIG["adaptive_attack_scale"] = attack_scale

            for frac in S3_FRACTIONS:
                n_byzantine = int(round(CONFIG["n_clients"] * frac))

                for aggregator in S3_AGGREGATORS:
                    run_id += 1
                    key = (dataset_name, aggregator, variant, attack_scale, frac)

                    try:
                        run_start = time.time()

                        result = run_federation({
                            "dataset_name": dataset_name,
                            "aggregator": aggregator,
                            "model_type": "VAE",
                            "partition": "iid",
                            "n_byzantine": n_byzantine,
                            "attack": variant,
                            "fed_rounds": CONFIG["fed_rounds"],
                            "max_local_batches": S3_MAX_LOCAL_BATCHES,
                        })

                        metrics = result["final_metrics"]
                        screening = result["screening"]
                        aqs_summary = result.get("aqs_summary", {})

                        finite_metrics = all(
                            np.isfinite(metrics[k])
                            for k in ["f1", "auc", "fpr", "asr"]
                        )

                        S3_RESULTS[key] = result

                        row = {
                            "dataset": dataset_name,
                            "aggregator": aggregator,
                            "attack": variant,
                            "attack_scale": attack_scale,
                            "byzantine_fraction": frac,
                            "n_byzantine": n_byzantine,
                            "f1": metrics["f1"],
                            "auc": metrics["auc"],
                            "fpr": metrics["fpr"],
                            "asr": metrics["asr"],
                            "threshold": metrics["threshold"],
                            "nonfinite_re_count": metrics.get("nonfinite_re_count", 0),
                            "mdr": screening["mdr"],
                            "bfrr": screening["bfrr"],
                            "mean_aqs_benign": aqs_summary.get("mean_aqs_benign", np.nan),
                            "mean_aqs_malicious": aqs_summary.get("mean_aqs_malicious", np.nan),
                            "aqs_gap": aqs_summary.get("aqs_gap", np.nan),
                            "runtime_sec": result["runtime_sec"],
                            "finite_metrics": finite_metrics,
                            "fed_rounds": CONFIG["fed_rounds"],
                            "max_local_batches": S3_MAX_LOCAL_BATCHES,
                        }

                        S3_ROWS.append(row)

                        pd.DataFrame(S3_ROWS).to_csv(s3_csv_path, index=False)
                        with open(s3_pickle_path, "wb") as f:
                            pickle.dump(S3_RESULTS, f)

                        metric_flag = "finite" if finite_metrics else "NONFINITE"

                        print(
                            f"[{run_id:03d}/{total_runs}] {dataset_name} | {aggregator} | "
                            f"{variant} | scale={attack_scale:.1f} | f={frac:.1f} -> "
                            f"F1={metrics['f1']:.4f}, AUC={metrics['auc']:.4f}, "
                            f"FPR={metrics['fpr']:.4f}, ASR={metrics['asr']:.4f}, "
                            f"MDR={screening['mdr']:.4f}, BFRR={screening['bfrr']:.4f}, "
                            f"AQS_gap={row['aqs_gap']:.4f}, {metric_flag}, "
                            f"time={time.time() - run_start:.1f}s"
                        )

                    except Exception as e:
                        S3_ERRORS[str(key)] = str(e)

                        with open(s3_errors_path, "w") as f:
                            json.dump(S3_ERRORS, f, indent=2)

                        print(
                            f"[{run_id:03d}/{total_runs}] ERROR | {dataset_name} | "
                            f"{aggregator} | {variant} | scale={attack_scale:.1f} | "
                            f"f={frac:.1f}: {e}"
                        )

CONFIG["adaptive_attack_scale"] = original_adaptive_scale

s3_df = pd.DataFrame(S3_ROWS)

nonfinite_count = 0
if len(s3_df) > 0 and "finite_metrics" in s3_df.columns:
    nonfinite_count = int((~s3_df["finite_metrics"]).sum())

cell_status = "OK" if not S3_ERRORS and nonfinite_count == 0 else "CHECK_REQUIRED"

print(f"\n[Cell 3.3] Status: {cell_status}")
print(f"Completed runs    : {len(S3_ROWS)}/{total_runs}")
print(f"Nonfinite results : {nonfinite_count}")
print(f"Saved summary     : {s3_csv_path}")
print(f"Saved results     : {s3_pickle_path}")
print(f"Saved errors      : {s3_errors_path if S3_ERRORS else 'none'}")
print(f"Restored scale    : {CONFIG.get('adaptive_attack_scale', None)}")
print(f"Total runtime     : {(time.time() - start_s3) / 60:.2f} min")

if len(s3_df) > 0:
    summary_cols = [
        "dataset", "aggregator", "attack", "attack_scale", "byzantine_fraction",
        "f1", "auc", "fpr", "asr",
        "mdr", "bfrr",
        "mean_aqs_benign", "mean_aqs_malicious", "aqs_gap",
        "nonfinite_re_count",
        "finite_metrics", "fed_rounds", "max_local_batches"
    ]

    display(
        s3_df[summary_cols]
        .sort_values(["dataset", "attack", "attack_scale", "byzantine_fraction", "aggregator"])
        .reset_index(drop=True)
    )

    print("\nBest F1 by adaptive variant, scale, and Byzantine fraction:")
    display(
        s3_df.sort_values(
            ["attack", "attack_scale", "byzantine_fraction", "f1"],
            ascending=[True, True, True, False]
        )
        .groupby(["attack", "attack_scale", "byzantine_fraction"])
        .head(3)
        [["attack", "attack_scale", "byzantine_fraction", "aggregator", "f1", "auc", "fpr", "asr"]]
        .reset_index(drop=True)
    )

    rafa_rows = s3_df[s3_df["aggregator"] == "RAFA"].copy()

    if len(rafa_rows) > 0:
        print("\nRAFA adaptive screening summary:")
        display(
            rafa_rows[
                [
                    "attack", "attack_scale", "byzantine_fraction",
                    "mdr", "bfrr",
                    "mean_aqs_benign", "mean_aqs_malicious", "aqs_gap",
                    "f1", "auc", "fpr", "asr"
                ]
            ].sort_values(["attack", "attack_scale", "byzantine_fraction"]).reset_index(drop=True)
        )

if S3_ERRORS:
    print("\nS3 errors:")
    for key, msg in S3_ERRORS.items():
        print(f"  - {key}: {msg}")


In [ ]:
# ======================== 3.4: S4 — Non-IID Fidelity, No Attack ========================

import pickle
import json

if "ACTIVE_DATASETS" not in globals():
    ACTIVE_DATASETS = ["CICIoT2023"]

S4_AGGREGATORS = ["RAFA", "FedAvg", "FedAvgM", "Krum", "TrimMean", "DnC", "FLTrust-AE"]
S4_PARTITION = "noniid"
S4_ATTACK = "none"
S4_N_BYZANTINE = 0
S4_MAX_LOCAL_BATCHES = 20

S4_RESULTS = {}
S4_ROWS = []
S4_ERRORS = {}

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

s4_pickle_path = results_dir / "s4_results.pkl"
s4_csv_path = results_dir / "s4_summary.csv"
s4_errors_path = results_dir / "s4_errors.json"

total_runs = len(ACTIVE_DATASETS) * len(S4_AGGREGATORS)
run_id = 0
start_s4 = time.time()

print("[Cell 3.4] S4 Non-IID fidelity experiment started")
print(f"Active datasets   : {ACTIVE_DATASETS}")
print(f"Aggregators       : {S4_AGGREGATORS}")
print(f"Partition         : {S4_PARTITION}")
print(f"Attack            : {S4_ATTACK}")
print(f"Fed rounds        : {CONFIG['fed_rounds']}")
print(f"Max local batches : {S4_MAX_LOCAL_BATCHES}")
print(f"Total runs        : {total_runs}")
print(f"Results dir       : {results_dir.resolve()}")

for dataset_name in ACTIVE_DATASETS:
    for aggregator in S4_AGGREGATORS:
        run_id += 1
        key = (dataset_name, aggregator, S4_PARTITION, S4_ATTACK)

        try:
            run_start = time.time()

            result = run_federation({
                "dataset_name": dataset_name,
                "aggregator": aggregator,
                "model_type": "VAE",
                "partition": S4_PARTITION,
                "n_byzantine": S4_N_BYZANTINE,
                "attack": S4_ATTACK,
                "fed_rounds": CONFIG["fed_rounds"],
                "max_local_batches": S4_MAX_LOCAL_BATCHES,
            })

            metrics = result["final_metrics"]
            screening = result["screening"]
            aqs_summary = result.get("aqs_summary", {})

            finite_metrics = all(
                np.isfinite(metrics[k])
                for k in ["f1", "auc", "fpr", "asr"]
            )

            S4_RESULTS[key] = result

            row = {
                "dataset": dataset_name,
                "aggregator": aggregator,
                "partition": S4_PARTITION,
                "attack": S4_ATTACK,
                "n_byzantine": S4_N_BYZANTINE,
                "f1": metrics["f1"],
                "auc": metrics["auc"],
                "fpr": metrics["fpr"],
                "asr": metrics["asr"],
                "threshold": metrics["threshold"],
                "nonfinite_re_count": metrics.get("nonfinite_re_count", 0),
                "mdr": screening["mdr"],
                "bfrr": screening["bfrr"],
                "mean_aqs_benign": aqs_summary.get("mean_aqs_benign", np.nan),
                "mean_aqs_malicious": aqs_summary.get("mean_aqs_malicious", np.nan),
                "aqs_gap": aqs_summary.get("aqs_gap", np.nan),
                "runtime_sec": result["runtime_sec"],
                "finite_metrics": finite_metrics,
                "fed_rounds": CONFIG["fed_rounds"],
                "max_local_batches": S4_MAX_LOCAL_BATCHES,
            }

            S4_ROWS.append(row)

            pd.DataFrame(S4_ROWS).to_csv(s4_csv_path, index=False)
            with open(s4_pickle_path, "wb") as f:
                pickle.dump(S4_RESULTS, f)

            metric_flag = "finite" if finite_metrics else "NONFINITE"

            print(
                f"[{run_id:03d}/{total_runs}] {dataset_name} | {aggregator} | "
                f"{S4_PARTITION} | no attack -> "
                f"F1={metrics['f1']:.4f}, AUC={metrics['auc']:.4f}, "
                f"FPR={metrics['fpr']:.4f}, ASR={metrics['asr']:.4f}, "
                f"BFRR={screening['bfrr']:.4f}, {metric_flag}, "
                f"time={time.time() - run_start:.1f}s"
            )

        except Exception as e:
            S4_ERRORS[str(key)] = str(e)

            with open(s4_errors_path, "w") as f:
                json.dump(S4_ERRORS, f, indent=2)

            print(
                f"[{run_id:03d}/{total_runs}] ERROR | {dataset_name} | "
                f"{aggregator} | {S4_PARTITION}: {e}"
            )

s4_df = pd.DataFrame(S4_ROWS)

nonfinite_count = 0
if len(s4_df) > 0 and "finite_metrics" in s4_df.columns:
    nonfinite_count = int((~s4_df["finite_metrics"]).sum())

# Compute fidelity gap relative to FedAvg per dataset
if len(s4_df) > 0:
    s4_df["fidelity_gap_vs_fedavg"] = np.nan

    for dataset_name in s4_df["dataset"].unique():
        fedavg_rows = s4_df[
            (s4_df["dataset"] == dataset_name) &
            (s4_df["aggregator"] == "FedAvg")
        ]

        if len(fedavg_rows) == 1:
            fedavg_f1 = float(fedavg_rows.iloc[0]["f1"])
            mask = s4_df["dataset"] == dataset_name
            s4_df.loc[mask, "fidelity_gap_vs_fedavg"] = np.abs(s4_df.loc[mask, "f1"] - fedavg_f1)

    s4_df.to_csv(s4_csv_path, index=False)

cell_status = "OK" if not S4_ERRORS and nonfinite_count == 0 else "CHECK_REQUIRED"

print(f"\n[Cell 3.4] Status: {cell_status}")
print(f"Completed runs    : {len(S4_ROWS)}/{total_runs}")
print(f"Nonfinite results : {nonfinite_count}")
print(f"Saved summary     : {s4_csv_path}")
print(f"Saved results     : {s4_pickle_path}")
print(f"Saved errors      : {s4_errors_path if S4_ERRORS else 'none'}")
print(f"Total runtime     : {(time.time() - start_s4) / 60:.2f} min")

if len(s4_df) > 0:
    summary_cols = [
        "dataset", "aggregator", "partition",
        "f1", "auc", "fpr", "asr",
        "fidelity_gap_vs_fedavg",
        "mdr", "bfrr",
        "mean_aqs_benign", "mean_aqs_malicious", "aqs_gap",
        "nonfinite_re_count",
        "finite_metrics", "fed_rounds", "max_local_batches"
    ]

    display(
        s4_df[summary_cols]
        .sort_values(["dataset", "fidelity_gap_vs_fedavg", "aggregator"])
        .reset_index(drop=True)
    )

    print("\nBest F1 under Non-IID no-attack setting:")
    display(
        s4_df.sort_values(["dataset", "f1"], ascending=[True, False])
        [["dataset", "aggregator", "f1", "auc", "fpr", "asr", "fidelity_gap_vs_fedavg"]]
        .reset_index(drop=True)
    )

    rafa_rows = s4_df[s4_df["aggregator"] == "RAFA"].copy()

    if len(rafa_rows) > 0:
        print("\nRAFA Non-IID fidelity summary:")
        display(
            rafa_rows[
                [
                    "dataset", "f1", "auc", "fpr", "asr",
                    "fidelity_gap_vs_fedavg",
                    "bfrr", "mean_aqs_benign"
                ]
            ].reset_index(drop=True)
        )

if S4_ERRORS:
    print("\nS4 errors:")
    for key, msg in S4_ERRORS.items():
        print(f"  - {key}: {msg}")


## 4. Configuration, AQS Shape, Reference Size, and Runtime Checks

In [ ]:
# ======================== 4.3: Conservative vs Balanced RAFA Trade-off ========================

import pickle
from pathlib import Path

TRADEOFF_CONFIGS = [
    {
        "config_name": "Conservative",
        "alpha": 10.0,
        "beta": 0.0,
        "tau": 0.2,
        "description": "zero-BFRR-oriented setting",
    },
    {
        "config_name": "Balanced",
        "alpha": 15.0,
        "beta": 0.0,
        "tau": 0.2,
        "description": "sensitivity-selected balanced setting",
    },
]

TRADEOFF_SCENARIOS = [
    {
        "scenario": "S2 sign-flip",
        "partition": "iid",
        "attack": "sign_flip",
        "attack_scale": 1.0,
        "byzantine_fraction": 0.2,
    },
    {
        "scenario": "S2 sign-flip",
        "partition": "iid",
        "attack": "sign_flip",
        "attack_scale": 1.0,
        "byzantine_fraction": 0.4,
    },
    {
        "scenario": "S1 recon-inflation",
        "partition": "iid",
        "attack": "recon_inflation",
        "attack_scale": 2.0,
        "byzantine_fraction": 0.4,
    },
    {
        "scenario": "S1 recon-inflation",
        "partition": "iid",
        "attack": "recon_inflation",
        "attack_scale": 5.0,
        "byzantine_fraction": 0.4,
    },
    {
        "scenario": "S3 adaptive-reference",
        "partition": "iid",
        "attack": "adaptive_reference",
        "attack_scale": 2.0,
        "byzantine_fraction": 0.4,
    },
    {
        "scenario": "S4 Non-IID no attack",
        "partition": "noniid",
        "attack": "none",
        "attack_scale": 1.0,
        "byzantine_fraction": 0.0,
    },
]


def set_runtime_attack_scale(scale):
    CONFIG["attack_scale"] = float(scale)
    CONFIG["recon_inflation_scale"] = float(scale)
    CONFIG["adaptive_attack_scale"] = float(scale)


results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

tradeoff_csv_path = results_dir / "rafa_config_tradeoff_summary.csv"
tradeoff_pkl_path = results_dir / "rafa_config_tradeoff_results.pkl"

original_alpha = CONFIG["alpha"]
original_beta = CONFIG["beta"]
original_tau = CONFIG["tau"]
original_attack_scale = CONFIG.get("attack_scale", 1.0)
original_recon_scale = CONFIG.get("recon_inflation_scale", 1.0)
original_adaptive_scale = CONFIG.get("adaptive_attack_scale", 1.0)

TRADEOFF_RESULTS = {}
TRADEOFF_ROWS = []
TRADEOFF_ERRORS = {}

total_runs = len(ACTIVE_DATASETS) * len(TRADEOFF_CONFIGS) * len(TRADEOFF_SCENARIOS)
run_id = 0
start_time = time.time()

print("[Cell 4.3] RAFA configuration trade-off started")
print(f"Active datasets : {ACTIVE_DATASETS}")
print(f"Configurations  : {[c['config_name'] for c in TRADEOFF_CONFIGS]}")
print(f"Scenarios       : {len(TRADEOFF_SCENARIOS)}")
print(f"Fed rounds      : {CONFIG['fed_rounds']}")
print(f"Max local batches: {CONFIG.get('max_local_batches', 20)}")
print(f"Total runs      : {total_runs}")
print(f"Results dir     : {results_dir}")

for dataset_name in ACTIVE_DATASETS:
    for cfg in TRADEOFF_CONFIGS:
        CONFIG["alpha"] = cfg["alpha"]
        CONFIG["beta"] = cfg["beta"]
        CONFIG["tau"] = cfg["tau"]

        for scenario in TRADEOFF_SCENARIOS:
            run_id += 1

            attack_name = scenario["attack"]
            frac = scenario["byzantine_fraction"]
            n_byzantine = int(round(CONFIG["n_clients"] * frac))
            set_runtime_attack_scale(scenario["attack_scale"])

            key = (
                dataset_name,
                cfg["config_name"],
                scenario["scenario"],
                attack_name,
                scenario["attack_scale"],
                frac,
            )

            try:
                run_start = time.time()

                result = run_federation({
                    "dataset_name": dataset_name,
                    "aggregator": "RAFA",
                    "model_type": "VAE",
                    "partition": scenario["partition"],
                    "n_byzantine": n_byzantine,
                    "attack": attack_name,
                    "fed_rounds": CONFIG["fed_rounds"],
                    "max_local_batches": CONFIG.get("max_local_batches", 20),
                })

                metrics = result["final_metrics"]
                screening = result["screening"]

                mean_aqs_benign = np.nan
                mean_aqs_malicious = np.nan
                aqs_gap = np.nan

                if "aqs_history" in result and result["aqs_history"]:
                    aqs_df = pd.DataFrame(result["aqs_history"])

                    if "type" in aqs_df.columns and "aqs" in aqs_df.columns:
                        benign_aqs = aqs_df.loc[aqs_df["type"] == "benign", "aqs"]
                        malicious_aqs = aqs_df.loc[aqs_df["type"] == "malicious", "aqs"]

                        if len(benign_aqs) > 0:
                            mean_aqs_benign = float(benign_aqs.mean())

                        if len(malicious_aqs) > 0:
                            mean_aqs_malicious = float(malicious_aqs.mean())

                        if np.isfinite(mean_aqs_benign) and np.isfinite(mean_aqs_malicious):
                            aqs_gap = mean_aqs_benign - mean_aqs_malicious

                finite_metrics = all(
                    np.isfinite(metrics.get(k, np.nan))
                    for k in ["f1", "auc", "fpr", "asr"]
                )

                row = {
                    "dataset": dataset_name,
                    "config_name": cfg["config_name"],
                    "description": cfg["description"],
                    "alpha": cfg["alpha"],
                    "beta": cfg["beta"],
                    "tau": cfg["tau"],
                    "scenario": scenario["scenario"],
                    "partition": scenario["partition"],
                    "attack": attack_name,
                    "attack_scale": scenario["attack_scale"],
                    "byzantine_fraction": frac,
                    "n_byzantine": n_byzantine,
                    "f1": metrics["f1"],
                    "auc": metrics["auc"],
                    "fpr": metrics["fpr"],
                    "asr": metrics["asr"],
                    "threshold": metrics["threshold"],
                    "mdr": screening["mdr"],
                    "bfrr": screening["bfrr"],
                    "mean_aqs_benign": mean_aqs_benign,
                    "mean_aqs_malicious": mean_aqs_malicious,
                    "aqs_gap": aqs_gap,
                    "finite_metrics": finite_metrics,
                    "runtime_sec": result["runtime_sec"],
                }

                TRADEOFF_RESULTS[key] = result
                TRADEOFF_ROWS.append(row)

                pd.DataFrame(TRADEOFF_ROWS).to_csv(tradeoff_csv_path, index=False)

                with open(tradeoff_pkl_path, "wb") as f:
                    pickle.dump(TRADEOFF_RESULTS, f)

                print(
                    f"[{run_id:02d}/{total_runs}] {dataset_name} | {cfg['config_name']} | "
                    f"{scenario['scenario']} | {attack_name} | "
                    f"scale={scenario['attack_scale']} | f={frac:.1f} -> "
                    f"F1={metrics['f1']:.4f}, AUC={metrics['auc']:.4f}, "
                    f"MDR={screening['mdr']:.4f}, BFRR={screening['bfrr']:.4f}, "
                    f"AQS_gap={aqs_gap:.4f}, time={time.time() - run_start:.1f}s"
                )

            except Exception as e:
                TRADEOFF_ERRORS[key] = str(e)

                print(
                    f"[{run_id:02d}/{total_runs}] ERROR | {dataset_name} | "
                    f"{cfg['config_name']} | {scenario['scenario']} | "
                    f"{attack_name} | scale={scenario['attack_scale']} | f={frac:.1f}: {e}"
                )

# Restore global config
CONFIG["alpha"] = original_alpha
CONFIG["beta"] = original_beta
CONFIG["tau"] = original_tau
CONFIG["attack_scale"] = original_attack_scale
CONFIG["recon_inflation_scale"] = original_recon_scale
CONFIG["adaptive_attack_scale"] = original_adaptive_scale

tradeoff_df = pd.DataFrame(TRADEOFF_ROWS)
cell_status = "OK" if not TRADEOFF_ERRORS else "CHECK_REQUIRED"

print(f"\n[Cell 4.3] Status: {cell_status}")
print(f"Completed runs : {len(TRADEOFF_ROWS)}/{total_runs}")
print(f"Saved summary  : {tradeoff_csv_path}")
print(f"Saved results  : {tradeoff_pkl_path}")
print(f"Restored config: alpha={CONFIG['alpha']}, beta={CONFIG['beta']}, tau={CONFIG['tau']}")
print(f"Total runtime  : {(time.time() - start_time) / 60:.2f} min")

if len(tradeoff_df) > 0:
    display_cols = [
        "config_name", "scenario", "attack", "attack_scale", "byzantine_fraction",
        "f1", "auc", "fpr", "asr", "mdr", "bfrr", "aqs_gap"
    ]

    display(
        tradeoff_df[display_cols]
        .sort_values(["scenario", "attack_scale", "byzantine_fraction", "config_name"])
        .reset_index(drop=True)
    )

    print("\nConfiguration-level mean summary:")
    config_summary = (
        tradeoff_df
        .groupby("config_name", as_index=False)
        .agg(
            mean_f1=("f1", "mean"),
            mean_auc=("auc", "mean"),
            mean_mdr=("mdr", "mean"),
            mean_bfrr=("bfrr", "mean"),
            mean_aqs_gap=("aqs_gap", "mean"),
        )
    )

    display(config_summary)

if TRADEOFF_ERRORS:
    print("\nTrade-off errors:")
    for key, msg in TRADEOFF_ERRORS.items():
        print(f"  - {key}: {msg}")


In [ ]:
# ======================== 4.4: AQS Scoring-Function Shape Ablation ========================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc
import inspect

print("[Cell 8.2] Fixed AQS scoring-function shape ablation started", flush=True)

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# Settings
# -------------------------

CELL_8_2_OUTPUT_PREFIX = CONFIG.get(
    "cell_8_2_output_prefix_v3",
    "cell_8_2_aqs_shape_ablation_ciciot2023_v3"
)

CELL_8_2_DATASET = CONFIG.get("cell_8_2_dataset", "CICIoT2023")
CELL_8_2_SEEDS = [int(s) for s in CONFIG.get("cell_8_2_seeds", [11, 22, 33])]
CELL_8_2_SCENARIOS = CONFIG.get(
    "cell_8_2_scenarios",
    ["recon_targeted_s5_f04", "sign_flip_f04"]
)
CELL_8_2_AQS_SHAPES = CONFIG.get(
    "cell_8_2_aqs_shapes",
    ["exponential", "linear", "sigmoid"]
)

CELL_8_2_BYZ_FRAC = float(CONFIG.get("cell_8_2_byzantine_fraction", 0.4))
CELL_8_2_FED_ROUNDS = int(CONFIG.get("cell_8_2_fed_rounds", 20))
CELL_8_2_MAX_LOCAL_BATCHES = int(CONFIG.get("cell_8_2_max_local_batches", 20))
CELL_8_2_RESUME = bool(CONFIG.get("cell_8_2_resume", True))

raw_path = results_dir / f"{CELL_8_2_OUTPUT_PREFIX}_raw.csv"
summary_path = results_dir / f"{CELL_8_2_OUTPUT_PREFIX}_summary.csv"
comparison_path = results_dir / f"{CELL_8_2_OUTPUT_PREFIX}_comparison.csv"
latex_path = results_dir / f"{CELL_8_2_OUTPUT_PREFIX}_latex.txt"

print(f"Dataset           : {CELL_8_2_DATASET}", flush=True)
print(f"Seeds             : {CELL_8_2_SEEDS}", flush=True)
print(f"Scenarios         : {CELL_8_2_SCENARIOS}", flush=True)
print(f"AQS shapes        : {CELL_8_2_AQS_SHAPES}", flush=True)
print(f"Byzantine fraction: {CELL_8_2_BYZ_FRAC}", flush=True)
print(f"Fed rounds        : {CELL_8_2_FED_ROUNDS}", flush=True)
print(f"Max local batches : {CELL_8_2_MAX_LOCAL_BATCHES}", flush=True)

if CELL_8_2_DATASET not in DATA:
    raise ValueError(f"{CELL_8_2_DATASET} is not loaded in DATA.")

if "compute_aqs" not in globals() or not callable(compute_aqs):
    raise RuntimeError("compute_aqs() was not found. Cannot run Cell 8.2 safely.")

ORIGINAL_COMPUTE_AQS = compute_aqs

print("\n[Cell 8.2] Detected compute_aqs signature:", flush=True)
print(inspect.signature(ORIGINAL_COMPUTE_AQS), flush=True)


# -------------------------
# AQS transformation helpers
# -------------------------

def c82_transform_score(default_score, shape, alpha):
    """
    The original compute_aqs implements the exponential scoring rule.
    If score = exp(-alpha * penalty), then penalty = -log(score) / alpha.

    We reuse the same estimated penalty and compare:
      exponential: original score
      linear     : max(0, 1 - alpha * penalty)
      sigmoid    : 2 / (1 + exp(alpha * penalty))
    """
    if shape == "exponential":
        return default_score

    eps = 1e-12
    alpha = max(float(alpha), eps)

    arr = np.asarray(default_score, dtype=float)
    clipped = np.clip(arr, eps, 1.0)
    penalty = -np.log(clipped) / alpha

    if shape == "linear":
        out = np.maximum(0.0, 1.0 - alpha * penalty)
    elif shape == "sigmoid":
        out = 2.0 / (1.0 + np.exp(alpha * penalty))
    else:
        raise ValueError(f"Unknown AQS shape: {shape}")

    out = np.clip(out, 0.0, 1.0)

    if np.isscalar(default_score):
        return float(out)
    return out


def c82_replace_score_in_output(output, shape, alpha):
    """
    Preserve the original compute_aqs() return structure.
    Replace only the AQS score itself.
    """
    if shape == "exponential":
        return output

    if isinstance(output, (int, float, np.floating)):
        return c82_transform_score(float(output), shape, alpha)

    if isinstance(output, dict):
        out = dict(output)
        if "aqs" not in out:
            raise RuntimeError(f"compute_aqs returned dict without 'aqs'. Keys: {list(out.keys())}")
        out["aqs"] = c82_transform_score(out["aqs"], shape, alpha)
        return out

    if isinstance(output, tuple):
        if len(output) == 0:
            return output
        out = list(output)
        out[0] = c82_transform_score(out[0], shape, alpha)
        return tuple(out)

    if isinstance(output, list):
        if len(output) == 0:
            return output
        out = list(output)
        out[0] = c82_transform_score(out[0], shape, alpha)
        return out

    if isinstance(output, np.ndarray):
        return c82_transform_score(output, shape, alpha)

    raise RuntimeError(f"Unsupported compute_aqs return type: {type(output)}")


def c82_make_patched_compute_aqs(shape):
    def patched_compute_aqs(global_model, client_update, reference_tensor, alpha, beta, model_type):
        original_output = ORIGINAL_COMPUTE_AQS(
            global_model,
            client_update,
            reference_tensor,
            alpha,
            beta,
            model_type,
        )
        return c82_replace_score_in_output(original_output, shape, alpha)
    return patched_compute_aqs


def c82_patch_aqs(shape):
    globals()["compute_aqs"] = c82_make_patched_compute_aqs(shape)


def c82_restore_aqs():
    globals()["compute_aqs"] = ORIGINAL_COMPUTE_AQS


# -------------------------
# Run helpers
# -------------------------

def c82_is_diverged(msg):
    msg = str(msg).lower()
    return any(x in msg for x in ["non-finite", "nonfinite", "nan", "inf", "diverged"])


def c82_scenario_to_cfg(scenario):
    """
    Scenario labels are for reporting.
    Attack names must match the notebook's actual attack implementation.
    """
    if scenario == "recon_targeted_s5_f04":
        return {
            "scenario": scenario,
            "attack": "recon_inflation",
            "attack_scale": 5.0,
        }

    if scenario == "sign_flip_f04":
        return {
            "scenario": scenario,
            "attack": "sign_flip",
            "attack_scale": 1.0,
        }

    raise ValueError(f"Unsupported Cell 8.2 scenario: {scenario}")


def c82_run(shape, scenario, seed):
    scenario_cfg = c82_scenario_to_cfg(scenario)

    run_cfg = {
        "dataset_name": CELL_8_2_DATASET,
        "aggregator": "RAFA",
        "model_type": "VAE",
        "partition": "iid",
        "byzantine_fraction": CELL_8_2_BYZ_FRAC,
        "n_byzantine": int(round(CONFIG["n_clients"] * CELL_8_2_BYZ_FRAC)),
        "fed_rounds": CELL_8_2_FED_ROUNDS,
        "max_local_batches": CELL_8_2_MAX_LOCAL_BATCHES,
        **scenario_cfg,
    }

    c82_patch_aqs(shape)

    try:
        _, row = run_review_trial(run_cfg, int(seed))
        if row.get("status") == "error" and c82_is_diverged(row.get("error_message", "")):
            row["status"] = "diverged"

    except Exception as e:
        row = {
            "dataset": CELL_8_2_DATASET,
            "scenario": scenario,
            "aggregator": "RAFA",
            "seed": int(seed),
            "status": "diverged" if c82_is_diverged(e) else "error",
            "error_message": str(e),
            "runtime_sec": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "fpr": np.nan,
            "asr": np.nan,
            "mdr": np.nan,
            "bfrr": np.nan,
            "mean_aqs_benign": np.nan,
            "mean_aqs_malicious": np.nan,
            "aqs_gap": np.nan,
        }

    finally:
        c82_restore_aqs()

    row["aqs_shape"] = shape
    return row


# -------------------------
# Build run queue and resume
# -------------------------

run_items = [
    (shape, scenario, seed)
    for shape in CELL_8_2_AQS_SHAPES
    for scenario in CELL_8_2_SCENARIOS
    for seed in CELL_8_2_SEEDS
]

rows = []
completed = set()

if CELL_8_2_RESUME and raw_path.exists():
    prev = pd.read_csv(raw_path)
    rows = prev.to_dict("records")
    done = prev[prev["status"].isin(["ok", "diverged"])].copy()
    completed = set(
        zip(
            done["aqs_shape"].astype(str),
            done["scenario"].astype(str),
            done["seed"].astype(int),
        )
    )
    print(f"Resume enabled    : found {len(completed)} completed/diverged runs", flush=True)
else:
    print("Resume enabled    : no previous completed/diverged runs found", flush=True)

print(f"Run items         : {len(run_items)}", flush=True)


# -------------------------
# Run experiments
# -------------------------

start_time = time.time()
new_runs = 0

for shape, scenario, seed in run_items:
    key = (str(shape), str(scenario), int(seed))
    if key in completed:
        continue

    t0 = time.time()
    row = c82_run(shape, scenario, seed)
    row["elapsed_sec"] = time.time() - t0
    rows.append(row)

    pd.DataFrame(rows).to_csv(raw_path, index=False)
    new_runs += 1

    print(
        f"  Run {new_runs:03d} | shape={shape} | {scenario} | seed={seed} | "
        f"status={row['status']} | F1={row['f1']:.4f} | AUC={row['auc']:.4f} | "
        f"MDR={row['mdr']:.4f} | BFRR={row['bfrr']:.4f} | "
        f"AQSgap={row['aqs_gap']:.4f} | {row['elapsed_sec']:.1f}s",
        flush=True
    )

    if row["status"] == "error":
        print(f"    Error: {row.get('error_message', '')}", flush=True)

    gc.collect()
    if "torch" in globals() and torch.cuda.is_available():
        torch.cuda.empty_cache()

CELL_8_2_RAW = pd.DataFrame(rows)

print(f"\n[Cell 8.2] New runs executed : {new_runs}", flush=True)
print(f"[Cell 8.2] Raw results saved: {raw_path}", flush=True)


# -------------------------
# Summarize
# -------------------------

if len(CELL_8_2_RAW) == 0:
    CELL_8_2_SUMMARY = pd.DataFrame()
    CELL_8_2_COMPARISON = pd.DataFrame()
    print("[Cell 8.2] No runs available to summarize.", flush=True)

else:
    metric_cols = [
        "f1", "auc", "fpr", "asr", "mdr", "bfrr",
        "mean_aqs_benign", "mean_aqs_malicious", "aqs_gap",
        "runtime_sec", "elapsed_sec"
    ]

    for col in metric_cols:
        if col in CELL_8_2_RAW.columns:
            CELL_8_2_RAW[col] = pd.to_numeric(CELL_8_2_RAW[col], errors="coerce")

    ok_df = CELL_8_2_RAW[CELL_8_2_RAW["status"] == "ok"].copy()

    status_counts = (
        CELL_8_2_RAW
        .groupby(["dataset", "scenario", "aqs_shape", "status"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ["ok", "diverged", "error"]:
        if col not in status_counts.columns:
            status_counts[col] = 0

    if len(ok_df) > 0:
        CELL_8_2_SUMMARY = (
            ok_df
            .groupby(["dataset", "scenario", "aqs_shape"], dropna=False)
            .agg(
                runs=("f1", "count"),
                f1_mean=("f1", "mean"),
                f1_std=("f1", "std"),
                auc_mean=("auc", "mean"),
                auc_std=("auc", "std"),
                fpr_mean=("fpr", "mean"),
                asr_mean=("asr", "mean"),
                mdr_mean=("mdr", "mean"),
                bfrr_mean=("bfrr", "mean"),
                mean_aqs_benign=("mean_aqs_benign", "mean"),
                mean_aqs_malicious=("mean_aqs_malicious", "mean"),
                aqs_gap_mean=("aqs_gap", "mean"),
                runtime_mean=("runtime_sec", "mean"),
            )
            .reset_index()
        )

        CELL_8_2_SUMMARY = CELL_8_2_SUMMARY.merge(
            status_counts,
            on=["dataset", "scenario", "aqs_shape"],
            how="outer"
        )

        # Compare each shape against the exponential/default rule.
        base = CELL_8_2_SUMMARY[
            CELL_8_2_SUMMARY["aqs_shape"] == "exponential"
        ][
            ["dataset", "scenario", "f1_mean", "auc_mean", "mdr_mean", "bfrr_mean", "aqs_gap_mean"]
        ].rename(
            columns={
                "f1_mean": "exp_f1_mean",
                "auc_mean": "exp_auc_mean",
                "mdr_mean": "exp_mdr_mean",
                "bfrr_mean": "exp_bfrr_mean",
                "aqs_gap_mean": "exp_aqs_gap_mean",
            }
        )

        CELL_8_2_COMPARISON = CELL_8_2_SUMMARY.merge(
            base,
            on=["dataset", "scenario"],
            how="left"
        )

        CELL_8_2_COMPARISON["delta_f1_vs_exp"] = (
            CELL_8_2_COMPARISON["f1_mean"] - CELL_8_2_COMPARISON["exp_f1_mean"]
        )
        CELL_8_2_COMPARISON["delta_auc_vs_exp"] = (
            CELL_8_2_COMPARISON["auc_mean"] - CELL_8_2_COMPARISON["exp_auc_mean"]
        )
        CELL_8_2_COMPARISON["delta_mdr_vs_exp"] = (
            CELL_8_2_COMPARISON["mdr_mean"] - CELL_8_2_COMPARISON["exp_mdr_mean"]
        )
        CELL_8_2_COMPARISON["delta_bfrr_vs_exp"] = (
            CELL_8_2_COMPARISON["bfrr_mean"] - CELL_8_2_COMPARISON["exp_bfrr_mean"]
        )

    else:
        CELL_8_2_SUMMARY = status_counts.copy()
        CELL_8_2_COMPARISON = pd.DataFrame()

    CELL_8_2_SUMMARY.to_csv(summary_path, index=False)
    CELL_8_2_COMPARISON.to_csv(comparison_path, index=False)

    print("\n[Cell 8.2] AQS shape ablation summary:", flush=True)
    display_summary = CELL_8_2_SUMMARY.copy()

    for col in display_summary.columns:
        if col not in ["dataset", "scenario", "aqs_shape"]:
            display_summary[col] = pd.to_numeric(display_summary[col], errors="coerce").round(4)

    print(display_summary.to_string(index=False), flush=True)

    if len(CELL_8_2_COMPARISON) > 0:
        print("\n[Cell 8.2] Shape comparison against exponential/default AQS:", flush=True)

        compact_cols = [
            "dataset", "scenario", "aqs_shape", "runs",
            "f1_mean", "delta_f1_vs_exp",
            "auc_mean", "delta_auc_vs_exp",
            "mdr_mean", "delta_mdr_vs_exp",
            "bfrr_mean", "delta_bfrr_vs_exp",
            "aqs_gap_mean",
        ]
        compact_cols = [c for c in compact_cols if c in CELL_8_2_COMPARISON.columns]

        display_comp = CELL_8_2_COMPARISON[compact_cols].copy()
        for col in display_comp.columns:
            if col not in ["dataset", "scenario", "aqs_shape"]:
                display_comp[col] = pd.to_numeric(display_comp[col], errors="coerce").round(4)

        print(display_comp.to_string(index=False), flush=True)

        with open(latex_path, "w") as f:
            f.write(display_comp.to_latex(index=False, escape=False))
    else:
        with open(latex_path, "w") as f:
            f.write(display_summary.to_latex(index=False, escape=False))

    print(f"\nSaved summary    : {summary_path}", flush=True)
    print(f"Saved comparison : {comparison_path}", flush=True)
    print(f"Saved LaTeX      : {latex_path}", flush=True)


# -------------------------
# Finish
# -------------------------

c82_restore_aqs()

hard_errors = int((CELL_8_2_RAW["status"] == "error").sum()) if len(CELL_8_2_RAW) else 0

print(f"\n[Cell 8.2] Restored original compute_aqs()", flush=True)
print(f"[Cell 8.2] Elapsed time: {(time.time() - start_time) / 60.0:.2f} min", flush=True)
print("[Cell 8.2] Status:", "OK" if hard_errors == 0 else "CHECK_REQUIRED", flush=True)


In [ ]:
# ======================== 4.5: Reference-Set Size Sensitivity ========================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import copy
import gc

print("[Cell 8.5] Reference-set size sensitivity started", flush=True)

# -------------------------
# Settings
# -------------------------

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

CELL_8_5_DATASET = "CICIoT2023"
CELL_8_5_REF_SIZES = [100, 250, 500, 1000]
CELL_8_5_SEEDS = [11, 22, 33]

CELL_8_5_ATTACK = "recon_inflation"
CELL_8_5_ATTACK_SCALE = 5.0
CELL_8_5_BYZ_FRAC = 0.4
CELL_8_5_N_BYZ = int(round(CONFIG["n_clients"] * CELL_8_5_BYZ_FRAC))

CELL_8_5_FED_ROUNDS = 20
CELL_8_5_MAX_LOCAL_BATCHES = 20

CELL_8_5_ALPHA = 15.0
CELL_8_5_BETA = 0.0
CELL_8_5_TAU = 0.2

raw_path = results_dir / "cell_8_5_reference_size_sensitivity_raw.csv"
summary_path = results_dir / "cell_8_5_reference_size_sensitivity_summary.csv"
paper_path = results_dir / "cell_8_5_reference_size_sensitivity_paper.csv"
latex_path = results_dir / "cell_8_5_reference_size_sensitivity_latex.txt"

print(f"Dataset           : {CELL_8_5_DATASET}", flush=True)
print(f"Reference sizes   : {CELL_8_5_REF_SIZES}", flush=True)
print(f"Seeds             : {CELL_8_5_SEEDS}", flush=True)
print(f"Attack            : {CELL_8_5_ATTACK}", flush=True)
print(f"Attack scale      : {CELL_8_5_ATTACK_SCALE}", flush=True)
print(f"Byzantine fraction: {CELL_8_5_BYZ_FRAC}", flush=True)
print(f"Byzantine clients : {CELL_8_5_N_BYZ}/{CONFIG['n_clients']}", flush=True)
print(f"Fed rounds        : {CELL_8_5_FED_ROUNDS}", flush=True)
print(f"Max local batches : {CELL_8_5_MAX_LOCAL_BATCHES}", flush=True)

if CELL_8_5_DATASET not in DATA:
    raise ValueError(f"{CELL_8_5_DATASET} is not loaded in DATA.")

# -------------------------
# Backup state
# -------------------------

obj = DATA[CELL_8_5_DATASET]
feature_cols = obj["feature_cols"]

backup_data = {
    key: copy.deepcopy(obj.get(key))
    for key in [
        "train", "reference", "benign_train_count", "benign_ref_count",
        "iid_parts", "iid_loaders", "partition_summary"
    ]
    if key in obj
}

backup_config = {
    "seed": CONFIG.get("seed"),
    "alpha": CONFIG.get("alpha"),
    "beta": CONFIG.get("beta"),
    "tau": CONFIG.get("tau"),
    "recon_inflation_scale": CONFIG.get("recon_inflation_scale"),
}

# Build a fixed benign pool. For each |R|, reference is sampled from this pool,
# and the remaining samples are used for client training.
if "review_benign_pool" in obj and len(obj["review_benign_pool"]) > 0:
    benign_pool = obj["review_benign_pool"].copy().reset_index(drop=True)
else:
    benign_pool = pd.concat([obj["train"], obj["reference"]], axis=0).reset_index(drop=True)

if len(benign_pool) < max(CELL_8_5_REF_SIZES):
    raise ValueError(
        f"Benign pool has only {len(benign_pool)} samples, "
        f"but max |R| is {max(CELL_8_5_REF_SIZES)}."
    )

print(f"Benign pool size  : {len(benign_pool)}", flush=True)

# -------------------------
# Helper
# -------------------------

def prepare_reference_split(ref_size, seed):
    ref_df = benign_pool.sample(n=ref_size, random_state=int(seed))
    train_df = benign_pool.drop(index=ref_df.index).reset_index(drop=True)
    ref_df = ref_df.reset_index(drop=True)

    iid_parts = iid_partition(
        df=train_df,
        n_clients=CONFIG["n_clients"],
        seed=int(seed),
    )

    obj["train"] = train_df
    obj["reference"] = ref_df
    obj["benign_train_count"] = len(train_df)
    obj["benign_ref_count"] = len(ref_df)
    obj["iid_parts"] = iid_parts
    obj["iid_loaders"] = {
        cid: make_loader_from_df(
            iid_parts[cid],
            feature_cols,
            CONFIG["batch_size"],
            shuffle=True,
        )
        for cid in range(CONFIG["n_clients"])
    }


def result_to_row(result, ref_size, seed, runtime_sec, status="ok", error_message=""):
    metrics = result.get("final_metrics", {}) if result is not None else {}
    screening = result.get("screening", {}) if result is not None else {}
    aqs = result.get("aqs_summary", {}) if result is not None else {}

    return {
        "dataset": CELL_8_5_DATASET,
        "reference_size": ref_size,
        "seed": seed,
        "aggregator": "RAFA",
        "attack": CELL_8_5_ATTACK,
        "attack_scale": CELL_8_5_ATTACK_SCALE,
        "byzantine_fraction": CELL_8_5_BYZ_FRAC,
        "n_byzantine": CELL_8_5_N_BYZ,
        "alpha": CELL_8_5_ALPHA,
        "beta": CELL_8_5_BETA,
        "tau": CELL_8_5_TAU,
        "fed_rounds": CELL_8_5_FED_ROUNDS,
        "max_local_batches": CELL_8_5_MAX_LOCAL_BATCHES,
        "status": status,
        "error_message": error_message,
        "runtime_sec": runtime_sec,
        "f1": metrics.get("f1", np.nan),
        "auc": metrics.get("auc", np.nan),
        "fpr": metrics.get("fpr", np.nan),
        "asr": metrics.get("asr", np.nan),
        "mdr": screening.get("mdr", np.nan),
        "bfrr": screening.get("bfrr", np.nan),
        "mean_aqs_benign": aqs.get("mean_aqs_benign", np.nan),
        "mean_aqs_malicious": aqs.get("mean_aqs_malicious", np.nan),
        "aqs_gap": aqs.get("aqs_gap", np.nan),
    }


# -------------------------
# Run experiment
# -------------------------

rows = []
start_all = time.time()
total_runs = len(CELL_8_5_REF_SIZES) * len(CELL_8_5_SEEDS)
run_id = 0

try:
    for ref_size in CELL_8_5_REF_SIZES:
        for seed in CELL_8_5_SEEDS:
            run_id += 1
            run_start = time.time()

            try:
                set_global_seed(int(seed))
                CONFIG["seed"] = int(seed)
                CONFIG["alpha"] = CELL_8_5_ALPHA
                CONFIG["beta"] = CELL_8_5_BETA
                CONFIG["tau"] = CELL_8_5_TAU
                CONFIG["recon_inflation_scale"] = CELL_8_5_ATTACK_SCALE

                prepare_reference_split(ref_size, seed)

                result = run_federation({
                    "dataset_name": CELL_8_5_DATASET,
                    "aggregator": "RAFA",
                    "model_type": "VAE",
                    "partition": "iid",
                    "n_byzantine": CELL_8_5_N_BYZ,
                    "attack": CELL_8_5_ATTACK,
                    "fed_rounds": CELL_8_5_FED_ROUNDS,
                    "max_local_batches": CELL_8_5_MAX_LOCAL_BATCHES,
                })

                row = result_to_row(
                    result=result,
                    ref_size=ref_size,
                    seed=seed,
                    runtime_sec=time.time() - run_start,
                    status="ok",
                )

            except Exception as e:
                row = result_to_row(
                    result=None,
                    ref_size=ref_size,
                    seed=seed,
                    runtime_sec=time.time() - run_start,
                    status="error",
                    error_message=str(e),
                )

            rows.append(row)

            print(
                f"  Run {run_id:03d}/{total_runs:03d} | "
                f"|R|={ref_size:<4} | seed={seed} | status={row['status']} | "
                f"F1={row['f1']:.4f} | AUC={row['auc']:.4f} | "
                f"MDR={row['mdr']:.4f} | BFRR={row['bfrr']:.4f} | "
                f"AQSgap={row['aqs_gap']:.4f} | {row['runtime_sec']:.1f}s",
                flush=True,
            )

            if row["status"] == "error":
                print(f"    Error: {row['error_message']}", flush=True)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

finally:
    for key, value in backup_data.items():
        obj[key] = value

    for key, value in backup_config.items():
        if value is None:
            CONFIG.pop(key, None)
        else:
            CONFIG[key] = value

# -------------------------
# Save raw results
# -------------------------

raw_df = pd.DataFrame(rows)
raw_df.to_csv(raw_path, index=False)

ok_df = raw_df[raw_df["status"].eq("ok")].copy()

summary_df = (
    ok_df.groupby("reference_size", as_index=False)
    .agg(
        runs=("seed", "count"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        mdr_mean=("mdr", "mean"),
        mdr_std=("mdr", "std"),
        bfrr_mean=("bfrr", "mean"),
        bfrr_std=("bfrr", "std"),
        aqs_gap_mean=("aqs_gap", "mean"),
        aqs_gap_std=("aqs_gap", "std"),
        runtime_mean=("runtime_sec", "mean"),
    )
    .sort_values("reference_size")
    .reset_index(drop=True)
)

summary_df.to_csv(summary_path, index=False)

paper_df = summary_df[
    ["reference_size", "runs", "f1_mean", "f1_std", "mdr_mean", "bfrr_mean", "aqs_gap_mean"]
].copy()

paper_df.columns = [
    r"$|\mathcal{R}|$", "Runs", "F1", "F1 Std.", "MDR", "BFRR", "AQS Gap"
]

for col in ["F1", "F1 Std.", "MDR", "BFRR", "AQS Gap"]:
    paper_df[col] = paper_df[col].round(4)

paper_df.to_csv(paper_path, index=False)

# -------------------------
# LaTeX table
# -------------------------

latex_body = paper_df.to_latex(
    index=False,
    escape=False,
    column_format="ccccccc",
    float_format=lambda x: f"{x:.4f}",
)

latex_table = r"""\begin{table}[t]
\centering
\caption{Reference-set size sensitivity on CICIoT2023 under reconstruction-targeted attacks.}
\label{tab:ref_size_sensitivity}
\scriptsize
\setlength{\tabcolsep}{4.2pt}
""" + latex_body + r"""
\vspace{1mm}
\footnotesize The setting uses reconstruction-targeted attacks with scale 5.0 and $f/N=0.4$.
\end{table}
"""

latex_path.write_text(latex_table, encoding="utf-8")

# -------------------------
# Print summary
# -------------------------

print("\n[Cell 8.5] Reference-set size sensitivity summary:", flush=True)
print(summary_df.to_string(index=False), flush=True)

if len(summary_df) > 0:
    f1_range = summary_df["f1_mean"].max() - summary_df["f1_mean"].min()
    mdr_range = summary_df["mdr_mean"].max() - summary_df["mdr_mean"].min()
    bfrr_range = summary_df["bfrr_mean"].max() - summary_df["bfrr_mean"].min()

    print("\n[Cell 8.5] Interpretation:", flush=True)
    print(
        f"Across |R|={CELL_8_5_REF_SIZES}, mean F1 changes by {f1_range:.4f}, "
        f"MDR changes by {mdr_range:.4f}, and BFRR changes by {bfrr_range:.4f}.",
        flush=True,
    )

print("\n[Cell 8.5] Saved outputs:", flush=True)
print(f"  Raw        : {raw_path}", flush=True)
print(f"  Summary    : {summary_path}", flush=True)
print(f"  Paper table: {paper_path}", flush=True)
print(f"  LaTeX      : {latex_path}", flush=True)

print(f"\n[Cell 8.5] Elapsed time: {(time.time() - start_all) / 60:.2f} min", flush=True)
print("[Cell 8.5] Status: OK", flush=True)


In [ ]:
# ======================== 4.6: Runtime Overhead ========================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import json

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

print("[Cell 5.3] Runtime overhead analysis started")
print(f"Results dir: {results_dir}")


def load_csv_if_exists(path, name):
    if path.exists():
        df = pd.read_csv(path)
        print(f"  Loaded {name}: {len(df)} rows")
        return df
    print(f"  Missing {name}: {path}")
    return None


s1_df = load_csv_if_exists(results_dir / "s1_summary.csv", "S1 summary")
s2_df = load_csv_if_exists(results_dir / "s2_summary.csv", "S2 summary")
s3_df = load_csv_if_exists(results_dir / "s3_summary.csv", "S3 summary")
s4_df = load_csv_if_exists(results_dir / "s4_summary.csv", "S4 summary")
tradeoff_df = load_csv_if_exists(results_dir / "rafa_config_tradeoff_summary.csv", "RAFA config trade-off summary")


runtime_outputs = {}


# -------------------------
# Runtime from completed experiment summaries
# -------------------------

runtime_frames = []

for name, df in [
    ("S1", s1_df),
    ("S2", s2_df),
    ("S3", s3_df),
    ("S4", s4_df),
]:
    if df is not None and "runtime_sec" in df.columns:
        temp = df.copy()
        temp["experiment"] = name
        runtime_frames.append(temp)

if runtime_frames:
    runtime_all = pd.concat(runtime_frames, ignore_index=True)

    runtime_by_experiment = (
        runtime_all
        .groupby("experiment", as_index=False)
        .agg(
            runs=("runtime_sec", "count"),
            mean_runtime_sec=("runtime_sec", "mean"),
            std_runtime_sec=("runtime_sec", "std"),
            total_runtime_min=("runtime_sec", lambda x: x.sum() / 60.0),
        )
    )

    runtime_by_experiment["mean_runtime_sec"] = runtime_by_experiment["mean_runtime_sec"].round(2)
    runtime_by_experiment["std_runtime_sec"] = runtime_by_experiment["std_runtime_sec"].round(2)
    runtime_by_experiment["total_runtime_min"] = runtime_by_experiment["total_runtime_min"].round(2)

    runtime_exp_path = results_dir / "runtime_by_experiment.csv"
    runtime_by_experiment.to_csv(runtime_exp_path, index=False)
    runtime_outputs["runtime_by_experiment"] = str(runtime_exp_path)

    print("\nRuntime by experiment:")
    display(runtime_by_experiment)

    print("\nLaTeX runtime by experiment:")
    print(
        runtime_by_experiment.to_latex(
            index=False,
            escape=False,
            column_format="lrrrr",
            caption="Runtime summary by experiment.",
            label="tab:runtime_by_experiment",
        )
    )

    if "aggregator" in runtime_all.columns:
        runtime_by_aggregator = (
            runtime_all
            .dropna(subset=["aggregator"])
            .groupby("aggregator", as_index=False)
            .agg(
                runs=("runtime_sec", "count"),
                mean_runtime_sec=("runtime_sec", "mean"),
                std_runtime_sec=("runtime_sec", "std"),
            )
        )

        runtime_by_aggregator["mean_runtime_sec"] = runtime_by_aggregator["mean_runtime_sec"].round(2)
        runtime_by_aggregator["std_runtime_sec"] = runtime_by_aggregator["std_runtime_sec"].round(2)

        fedavg_mean = runtime_by_aggregator.loc[
            runtime_by_aggregator["aggregator"] == "FedAvg",
            "mean_runtime_sec",
        ]

        if len(fedavg_mean) > 0 and float(fedavg_mean.iloc[0]) > 0:
            base = float(fedavg_mean.iloc[0])
            runtime_by_aggregator["overhead_vs_fedavg"] = (
                runtime_by_aggregator["mean_runtime_sec"] / base
            ).round(3)
        else:
            runtime_by_aggregator["overhead_vs_fedavg"] = np.nan

        runtime_agg_path = results_dir / "runtime_by_aggregator.csv"
        runtime_by_aggregator.to_csv(runtime_agg_path, index=False)
        runtime_outputs["runtime_by_aggregator"] = str(runtime_agg_path)

        print("\nRuntime by aggregator:")
        display(runtime_by_aggregator)

        print("\nLaTeX runtime by aggregator:")
        print(
            runtime_by_aggregator.to_latex(
                index=False,
                escape=False,
                column_format="lrrrr",
                caption="Runtime summary by aggregation method.",
                label="tab:runtime_by_aggregator",
            )
        )


# -------------------------
# Cell 4.3 runtime summary
# -------------------------

if tradeoff_df is not None and "runtime_sec" in tradeoff_df.columns:
    tradeoff_runtime = (
        tradeoff_df
        .groupby("config_name", as_index=False)
        .agg(
            runs=("runtime_sec", "count"),
            mean_runtime_sec=("runtime_sec", "mean"),
            std_runtime_sec=("runtime_sec", "std"),
            total_runtime_min=("runtime_sec", lambda x: x.sum() / 60.0),
        )
    )

    tradeoff_runtime["mean_runtime_sec"] = tradeoff_runtime["mean_runtime_sec"].round(2)
    tradeoff_runtime["std_runtime_sec"] = tradeoff_runtime["std_runtime_sec"].round(2)
    tradeoff_runtime["total_runtime_min"] = tradeoff_runtime["total_runtime_min"].round(2)

    tradeoff_runtime_path = results_dir / "runtime_rafa_config_tradeoff.csv"
    tradeoff_runtime.to_csv(tradeoff_runtime_path, index=False)
    runtime_outputs["runtime_rafa_config_tradeoff"] = str(tradeoff_runtime_path)

    print("\nCell 4.3 RAFA configuration trade-off runtime:")
    display(tradeoff_runtime)

    print("\nLaTeX runtime table for Cell 4.3:")
    print(
        tradeoff_runtime.to_latex(
            index=False,
            escape=False,
            column_format="lrrrr",
            caption="Runtime summary for the RAFA configuration trade-off experiment.",
            label="tab:rafa_config_tradeoff_runtime",
        )
    )


# -------------------------
# Optional direct runtime overhead micro-benchmark
# -------------------------

RUN_RUNTIME_MICROBENCHMARK = False

if RUN_RUNTIME_MICROBENCHMARK:
    print("\nRunning direct runtime micro-benchmark...")

    runtime_rows = []
    dataset_name = ACTIVE_DATASETS[0]
    benchmark_rounds = 10

    for aggregator in AGGREGATORS:
        try:
            start = time.time()

            result = run_federation({
                "dataset_name": dataset_name,
                "aggregator": aggregator,
                "model_type": "VAE",
                "partition": "iid",
                "n_byzantine": 0,
                "attack": "none",
                "fed_rounds": benchmark_rounds,
                "max_local_batches": CONFIG.get("max_local_batches", 20),
            })

            total_sec = time.time() - start

            runtime_rows.append({
                "aggregator": aggregator,
                "rounds": benchmark_rounds,
                "total_runtime_sec": total_sec,
                "mean_runtime_per_round_sec": total_sec / benchmark_rounds,
                "final_f1": result["final_metrics"]["f1"],
            })

            print(
                f"  {aggregator}: total={total_sec:.2f}s, "
                f"per_round={total_sec / benchmark_rounds:.2f}s"
            )

        except Exception as e:
            runtime_rows.append({
                "aggregator": aggregator,
                "rounds": benchmark_rounds,
                "total_runtime_sec": np.nan,
                "mean_runtime_per_round_sec": np.nan,
                "final_f1": np.nan,
                "error": str(e),
            })

            print(f"  ERROR {aggregator}: {e}")

    runtime_micro_df = pd.DataFrame(runtime_rows)

    fedavg_runtime = runtime_micro_df.loc[
        runtime_micro_df["aggregator"] == "FedAvg",
        "mean_runtime_per_round_sec",
    ]

    if len(fedavg_runtime) > 0 and np.isfinite(fedavg_runtime.iloc[0]) and fedavg_runtime.iloc[0] > 0:
        base = fedavg_runtime.iloc[0]
        runtime_micro_df["overhead_vs_fedavg"] = (
            runtime_micro_df["mean_runtime_per_round_sec"] / base
        ).round(3)
    else:
        runtime_micro_df["overhead_vs_fedavg"] = np.nan

    runtime_micro_path = results_dir / "runtime_microbenchmark.csv"
    runtime_micro_df.to_csv(runtime_micro_path, index=False)
    runtime_outputs["runtime_microbenchmark"] = str(runtime_micro_path)

    print("\nDirect runtime micro-benchmark:")
    display(runtime_micro_df)


# -------------------------
# Save runtime manifest
# -------------------------

runtime_manifest_path = results_dir / "runtime_manifest.json"

with open(runtime_manifest_path, "w") as f:
    json.dump(runtime_outputs, f, indent=2)

print("\n[Cell 5.3] Status: OK")
print(f"Runtime outputs: {len(runtime_outputs)}")
print(f"Manifest       : {runtime_manifest_path}")

for name, path in runtime_outputs.items():
    print(f"  - {name}: {path}")


## 5. Figures

In [ ]:

# ======================== 6.2: Paper Figures from Generated Result CSV Files ========================
#
# This cell creates lightweight versions of the main result figures.
# Figure style may differ from the submitted manuscript, but the plotted values
# are read from the same result CSV files generated above.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

results_dir = Path(CONFIG.get("results_dir", "results"))
fig_dir = results_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)


def load_csv_if_exists(path, name):
    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded {name}: {len(df)} rows")
        return df
    print(f"Missing {name}: {path}")
    return pd.DataFrame()


s1_df = load_csv_if_exists(results_dir / "s1_summary.csv", "S1 reconstruction-targeted summary")
s2_df = load_csv_if_exists(results_dir / "s2_summary.csv", "S2 untargeted summary")

# Figure 1: RAFA AQS gap under reconstruction-targeted attacks.
if len(s1_df) > 0:
    rafa_s1 = s1_df[s1_df["aggregator"] == "RAFA"].copy()
    for col in ["attack_scale", "byzantine_fraction", "aqs_gap"]:
        rafa_s1[col] = pd.to_numeric(rafa_s1[col], errors="coerce")

    fig, ax = plt.subplots(figsize=(6, 4))
    for frac, group in rafa_s1.groupby("byzantine_fraction"):
        group = group.sort_values("attack_scale")
        ax.plot(group["attack_scale"], group["aqs_gap"], marker="o", label=f"f={frac:.1f}")

    ax.set_xlabel("Reconstruction attack scale")
    ax.set_ylabel("AQS gap")
    ax.set_title("RAFA AQS separation under reconstruction-targeted attacks")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    out_path = fig_dir / "fig_s1_rafa_aqs_gap.pdf"
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

# Figure 2: F1 under sign-flip attacks.
if len(s2_df) > 0:
    sign_df = s2_df[s2_df["attack"] == "sign_flip"].copy()
    for col in ["byzantine_fraction", "f1"]:
        sign_df[col] = pd.to_numeric(sign_df[col], errors="coerce")

    fig, ax = plt.subplots(figsize=(6, 4))
    for aggregator, group in sign_df.groupby("aggregator"):
        group = group.sort_values("byzantine_fraction")
        ax.plot(group["byzantine_fraction"], group["f1"], marker="o", label=aggregator)

    ax.set_xlabel("Byzantine fraction")
    ax.set_ylabel("F1")
    ax.set_title("S2 sign-flip attack performance")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    out_path = fig_dir / "fig_s2_signflip_f1.pdf"
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")


## 6. Cross-Dataset CICDDoS2019 Evaluation

The paper reports CICDDoS2019 results from two selected source files: `MSSQL.csv` and `DrDoS_DNS.csv`. The dataset is not redistributed here. Put these files under `data/CICDDoS2019/`.

In [ ]:
# ======================== 7.1: CICDDoS2019 Setup for Paper Subsets ========================

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

print("[Cell 7.1] CICDDoS2019 memory-safe setup started", flush=True)

# -------------------------
# Path and compact settings
# -------------------------

CICDDOS_ROOT_CANDIDATES = [
    DATA_DIR / "CICDDoS2019",
    NOTEBOOK_DIR / "CICDDoS2019",
]

CICDDOS_ROOT = next((p.resolve() for p in CICDDOS_ROOT_CANDIDATES if p.exists()), None)

if CICDDOS_ROOT is None:
    raise FileNotFoundError(
        "CICDDoS2019 folder not found. Checked: "
        + ", ".join(str(p) for p in CICDDOS_ROOT_CANDIDATES)
    )

CICDDOS_FILES = [
    "MSSQL.csv",
    "DrDoS_DNS.csv",
]

CICDDOS_PREFERRED_FILES = [
    "MSSQL.csv",
    "DrDoS_DNS.csv",
]

# Keep this conservative first. Increase later only after Cell 7.1 is stable.
CICDDOS_MAX_SELECTED = int(CONFIG.get("cicddos_max_selected_subsets", 2))

CICDDOS_MIN_BENIGN = int(CONFIG.get("cicddos_min_benign_samples", 2000))
CICDDOS_MIN_ATTACK = int(CONFIG.get("cicddos_min_attack_samples", 2000))

# Memory-safe row caps per selected CSV.
CICDDOS_MAX_BENIGN_ROWS = int(CONFIG.get("cicddos_max_benign_rows", 120000))
CICDDOS_MAX_ATTACK_ROWS = int(CONFIG.get("cicddos_max_attack_rows", 300000))
CICDDOS_CHUNKSIZE = int(CONFIG.get("cicddos_chunksize", 100000))

CICDDOS_DROP_COLS = {
    "Unnamed: 0", "Flow ID", "Source IP", "Destination IP",
    "Source Port", "Destination Port", "Timestamp", "SimillarHTTP"
}

print(f"Dataset root       : {CICDDOS_ROOT}", flush=True)
print(f"Max selected files : {CICDDOS_MAX_SELECTED}", flush=True)
print(f"Benign row cap     : {CICDDOS_MAX_BENIGN_ROWS:,}", flush=True)
print(f"Attack row cap     : {CICDDOS_MAX_ATTACK_ROWS:,}", flush=True)
print(f"Chunk size         : {CICDDOS_CHUNKSIZE:,}", flush=True)


# -------------------------
# Helpers
# -------------------------

def _strip_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def _find_label_col(columns):
    for col in columns:
        if str(col).strip().lower() == "label":
            return col
    raise ValueError("No Label column found.")


def _safe_dataset_name(csv_name):
    stem = Path(csv_name).stem.replace("DrDoS_", "")
    return f"CICDDoS2019_{stem}"


def _raw_col_map(path):
    header_raw = pd.read_csv(path, nrows=0, low_memory=True)
    return {str(c).strip(): c for c in header_raw.columns}


def _scan_cicddos_file(csv_name):
    path = CICDDOS_ROOT / csv_name

    if not path.exists():
        return {
            "file": csv_name,
            "path": str(path),
            "exists": False,
            "usable": False,
            "reason": "missing_file",
        }

    header_raw = pd.read_csv(path, nrows=0, low_memory=True)
    raw_label_col = _find_label_col(header_raw.columns)

    sample = pd.read_csv(path, nrows=2000, low_memory=True)
    sample = _strip_columns(sample)

    label_col = "Label"

    candidate_cols = [
        c for c in sample.columns
        if c != label_col and c not in CICDDOS_DROP_COLS
    ]

    numeric_cols = []
    for col in candidate_cols:
        converted = pd.to_numeric(sample[col], errors="coerce")
        if converted.notna().mean() > 0.95:
            numeric_cols.append(col)

    benign_count = 0
    attack_count = 0
    attack_labels = set()
    total_rows = 0

    for chunk in pd.read_csv(
        path,
        usecols=[raw_label_col],
        chunksize=CICDDOS_CHUNKSIZE,
        low_memory=True,
    ):
        chunk = _strip_columns(chunk)
        labels = chunk[label_col].astype(str).str.strip()
        benign_mask = labels.str.lower().eq("benign")

        benign_count += int(benign_mask.sum())
        attack_count += int((~benign_mask).sum())
        total_rows += len(labels)

        attack_labels.update(labels.loc[~benign_mask].dropna().unique().tolist())

    usable = (
        benign_count >= CICDDOS_MIN_BENIGN
        and attack_count >= CICDDOS_MIN_ATTACK
        and len(numeric_cols) > 0
    )

    if benign_count < CICDDOS_MIN_BENIGN:
        reason = "too_few_benign"
    elif attack_count < CICDDOS_MIN_ATTACK:
        reason = "too_few_attack"
    elif len(numeric_cols) == 0:
        reason = "no_numeric_features"
    else:
        reason = "selected_candidate"

    return {
        "file": csv_name,
        "path": str(path),
        "exists": True,
        "usable": usable,
        "reason": reason,
        "raw_label_col": raw_label_col,
        "label_col": label_col,
        "rows": int(total_rows),
        "benign": int(benign_count),
        "attack": int(attack_count),
        "attack_labels": sorted(list(attack_labels)),
        "numeric_feature_count": int(len(numeric_cols)),
        "numeric_cols": numeric_cols,
    }


def _load_capped_cicddos_frame(scan_info, feature_cols):
    path = Path(scan_info["path"])
    raw_label_col = scan_info["raw_label_col"]
    label_col = scan_info["label_col"]

    col_map = _raw_col_map(path)
    raw_feature_cols = [col_map[c] for c in feature_cols if c in col_map]
    usecols = raw_feature_cols + [raw_label_col]

    benign_chunks = []
    attack_chunks = []
    benign_seen = 0
    attack_seen = 0

    for chunk in pd.read_csv(
        path,
        usecols=usecols,
        chunksize=CICDDOS_CHUNKSIZE,
        low_memory=True,
    ):
        chunk = _strip_columns(chunk)

        labels = chunk[label_col].astype(str).str.strip()
        benign_mask = labels.str.lower().eq("benign")

        if benign_seen < CICDDOS_MAX_BENIGN_ROWS:
            take_benign = chunk.loc[benign_mask]
            remaining = CICDDOS_MAX_BENIGN_ROWS - benign_seen
            if len(take_benign) > remaining:
                take_benign = take_benign.sample(
                    n=remaining,
                    random_state=CONFIG["seed"] + benign_seen,
                )
            if len(take_benign) > 0:
                benign_chunks.append(take_benign)
                benign_seen += len(take_benign)

        if attack_seen < CICDDOS_MAX_ATTACK_ROWS:
            take_attack = chunk.loc[~benign_mask]
            remaining = CICDDOS_MAX_ATTACK_ROWS - attack_seen
            if len(take_attack) > remaining:
                take_attack = take_attack.sample(
                    n=remaining,
                    random_state=CONFIG["seed"] + attack_seen,
                )
            if len(take_attack) > 0:
                attack_chunks.append(take_attack)
                attack_seen += len(take_attack)

        if benign_seen >= CICDDOS_MAX_BENIGN_ROWS and attack_seen >= CICDDOS_MAX_ATTACK_ROWS:
            break

    if not benign_chunks:
        raise ValueError(f"No benign rows loaded from {scan_info['file']}")

    if not attack_chunks:
        raise ValueError(f"No attack rows loaded from {scan_info['file']}")

    benign_df = pd.concat(benign_chunks, axis=0).reset_index(drop=True)
    attack_df = pd.concat(attack_chunks, axis=0).reset_index(drop=True)

    df = pd.concat([benign_df, attack_df], axis=0).reset_index(drop=True)

    return df, len(benign_df), len(attack_df)


def _prepare_cicddos_subset(scan_info, feature_cols):
    dataset_name = _safe_dataset_name(scan_info["file"])
    label_col = scan_info["label_col"]

    df_raw, loaded_benign, loaded_attack = _load_capped_cicddos_frame(scan_info, feature_cols)

    raw_rows = len(df_raw)

    for col in feature_cols:
        df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

    labels = df_raw[label_col].astype(str).str.strip()
    df_raw["binary_label"] = (~labels.str.lower().eq("benign")).astype(int)
    df_raw["attack_category"] = np.where(df_raw["binary_label"] == 0, "Benign", labels)

    before_drop = len(df_raw)
    df = (
        df_raw.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=feature_cols + ["binary_label"])
        .reset_index(drop=True)
    )
    row_drops = before_drop - len(df)

    benign_df = df[df["binary_label"] == 0].copy().reset_index(drop=True)
    attack_df = df[df["binary_label"] == 1].copy().reset_index(drop=True)

    if len(benign_df) < CICDDOS_MIN_BENIGN:
        raise ValueError(f"Too few clean benign rows after loading: {len(benign_df)}")

    if len(attack_df) < CICDDOS_MIN_ATTACK:
        raise ValueError(f"Too few clean attack rows after loading: {len(attack_df)}")

    benign_train_pool, benign_temp = train_test_split(
        benign_df,
        test_size=0.30,
        random_state=CONFIG["seed"],
        shuffle=True,
    )

    benign_val, benign_test = train_test_split(
        benign_temp,
        test_size=0.50,
        random_state=CONFIG["seed"],
        shuffle=True,
    )

    benign_train_pool = benign_train_pool.reset_index(drop=True)
    benign_val = benign_val.reset_index(drop=True)
    benign_test = benign_test.reset_index(drop=True)

    ref_size = min(CONFIG["default_reference_size"], len(benign_train_pool))
    reference_df = benign_train_pool.sample(n=ref_size, random_state=CONFIG["seed"])
    train_df = benign_train_pool.drop(index=reference_df.index).reset_index(drop=True)
    reference_df = reference_df.reset_index(drop=True)

    test_df = pd.concat([benign_test, attack_df], axis=0).reset_index(drop=True)

    scaler = MinMaxScaler()
    scaler.fit(train_df[feature_cols])

    def apply_scaler(split_df):
        out = split_df.copy()
        out[feature_cols] = scaler.transform(out[feature_cols])
        return out.reset_index(drop=True)

    train_df = apply_scaler(train_df)
    reference_df = apply_scaler(reference_df)
    benign_val = apply_scaler(benign_val)
    test_df = apply_scaler(test_df)

    train_min = float(np.nanmin(train_df[feature_cols].to_numpy()))
    train_max = float(np.nanmax(train_df[feature_cols].to_numpy()))
    test_min = float(np.nanmin(test_df[feature_cols].to_numpy()))
    test_max = float(np.nanmax(test_df[feature_cols].to_numpy()))

    attack_counts = (
        test_df[test_df["binary_label"] == 1]["attack_category"]
        .value_counts()
        .to_dict()
    )

    return {
        "name": dataset_name,
        "path": scan_info["path"],
        "label_col": label_col,
        "raw_rows": raw_rows,
        "clean_rows": len(df),
        "loaded_benign_rows": loaded_benign,
        "loaded_attack_rows": loaded_attack,
        "feature_cols": feature_cols,
        "n_features": len(feature_cols),
        "expected_features": None,
        "forced_drop_cols": sorted(CICDDOS_DROP_COLS),
        "heavy_nan_cols": [],
        "non_numeric_cols": [],
        "excluded_cols": sorted(CICDDOS_DROP_COLS),
        "row_drops": row_drops,
        "scaler": scaler,
        "train": train_df,
        "reference": reference_df,
        "val": benign_val,
        "test": test_df,
        "attack_counts": attack_counts,
        "benign_train_count": len(train_df),
        "benign_ref_count": len(reference_df),
        "benign_val_count": len(benign_val),
        "benign_test_count": int((test_df["binary_label"] == 0).sum()),
        "attack_test_count": int((test_df["binary_label"] == 1).sum()),
        "scaled_train_min": train_min,
        "scaled_train_max": train_max,
        "scaled_test_min": test_min,
        "scaled_test_max": test_max,
    }


def _add_partitions_and_models(dataset_name):
    obj = DATA[dataset_name]
    feature_cols = obj["feature_cols"]

    iid_parts = iid_partition(
        df=obj["train"],
        n_clients=CONFIG["n_clients"],
        seed=CONFIG["seed"],
    )

    noniid_parts, noniid_key_col = balanced_shard_noniid_partition(
        df=obj["train"],
        feature_cols=feature_cols,
        n_clients=CONFIG["n_clients"],
        seed=CONFIG["seed"],
        shards_per_client=3,
        n_bins=CONFIG["n_clients"],
    )

    obj["iid_parts"] = iid_parts
    obj["noniid_parts"] = noniid_parts
    obj["iid_loaders"] = {
        cid: make_loader_from_df(part, feature_cols, CONFIG["batch_size"], shuffle=True)
        for cid, part in iid_parts.items()
    }
    obj["noniid_loaders"] = {
        cid: make_loader_from_df(part, feature_cols, CONFIG["batch_size"], shuffle=True)
        for cid, part in noniid_parts.items()
    }
    obj["partition_summary"] = {
        "iid": summarize_partitions(iid_parts),
        "noniid": summarize_partitions(noniid_parts),
        "noniid_key_col": noniid_key_col,
        "noniid_heterogeneity": summarize_noniid_heterogeneity(
            noniid_parts,
            noniid_key_col,
            n_bins=CONFIG["n_clients"],
        ),
    }

    input_dim = len(feature_cols)

    ae = SimpleAE(
        input_dim=input_dim,
        latent_dim=CONFIG["latent_dim"],
        dropout=CONFIG["dropout"],
    ).to(DEVICE)

    vae = SimpleVAE(
        input_dim=input_dim,
        latent_dim=CONFIG["latent_dim"],
        dropout=CONFIG["dropout"],
    ).to(DEVICE)

    MODEL_REGISTRY[dataset_name] = {
        "input_dim": input_dim,
        "AE": ae,
        "VAE": vae,
        "AE_params": count_parameters(ae),
        "VAE_params": count_parameters(vae),
    }


# -------------------------
# Scan, select, prepare
# -------------------------

scan_rows = []

for csv_name in CICDDOS_FILES:
    try:
        info = _scan_cicddos_file(csv_name)
        scan_rows.append(info)

        if info.get("exists", False):
            print(
                f"  Scanned {csv_name}: "
                f"benign={info.get('benign', 0):,}, "
                f"attack={info.get('attack', 0):,}, "
                f"features={info.get('numeric_feature_count', 0)}, "
                f"usable={info.get('usable', False)}",
                flush=True,
            )
        else:
            print(f"  Missing {csv_name}", flush=True)

    except Exception as e:
        scan_rows.append({
            "file": csv_name,
            "path": str(CICDDOS_ROOT / csv_name),
            "exists": True,
            "usable": False,
            "reason": f"scan_error: {e}",
        })
        print(f"  Scan error {csv_name}: {e}", flush=True)

CICDDOS_SCAN_DF = pd.DataFrame([
    {k: v for k, v in row.items() if k not in {"numeric_cols"}}
    for row in scan_rows
])

usable_infos = [row for row in scan_rows if row.get("usable", False)]

preferred_rank = {name: rank for rank, name in enumerate(CICDDOS_PREFERRED_FILES)}

usable_infos = sorted(
    usable_infos,
    key=lambda x: (
        preferred_rank.get(x["file"], len(preferred_rank)),
        -x["benign"],
        -x["attack"],
    )
)

selected_infos = usable_infos[:CICDDOS_MAX_SELECTED]

if not selected_infos:
    raise ValueError("No usable CICDDoS2019 subset found after scanning.")

common_features = sorted(
    set(selected_infos[0]["numeric_cols"]).intersection(
        *[set(info["numeric_cols"]) for info in selected_infos]
    )
)

if not common_features:
    raise ValueError("Selected CICDDoS2019 subsets have no common numeric features.")

CICDDOS_SELECTED_DATASETS = []
CICDDOS_LOAD_ERRORS = {}

for info in selected_infos:
    dataset_name = _safe_dataset_name(info["file"])

    try:
        print(f"\n  Loading capped subset {dataset_name} from {info['file']}...", flush=True)

        DATA[dataset_name] = _prepare_cicddos_subset(info, common_features)
        _add_partitions_and_models(dataset_name)

        CICDDOS_SELECTED_DATASETS.append(dataset_name)

        obj = DATA[dataset_name]
        print(
            f"  Loaded {dataset_name}: "
            f"train={obj['benign_train_count']:,}, "
            f"ref={obj['benign_ref_count']:,}, "
            f"val={obj['benign_val_count']:,}, "
            f"test={len(obj['test']):,}, "
            f"features={obj['n_features']}",
            flush=True,
        )

    except Exception as e:
        CICDDOS_LOAD_ERRORS[dataset_name] = str(e)
        print(f"  Load error {dataset_name}: {e}", flush=True)

RAFA_REVIEW_DATASETS = [
    name for name in ["CICIoT2023"] + CICDDOS_SELECTED_DATASETS
    if name in DATA
]

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

CICDDOS_SCAN_DF.to_csv(results_dir / "cell_7_1_cicddos_scan.csv", index=False)

pd.DataFrame({
    "dataset": CICDDOS_SELECTED_DATASETS,
    "source_file": [
        DATA[name]["path"] if name in DATA else ""
        for name in CICDDOS_SELECTED_DATASETS
    ],
    "n_features": [
        DATA[name]["n_features"] if name in DATA else np.nan
        for name in CICDDOS_SELECTED_DATASETS
    ],
    "benign_train": [
        DATA[name]["benign_train_count"] if name in DATA else np.nan
        for name in CICDDOS_SELECTED_DATASETS
    ],
    "benign_ref": [
        DATA[name]["benign_ref_count"] if name in DATA else np.nan
        for name in CICDDOS_SELECTED_DATASETS
    ],
    "benign_val": [
        DATA[name]["benign_val_count"] if name in DATA else np.nan
        for name in CICDDOS_SELECTED_DATASETS
    ],
    "benign_test": [
        DATA[name]["benign_test_count"] if name in DATA else np.nan
        for name in CICDDOS_SELECTED_DATASETS
    ],
    "attack_test": [
        DATA[name]["attack_test_count"] if name in DATA else np.nan
        for name in CICDDOS_SELECTED_DATASETS
    ],
}).to_csv(results_dir / "cell_7_1_cicddos_selected.csv", index=False)


# -------------------------
# Short dynamic diagnostics
# -------------------------

print(f"\n[Cell 7.1] Scan complete", flush=True)
print(f"Files found       : {int(CICDDOS_SCAN_DF['exists'].sum())}/{len(CICDDOS_FILES)}", flush=True)
print(f"Usable candidates : {len(usable_infos)}", flush=True)
print(f"Common features   : {len(common_features)}", flush=True)
print(f"Loaded subsets    : {len(CICDDOS_SELECTED_DATASETS)}", flush=True)
print(f"Review datasets   : {RAFA_REVIEW_DATASETS}", flush=True)

print("\nSelected CICDDoS2019 subsets:", flush=True)

for name in CICDDOS_SELECTED_DATASETS:
    obj = DATA[name]
    ps = obj["partition_summary"]

    print(f"\n{name}", flush=True)
    print(f"  Features  : {obj['n_features']}", flush=True)
    print(
        f"  Loaded    : benign={obj['loaded_benign_rows']:,}, "
        f"attack={obj['loaded_attack_rows']:,}, dropped={obj['row_drops']:,}",
        flush=True,
    )
    print(
        f"  Splits    : train={obj['benign_train_count']:,}, "
        f"ref={obj['benign_ref_count']:,}, val={obj['benign_val_count']:,}, "
        f"test={len(obj['test']):,}",
        flush=True,
    )
    print(
        f"  Test mix  : benign={obj['benign_test_count']:,}, "
        f"attack={obj['attack_test_count']:,}",
        flush=True,
    )
    print(f"  Attacks   : {obj['attack_counts']}", flush=True)
    print(
        f"  IID       : min={ps['iid']['min']:,}, max={ps['iid']['max']:,}, "
        f"empty={ps['iid']['empty_clients']}",
        flush=True,
    )
    print(
        f"  Non-IID   : key={ps['noniid_key_col']}, "
        f"empty={ps['noniid']['empty_clients']}",
        flush=True,
    )
    print(
        f"  Model     : input_dim={MODEL_REGISTRY[name]['input_dim']}, "
        f"VAE_params={MODEL_REGISTRY[name]['VAE_params']:,}",
        flush=True,
    )

if CICDDOS_LOAD_ERRORS:
    print("\nLoad errors:", flush=True)
    for name, msg in CICDDOS_LOAD_ERRORS.items():
        print(f"  - {name}: {msg}", flush=True)

print("\n[Cell 7.1] Status:", "OK" if not CICDDOS_LOAD_ERRORS else "CHECK_REQUIRED", flush=True)


In [ ]:
# ======================== 7.2: Helper Functions for Cross-Dataset Evaluation ========================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import json
import pickle
import gc

print("[Cell 7.2] Review dataset selection and repeated-seed runner setup started", flush=True)

# -------------------------
# User-selected datasets
# -------------------------

RAFA_REQUESTED_CICDDOS_FILES = [
    "MSSQL.csv",
    "DrDoS_DNS.csv",
]

RAFA_REQUESTED_CICDDOS_DATASETS = [
    _safe_dataset_name(csv_name)
    for csv_name in RAFA_REQUESTED_CICDDOS_FILES
]

RAFA_KEEP_DATASETS = ["CICIoT2023"] + RAFA_REQUESTED_CICDDOS_DATASETS

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Requested CIC-DDoS2019 files : {RAFA_REQUESTED_CICDDOS_FILES}", flush=True)
print(f"Requested review datasets    : {RAFA_KEEP_DATASETS}", flush=True)


# -------------------------
# Helper: free unused CIC-DDoS2019 datasets
# -------------------------

def _remove_unused_review_datasets(keep_datasets):
    removed = []

    for dataset_name in list(DATA.keys()):
        if dataset_name.startswith("CICDDoS2019_") and dataset_name not in keep_datasets:
            DATA.pop(dataset_name, None)
            MODEL_REGISTRY.pop(dataset_name, None)
            removed.append(dataset_name)

    gc.collect()

    if "torch" in globals() and torch.cuda.is_available():
        torch.cuda.empty_cache()

    return removed


removed_datasets = _remove_unused_review_datasets(RAFA_KEEP_DATASETS)

if removed_datasets:
    print(f"Removed unused datasets       : {removed_datasets}", flush=True)
else:
    print("Removed unused datasets       : none", flush=True)


# -------------------------
# Helper: find scan info
# -------------------------

def _get_scan_info_by_file(csv_name):
    matches = [
        row for row in scan_rows
        if row.get("file") == csv_name and row.get("usable", False)
    ]

    if not matches:
        raise ValueError(f"No usable scan info found for {csv_name}. Check Cell 7.1 scan output.")

    return matches[0]


requested_scan_infos = [
    _get_scan_info_by_file(csv_name)
    for csv_name in RAFA_REQUESTED_CICDDOS_FILES
]

requested_common_features = sorted(
    set(requested_scan_infos[0]["numeric_cols"]).intersection(
        *[set(info["numeric_cols"]) for info in requested_scan_infos]
    )
)

if not requested_common_features:
    raise ValueError("Requested CIC-DDoS2019 datasets have no common numeric features.")

print(f"Requested common features     : {len(requested_common_features)}", flush=True)


# -------------------------
# Load missing requested CIC-DDoS2019 datasets
# -------------------------

CICDDOS_REVIEW_LOAD_ERRORS = {}

for info in requested_scan_infos:
    dataset_name = _safe_dataset_name(info["file"])

    if dataset_name in DATA:
        existing_features = DATA[dataset_name]["feature_cols"]

        if set(existing_features) != set(requested_common_features):
            print(f"Reloading {dataset_name}: feature set differs from requested common features", flush=True)
            DATA.pop(dataset_name, None)
            MODEL_REGISTRY.pop(dataset_name, None)
        else:
            print(f"Already loaded               : {dataset_name}", flush=True)
            continue

    try:
        print(f"Loading requested subset      : {dataset_name} from {info['file']}", flush=True)

        DATA[dataset_name] = _prepare_cicddos_subset(info, requested_common_features)
        _add_partitions_and_models(dataset_name)

        obj = DATA[dataset_name]
        print(
            f"  Loaded {dataset_name}: "
            f"train={obj['benign_train_count']:,}, "
            f"ref={obj['benign_ref_count']:,}, "
            f"val={obj['benign_val_count']:,}, "
            f"test={len(obj['test']):,}, "
            f"features={obj['n_features']}",
            flush=True,
        )

    except Exception as e:
        CICDDOS_REVIEW_LOAD_ERRORS[dataset_name] = str(e)
        print(f"  Load error {dataset_name}: {e}", flush=True)


RAFA_REVIEW_DATASETS = [
    dataset_name for dataset_name in RAFA_KEEP_DATASETS
    if dataset_name in DATA
]

if len(RAFA_REVIEW_DATASETS) < len(RAFA_KEEP_DATASETS):
    missing = sorted(set(RAFA_KEEP_DATASETS) - set(RAFA_REVIEW_DATASETS))
    print(f"Missing review datasets       : {missing}", flush=True)

if CICDDOS_REVIEW_LOAD_ERRORS:
    print(f"Load errors                   : {CICDDOS_REVIEW_LOAD_ERRORS}", flush=True)


# -------------------------
# Repeated-seed preparation
# -------------------------

RAFA_REVIEW_SEEDS = CONFIG.get("review_seeds", [11, 22, 33])
RAFA_REVIEW_SEEDS = [int(s) for s in RAFA_REVIEW_SEEDS]

RAFA_REVIEW_MAX_LOCAL_BATCHES = int(CONFIG.get("review_max_local_batches", 20))
RAFA_REVIEW_FED_ROUNDS = int(CONFIG.get("review_fed_rounds", CONFIG["fed_rounds"]))

print(f"Review seeds                 : {RAFA_REVIEW_SEEDS}", flush=True)
print(f"Review fed rounds            : {RAFA_REVIEW_FED_ROUNDS}", flush=True)
print(f"Review max local batches     : {RAFA_REVIEW_MAX_LOCAL_BATCHES}", flush=True)


def _init_review_benign_pool(dataset_name):
    obj = DATA[dataset_name]

    if "review_benign_pool" not in obj:
        obj["review_benign_pool"] = pd.concat(
            [obj["train"], obj["reference"]],
            axis=0,
        ).reset_index(drop=True)

    if "review_reference_size" not in obj:
        obj["review_reference_size"] = int(len(obj["reference"]))


for dataset_name in RAFA_REVIEW_DATASETS:
    _init_review_benign_pool(dataset_name)


def _refresh_dataset_for_review_seed(dataset_name, seed):
    obj = DATA[dataset_name]
    feature_cols = obj["feature_cols"]

    pool_df = obj["review_benign_pool"].copy().reset_index(drop=True)
    reference_size = min(int(obj["review_reference_size"]), len(pool_df))

    reference_df = pool_df.sample(
        n=reference_size,
        random_state=int(seed),
    )

    train_df = pool_df.drop(index=reference_df.index).reset_index(drop=True)
    reference_df = reference_df.reset_index(drop=True)

    obj["train"] = train_df
    obj["reference"] = reference_df
    obj["benign_train_count"] = len(train_df)
    obj["benign_ref_count"] = len(reference_df)

    iid_parts = iid_partition(
        df=obj["train"],
        n_clients=CONFIG["n_clients"],
        seed=int(seed),
    )

    noniid_parts, noniid_key_col = balanced_shard_noniid_partition(
        df=obj["train"],
        feature_cols=feature_cols,
        n_clients=CONFIG["n_clients"],
        seed=int(seed),
        shards_per_client=3,
        n_bins=CONFIG["n_clients"],
    )

    obj["iid_parts"] = iid_parts
    obj["noniid_parts"] = noniid_parts
    obj["iid_loaders"] = {
        cid: make_loader_from_df(part, feature_cols, CONFIG["batch_size"], shuffle=True)
        for cid, part in iid_parts.items()
    }
    obj["noniid_loaders"] = {
        cid: make_loader_from_df(part, feature_cols, CONFIG["batch_size"], shuffle=True)
        for cid, part in noniid_parts.items()
    }
    obj["partition_summary"] = {
        "iid": summarize_partitions(iid_parts),
        "noniid": summarize_partitions(noniid_parts),
        "noniid_key_col": noniid_key_col,
        "noniid_heterogeneity": summarize_noniid_heterogeneity(
            noniid_parts,
            noniid_key_col,
            n_bins=CONFIG["n_clients"],
        ),
    }


def _set_review_attack_scale(attack, attack_scale):
    old_values = {
        "seed": CONFIG.get("seed", None),
        "recon_inflation_scale": CONFIG.get("recon_inflation_scale", None),
        "adaptive_attack_scale": CONFIG.get("adaptive_attack_scale", None),
        "train_on_attack_scale": CONFIG.get("train_on_attack_scale", None),
    }

    if attack == "recon_inflation":
        CONFIG["recon_inflation_scale"] = float(attack_scale)

    if attack in ["adaptive_score", "adaptive_reference"]:
        CONFIG["adaptive_attack_scale"] = float(attack_scale)

    if attack == "train_on_attack":
        CONFIG["train_on_attack_scale"] = float(attack_scale)

    return old_values


def _restore_review_config(old_values):
    for key, value in old_values.items():
        if value is None:
            CONFIG.pop(key, None)
        else:
            CONFIG[key] = value


def _extract_review_row(result, run_cfg, seed, status="ok", error_message=""):
    metrics = result.get("final_metrics", {}) if result is not None else {}
    screening = result.get("screening", {}) if result is not None else {}
    aqs_summary = result.get("aqs_summary", {}) if result is not None else {}

    row = {
        "dataset": run_cfg["dataset_name"],
        "scenario": run_cfg.get("scenario", ""),
        "aggregator": run_cfg["aggregator"],
        "model_type": run_cfg.get("model_type", "VAE"),
        "partition": run_cfg.get("partition", "iid"),
        "attack": run_cfg.get("attack", "none"),
        "attack_scale": run_cfg.get("attack_scale", 1.0),
        "byzantine_fraction": run_cfg.get("byzantine_fraction", np.nan),
        "n_byzantine": run_cfg.get("n_byzantine", 0),
        "seed": int(seed),
        "fed_rounds": run_cfg.get("fed_rounds", np.nan),
        "max_local_batches": run_cfg.get("max_local_batches", np.nan),
        "status": status,
        "error_message": error_message,
        "runtime_sec": result.get("runtime_sec", np.nan) if result is not None else np.nan,
        "f1": metrics.get("f1", np.nan),
        "auc": metrics.get("auc", np.nan),
        "fpr": metrics.get("fpr", np.nan),
        "asr": metrics.get("asr", np.nan),
        "threshold": metrics.get("threshold", np.nan),
        "nonfinite_re_count": metrics.get("nonfinite_re_count", np.nan),
        "mdr": screening.get("mdr", np.nan),
        "bfrr": screening.get("bfrr", np.nan),
        "mean_aqs_benign": aqs_summary.get("mean_aqs_benign", np.nan),
        "mean_aqs_malicious": aqs_summary.get("mean_aqs_malicious", np.nan),
        "aqs_gap": aqs_summary.get("aqs_gap", np.nan),
    }

    return row


def run_review_trial(run_cfg, seed):
    old_values = _set_review_attack_scale(
        attack=run_cfg.get("attack", "none"),
        attack_scale=run_cfg.get("attack_scale", 1.0),
    )

    try:
        CONFIG["seed"] = int(seed)

        _refresh_dataset_for_review_seed(
            dataset_name=run_cfg["dataset_name"],
            seed=int(seed),
        )

        result = run_federation({
            "dataset_name": run_cfg["dataset_name"],
            "aggregator": run_cfg["aggregator"],
            "model_type": run_cfg.get("model_type", "VAE"),
            "partition": run_cfg.get("partition", "iid"),
            "n_byzantine": int(run_cfg.get("n_byzantine", 0)),
            "attack": run_cfg.get("attack", "none"),
            "fed_rounds": int(run_cfg.get("fed_rounds", RAFA_REVIEW_FED_ROUNDS)),
            "max_local_batches": int(run_cfg.get("max_local_batches", RAFA_REVIEW_MAX_LOCAL_BATCHES)),
        })

        row = _extract_review_row(
            result=result,
            run_cfg=run_cfg,
            seed=seed,
            status="ok",
            error_message="",
        )

    except Exception as e:
        result = None
        row = _extract_review_row(
            result=None,
            run_cfg=run_cfg,
            seed=seed,
            status="error",
            error_message=str(e),
        )

    finally:
        _restore_review_config(old_values)

    return result, row


def run_review_grid(run_configs, seeds, output_prefix):
    rows = []
    results = {}
    total_runs = len(run_configs) * len(seeds)
    run_id = 0
    start_time = time.time()

    print(f"\n[Cell 7.2 runner] Starting grid: {output_prefix}", flush=True)
    print(f"Run configs : {len(run_configs)}", flush=True)
    print(f"Seeds       : {list(seeds)}", flush=True)
    print(f"Total runs  : {total_runs}", flush=True)

    for run_cfg in run_configs:
        for seed in seeds:
            run_id += 1
            run_key = (
                run_cfg["dataset_name"],
                run_cfg.get("scenario", ""),
                run_cfg["aggregator"],
                run_cfg.get("attack", "none"),
                run_cfg.get("attack_scale", 1.0),
                run_cfg.get("n_byzantine", 0),
                int(seed),
            )

            trial_start = time.time()
            result, row = run_review_trial(run_cfg, seed)
            rows.append(row)
            results[run_key] = result

            print(
                f"  [{run_id:03d}/{total_runs:03d}] "
                f"{row['dataset']} | {row['scenario']} | {row['aggregator']} | "
                f"seed={row['seed']} | status={row['status']} | "
                f"F1={row['f1']:.4f} | AUC={row['auc']:.4f} | "
                f"MDR={row['mdr']:.4f} | BFRR={row['bfrr']:.4f} | "
                f"{time.time() - trial_start:.1f}s",
                flush=True,
            )

    raw_df = pd.DataFrame(rows)

    summary_cols = ["f1", "auc", "fpr", "asr", "mdr", "bfrr", "aqs_gap", "runtime_sec"]
    group_cols = [
        "dataset", "scenario", "aggregator", "partition",
        "attack", "attack_scale", "byzantine_fraction", "n_byzantine",
    ]

    ok_df = raw_df[raw_df["status"] == "ok"].copy()

    if len(ok_df) > 0:
        summary_df = (
            ok_df
            .groupby(group_cols, dropna=False)[summary_cols]
            .agg(["mean", "std", "count"])
            .reset_index()
        )

        summary_df.columns = [
            "_".join([str(x) for x in col if str(x) != ""]).strip("_")
            for col in summary_df.columns.to_flat_index()
        ]
    else:
        summary_df = pd.DataFrame()

    raw_path = results_dir / f"{output_prefix}_raw.csv"
    summary_path = results_dir / f"{output_prefix}_summary.csv"
    pkl_path = results_dir / f"{output_prefix}_results.pkl"

    raw_df.to_csv(raw_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    with open(pkl_path, "wb") as f:
        pickle.dump(results, f)

    print(f"\n[Cell 7.2 runner] Finished grid: {output_prefix}", flush=True)
    print(f"Raw results     : {raw_path}", flush=True)
    print(f"Summary results : {summary_path}", flush=True)
    print(f"Pickle results  : {pkl_path}", flush=True)
    print(f"Elapsed time    : {(time.time() - start_time) / 60.0:.2f} min", flush=True)

    return {
        "raw": raw_df,
        "summary": summary_df,
        "results": results,
        "paths": {
            "raw": str(raw_path),
            "summary": str(summary_path),
            "pickle": str(pkl_path),
        },
    }


# -------------------------
# Final diagnostics
# -------------------------

review_dataset_rows = []

for dataset_name in RAFA_REVIEW_DATASETS:
    obj = DATA[dataset_name]
    review_dataset_rows.append({
        "dataset": dataset_name,
        "features": len(obj["feature_cols"]),
        "train": len(obj["train"]),
        "reference": len(obj["reference"]),
        "validation": len(obj["val"]),
        "test": len(obj["test"]),
        "test_benign": int((obj["test"]["binary_label"] == 0).sum()),
        "test_attack": int((obj["test"]["binary_label"] == 1).sum()),
    })

RAFA_REVIEW_DATASET_SUMMARY = pd.DataFrame(review_dataset_rows)
RAFA_REVIEW_DATASET_SUMMARY.to_csv(
    results_dir / "cell_7_2_review_dataset_summary.csv",
    index=False,
)

print("\n[Cell 7.2] Review datasets ready:", flush=True)
print(RAFA_REVIEW_DATASET_SUMMARY.to_string(index=False), flush=True)

print("\n[Cell 7.2] Runner functions ready:", flush=True)
print("  - run_review_trial(run_cfg, seed)", flush=True)
print("  - run_review_grid(run_configs, seeds, output_prefix)", flush=True)

print("\n[Cell 7.2] Status:", "OK" if not CICDDOS_REVIEW_LOAD_ERRORS else "CHECK_REQUIRED", flush=True)


In [ ]:
# ======================== 7.3: Load Final CICDDoS2019 Paper Subsets with Source-Specific Names ========================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc
import re

print("[Cell 7.5b] CICDDoS2019 clean screening with source-file-unique names started", flush=True)

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# Screening settings
# -------------------------
# Goal:
#   Find better CIC-DDoS2019 subsets for final RAFA experiments.
#   Each CSV file is treated as a separate source-specific dataset.
#
# Selection rule:
#   Prefer subsets with strong base IDS separability:
#   F1 >= 0.80, AUC >= 0.80, ASR <= 0.20.
#
# Reason:
#   RAFA robustness should be tested on datasets where the underlying
#   reconstruction-based IDS can detect attacks in the clean setting.

CELL_7_5B_SEED = int(CONFIG.get("cell_7_5b_seed", 11))
CELL_7_5B_FED_ROUNDS = int(CONFIG.get("cell_7_5b_fed_rounds", 20))
CELL_7_5B_MAX_LOCAL_BATCHES = int(CONFIG.get("cell_7_5b_max_local_batches", 20))

CELL_7_5B_MIN_F1 = float(CONFIG.get("cell_7_5b_min_f1", 0.80))
CELL_7_5B_MIN_AUC = float(CONFIG.get("cell_7_5b_min_auc", 0.80))
CELL_7_5B_MAX_ASR = float(CONFIG.get("cell_7_5b_max_asr", 0.20))

CELL_7_5B_MAX_SELECTED = int(CONFIG.get("cell_7_5b_max_selected", 2))

# Force include by exact source file, not attack-name alias.
CELL_7_5B_FORCE_INCLUDE_FILES = CONFIG.get(
    "cell_7_5b_force_include_files",
    ["MSSQL.csv", "DrDoS_DNS.csv"],
)

# Avoid selecting multiple files from the same broad attack family unless needed.
CELL_7_5B_ENFORCE_FAMILY_DIVERSITY = bool(CONFIG.get("cell_7_5b_enforce_family_diversity", True))

CELL_7_5B_RESUME = bool(CONFIG.get("cell_7_5b_resume", True))
CELL_7_5B_DROP_NONSELECTED = bool(CONFIG.get("cell_7_5b_drop_nonselected", True))

CELL_7_5B_OUTPUT_PREFIX = CONFIG.get(
    "cell_7_5b_output_prefix",
    "cell_7_5b_expanded_cicddos_clean_screen_unique_20r",
)

raw_path = results_dir / f"{CELL_7_5B_OUTPUT_PREFIX}_raw.csv"
ranked_path = results_dir / f"{CELL_7_5B_OUTPUT_PREFIX}_ranked.csv"
selected_path = results_dir / f"{CELL_7_5B_OUTPUT_PREFIX}_selected.csv"
selected_files_path = results_dir / f"{CELL_7_5B_OUTPUT_PREFIX}_selected_files.txt"

print(f"Seed                    : {CELL_7_5B_SEED}", flush=True)
print(f"Fed rounds              : {CELL_7_5B_FED_ROUNDS}", flush=True)
print(f"Max local batches       : {CELL_7_5B_MAX_LOCAL_BATCHES}", flush=True)
print(f"Selection thresholds    : F1 >= {CELL_7_5B_MIN_F1}, AUC >= {CELL_7_5B_MIN_AUC}, ASR <= {CELL_7_5B_MAX_ASR}", flush=True)
print(f"Max selected subsets    : {CELL_7_5B_MAX_SELECTED}", flush=True)
print(f"Force include CSV files : {CELL_7_5B_FORCE_INCLUDE_FILES}", flush=True)
print(f"Family diversity        : {CELL_7_5B_ENFORCE_FAMILY_DIVERSITY}", flush=True)
print(f"Resume                  : {CELL_7_5B_RESUME}", flush=True)


# -------------------------
# Helper functions
# -------------------------

def _cell_7_5b_csv_to_unique_dataset_name(csv_name):
    """
    Source-file-specific dataset name.
    Examples:
      MSSQL.csv       -> CICDDoS2019_MSSQL
      DrDoS_MSSQL.csv -> CICDDoS2019_DrDoS_MSSQL
      UDPLag_2.csv    -> CICDDoS2019_UDPLag_2
    """
    stem = Path(str(csv_name)).stem
    safe = re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_")
    return f"CICDDoS2019_{safe}"


def _cell_7_5b_attack_family(csv_name):
    """
    Broad family label used only for optional selection diversity.
    DrDoS_MSSQL.csv and MSSQL.csv both map to MSSQL.
    DrDoS_UDP.csv and UDP.csv both map to UDP.
    """
    stem = Path(str(csv_name)).stem
    family = stem

    if family.startswith("DrDoS_"):
        family = family.replace("DrDoS_", "", 1)

    if family.endswith("_1") or family.endswith("_2"):
        family = re.sub(r"_[0-9]+$", "", family)

    return family


def _cell_7_5b_is_divergence_error(error_message):
    msg = str(error_message).lower()
    terms = [
        "non-finite",
        "nonfinite",
        "nan",
        "inf",
        "infinite",
        "diverged",
        "reconstruction errors are all non-finite",
        "validation reconstruction errors are all non-finite",
    ]
    return any(t in msg for t in terms)


def _cell_7_5b_normalize_status(row):
    row = dict(row)

    if row.get("status") == "error" and _cell_7_5b_is_divergence_error(row.get("error_message", "")):
        row["status"] = "diverged"

    return row


def _cell_7_5b_drop_dataset(dataset_name):
    DATA.pop(dataset_name, None)
    MODEL_REGISTRY.pop(dataset_name, None)

    gc.collect()

    if "torch" in globals() and torch.cuda.is_available():
        torch.cuda.empty_cache()


def _cell_7_5b_get_scan_info(csv_name):
    matches = [
        row for row in scan_rows
        if row.get("file") == csv_name and row.get("usable", False)
    ]

    if not matches:
        raise ValueError(f"No usable scan info found for {csv_name}.")

    return matches[0]


def _cell_7_5b_prepare_unique_subset(info, feature_cols, unique_dataset_name):
    """
    Reuse Cell 7.1's loader, then store source-file-specific metadata.
    This avoids changing the original loader while preventing dataset-name collisions.
    """
    obj = _prepare_cicddos_subset(info, feature_cols)
    obj["dataset_name"] = unique_dataset_name
    obj["source_file"] = info["file"]
    obj["attack_family"] = _cell_7_5b_attack_family(info["file"])
    return obj


def _cell_7_5b_ensure_dataset_loaded(csv_name):
    info = _cell_7_5b_get_scan_info(csv_name)
    dataset_name = _cell_7_5b_csv_to_unique_dataset_name(csv_name)

    if dataset_name in DATA:
        if "review_benign_pool" not in DATA[dataset_name]:
            _init_review_benign_pool(dataset_name)

        return dataset_name, "already_loaded"

    print(f"  Loading source-specific subset: {dataset_name} from {csv_name}", flush=True)

    DATA[dataset_name] = _cell_7_5b_prepare_unique_subset(
        info=info,
        feature_cols=info["numeric_cols"],
        unique_dataset_name=dataset_name,
    )

    _add_partitions_and_models(dataset_name)
    _init_review_benign_pool(dataset_name)

    return dataset_name, "loaded"


def _cell_7_5b_screen_one_dataset(csv_name):
    dataset_name, load_status = _cell_7_5b_ensure_dataset_loaded(csv_name)
    obj = DATA[dataset_name]

    run_cfg = {
        "dataset_name": dataset_name,
        "scenario": "clean_no_attack_screen_expanded",
        "aggregator": "FedAvg",
        "model_type": "VAE",
        "partition": "iid",
        "attack": "none",
        "attack_scale": 1.0,
        "byzantine_fraction": 0.0,
        "n_byzantine": 0,
        "fed_rounds": CELL_7_5B_FED_ROUNDS,
        "max_local_batches": CELL_7_5B_MAX_LOCAL_BATCHES,
    }

    try:
        result, row = run_review_trial(run_cfg, CELL_7_5B_SEED)
        row = _cell_7_5b_normalize_status(row)

    except Exception as e:
        row = {
            "dataset": dataset_name,
            "scenario": "clean_no_attack_screen_expanded",
            "aggregator": "FedAvg",
            "model_type": "VAE",
            "partition": "iid",
            "attack": "none",
            "attack_scale": 1.0,
            "byzantine_fraction": 0.0,
            "n_byzantine": 0,
            "seed": CELL_7_5B_SEED,
            "fed_rounds": CELL_7_5B_FED_ROUNDS,
            "max_local_batches": CELL_7_5B_MAX_LOCAL_BATCHES,
            "status": "diverged" if _cell_7_5b_is_divergence_error(e) else "error",
            "error_message": str(e),
            "runtime_sec": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "fpr": np.nan,
            "asr": np.nan,
            "threshold": np.nan,
            "nonfinite_re_count": np.nan,
            "mdr": np.nan,
            "bfrr": np.nan,
            "mean_aqs_benign": np.nan,
            "mean_aqs_malicious": np.nan,
            "aqs_gap": np.nan,
        }

    row["source_file"] = csv_name
    row["attack_family"] = _cell_7_5b_attack_family(csv_name)
    row["load_status"] = load_status
    row["n_features"] = len(obj["feature_cols"])
    row["train_size"] = len(obj["train"])
    row["reference_size"] = len(obj["reference"])
    row["validation_size"] = len(obj["val"])
    row["test_size"] = len(obj["test"])
    row["test_benign"] = int((obj["test"]["binary_label"] == 0).sum())
    row["test_attack"] = int((obj["test"]["binary_label"] == 1).sum())

    row["screen_pass_strong"] = (
        row["status"] == "ok"
        and pd.notna(row["f1"])
        and pd.notna(row["auc"])
        and pd.notna(row["asr"])
        and float(row["f1"]) >= CELL_7_5B_MIN_F1
        and float(row["auc"]) >= CELL_7_5B_MIN_AUC
        and float(row["asr"]) <= CELL_7_5B_MAX_ASR
    )

    return row


# -------------------------
# Build candidate list
# -------------------------

if "scan_rows" not in globals():
    raise ValueError("scan_rows is not available. Run Cell 7.1 first.")

usable_scan_rows = [
    row for row in scan_rows
    if row.get("usable", False) and row.get("exists", False)
]

if len(usable_scan_rows) == 0:
    raise ValueError("No usable CIC-DDoS2019 subsets found from Cell 7.1 scan output.")

candidate_df = pd.DataFrame([
    {
        "file": row["file"],
        "dataset": _cell_7_5b_csv_to_unique_dataset_name(row["file"]),
        "attack_family": _cell_7_5b_attack_family(row["file"]),
        "benign": row.get("benign", np.nan),
        "attack": row.get("attack", np.nan),
        "numeric_feature_count": row.get("numeric_feature_count", np.nan),
    }
    for row in usable_scan_rows
])

candidate_df["priority_force_include"] = candidate_df["file"].isin(CELL_7_5B_FORCE_INCLUDE_FILES)

candidate_df = candidate_df.sort_values(
    ["priority_force_include", "benign", "attack"],
    ascending=[False, False, False],
).reset_index(drop=True)

CELL_7_5B_CANDIDATE_FILES = candidate_df["file"].tolist()

print(f"\nUsable source-file-specific candidates: {len(CELL_7_5B_CANDIDATE_FILES)}", flush=True)
print(candidate_df.to_string(index=False), flush=True)


# -------------------------
# Resume support
# -------------------------

rows = []
completed_files = set()

if CELL_7_5B_RESUME and raw_path.exists():
    previous_df = pd.read_csv(raw_path)
    rows = previous_df.to_dict("records")

    done_df = previous_df[previous_df["status"].isin(["ok", "diverged"])].copy()
    completed_files = set(done_df["source_file"].astype(str).tolist())

    print(f"\nResume enabled: found {len(completed_files)} completed/diverged source files", flush=True)
else:
    print("\nResume enabled: no previous completed files found", flush=True)


# -------------------------
# Run screening
# -------------------------

start_time = time.time()
new_runs = 0

print("\n[Cell 7.5b] Running expanded clean screening...", flush=True)

for csv_name in CELL_7_5B_CANDIDATE_FILES:
    if csv_name in completed_files:
        print(f"  Skip completed: {csv_name}", flush=True)
        continue

    new_runs += 1
    trial_start = time.time()

    try:
        row = _cell_7_5b_screen_one_dataset(csv_name)

    except Exception as e:
        dataset_name = _cell_7_5b_csv_to_unique_dataset_name(csv_name)

        row = {
            "dataset": dataset_name,
            "source_file": csv_name,
            "attack_family": _cell_7_5b_attack_family(csv_name),
            "scenario": "clean_no_attack_screen_expanded",
            "aggregator": "FedAvg",
            "model_type": "VAE",
            "partition": "iid",
            "attack": "none",
            "attack_scale": 1.0,
            "byzantine_fraction": 0.0,
            "n_byzantine": 0,
            "seed": CELL_7_5B_SEED,
            "fed_rounds": CELL_7_5B_FED_ROUNDS,
            "max_local_batches": CELL_7_5B_MAX_LOCAL_BATCHES,
            "status": "diverged" if _cell_7_5b_is_divergence_error(e) else "error",
            "error_message": str(e),
            "runtime_sec": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "fpr": np.nan,
            "asr": np.nan,
            "threshold": np.nan,
            "nonfinite_re_count": np.nan,
            "mdr": np.nan,
            "bfrr": np.nan,
            "mean_aqs_benign": np.nan,
            "mean_aqs_malicious": np.nan,
            "aqs_gap": np.nan,
            "n_features": np.nan,
            "train_size": np.nan,
            "reference_size": np.nan,
            "validation_size": np.nan,
            "test_size": np.nan,
            "test_benign": np.nan,
            "test_attack": np.nan,
            "load_status": "error",
            "screen_pass_strong": False,
        }

    rows.append(row)
    pd.DataFrame(rows).to_csv(raw_path, index=False)

    print(
        f"  Screen {new_runs:03d} | {row['dataset']} | file={row['source_file']} | "
        f"status={row['status']} | F1={row['f1']:.4f} | AUC={row['auc']:.4f} | "
        f"FPR={row['fpr']:.4f} | ASR={row['asr']:.4f} | "
        f"pass={row['screen_pass_strong']} | {time.time() - trial_start:.1f}s",
        flush=True,
    )

    gc.collect()

    if "torch" in globals() and torch.cuda.is_available():
        torch.cuda.empty_cache()

CELL_7_5B_RAW = pd.DataFrame(rows)

print(f"\n[Cell 7.5b] New screening runs executed: {new_runs}", flush=True)
print(f"[Cell 7.5b] Raw screening saved: {raw_path}", flush=True)


# -------------------------
# Rank and select subsets
# -------------------------

screen_df = CELL_7_5B_RAW.copy()

for col in ["f1", "auc", "fpr", "asr", "train_size", "test_benign", "test_attack"]:
    if col in screen_df.columns:
        screen_df[col] = pd.to_numeric(screen_df[col], errors="coerce")

screen_df["strong_rank_score"] = (
    screen_df["f1"].fillna(-1.0)
    + screen_df["auc"].fillna(-1.0)
    - screen_df["asr"].fillna(1.0)
)

screen_df["forced_include"] = screen_df["source_file"].isin(CELL_7_5B_FORCE_INCLUDE_FILES)

ranked = screen_df.sort_values(
    [
        "screen_pass_strong",
        "forced_include",
        "f1",
        "auc",
        "asr",
        "test_benign",
    ],
    ascending=[
        False,
        False,
        False,
        False,
        True,
        False,
    ],
).reset_index(drop=True)

ranked["rank"] = np.arange(1, len(ranked) + 1)

ranked.to_csv(ranked_path, index=False)

passed = ranked[
    (ranked["status"] == "ok")
    & (ranked["screen_pass_strong"] == True)
].copy()

selected_rows = []
selected_files = set()
selected_families = set()

# First include forced CSV files, but only if they pass the strong screen.
for csv_name in CELL_7_5B_FORCE_INCLUDE_FILES:
    match = passed[passed["source_file"] == csv_name]

    if len(match) > 0 and len(selected_rows) < CELL_7_5B_MAX_SELECTED:
        row = match.iloc[0].to_dict()
        selected_rows.append(row)
        selected_files.add(row["source_file"])
        selected_families.add(row["attack_family"])

# Then fill remaining slots by ranked quality.
for _, row in passed.iterrows():
    if len(selected_rows) >= CELL_7_5B_MAX_SELECTED:
        break

    if row["source_file"] in selected_files:
        continue

    if CELL_7_5B_ENFORCE_FAMILY_DIVERSITY and row["attack_family"] in selected_families:
        continue

    selected_rows.append(row.to_dict())
    selected_files.add(row["source_file"])
    selected_families.add(row["attack_family"])

# If diversity prevents reaching max selected, fill remaining slots without diversity restriction.
if len(selected_rows) < CELL_7_5B_MAX_SELECTED:
    for _, row in passed.iterrows():
        if len(selected_rows) >= CELL_7_5B_MAX_SELECTED:
            break

        if row["source_file"] in selected_files:
            continue

        selected_rows.append(row.to_dict())
        selected_files.add(row["source_file"])
        selected_families.add(row["attack_family"])

CELL_7_5B_SELECTED_DF = pd.DataFrame(selected_rows)
CELL_7_5B_SELECTED_DATASETS = (
    CELL_7_5B_SELECTED_DF["dataset"].tolist()
    if len(CELL_7_5B_SELECTED_DF) > 0
    else []
)
CELL_7_5B_SELECTED_FILES = (
    CELL_7_5B_SELECTED_DF["source_file"].tolist()
    if len(CELL_7_5B_SELECTED_DF) > 0
    else []
)

CELL_7_5B_SELECTED_DF.to_csv(selected_path, index=False)

with open(selected_files_path, "w") as f:
    for csv_name in CELL_7_5B_SELECTED_FILES:
        f.write(f"{csv_name}\n")

print("\n[Cell 7.5b] Ranked screening results:", flush=True)

show_cols = [
    "rank", "dataset", "source_file", "attack_family", "status",
    "f1", "auc", "fpr", "asr",
    "train_size", "test_benign", "test_attack",
    "screen_pass_strong", "forced_include"
]

show_cols = [c for c in show_cols if c in ranked.columns]
ranked_display = ranked[show_cols].copy()

for col in ["f1", "auc", "fpr", "asr"]:
    if col in ranked_display.columns:
        ranked_display[col] = pd.to_numeric(ranked_display[col], errors="coerce").round(4)

print(ranked_display.to_string(index=False), flush=True)

print("\n[Cell 7.5b] Selected datasets for final RAFA experiments:", flush=True)
print(CELL_7_5B_SELECTED_DATASETS, flush=True)

print("\n[Cell 7.5b] Exact selected CSV files for final RAFA experiments:", flush=True)
for idx, csv_name in enumerate(CELL_7_5B_SELECTED_FILES, start=1):
    dataset_name = CELL_7_5B_SELECTED_DF.iloc[idx - 1]["dataset"]
    family = CELL_7_5B_SELECTED_DF.iloc[idx - 1]["attack_family"]
    print(f"  {idx}. {csv_name}  ->  {dataset_name}  (family={family})", flush=True)

if len(CELL_7_5B_SELECTED_DF) > 0:
    selected_display = CELL_7_5B_SELECTED_DF[show_cols].copy()

    for col in ["f1", "auc", "fpr", "asr"]:
        if col in selected_display.columns:
            selected_display[col] = pd.to_numeric(selected_display[col], errors="coerce").round(4)

    print("\n[Cell 7.5b] Selected subset details:", flush=True)
    print(selected_display.to_string(index=False), flush=True)

print(f"\nSaved ranked screening : {ranked_path}", flush=True)
print(f"Saved selected subsets : {selected_path}", flush=True)
print(f"Saved selected CSV list: {selected_files_path}", flush=True)


# -------------------------
# Memory cleanup
# -------------------------

if CELL_7_5B_DROP_NONSELECTED:
    selected_set = set(CELL_7_5B_SELECTED_DATASETS)

    for dataset_name in list(DATA.keys()):
        if dataset_name.startswith("CICDDoS2019_") and dataset_name not in selected_set:
            _cell_7_5b_drop_dataset(dataset_name)
            print(f"  Dropped non-selected subset from memory: {dataset_name}", flush=True)

    # Ensure selected source-specific datasets remain loaded for the next cell.
    for _, row in CELL_7_5B_SELECTED_DF.iterrows():
        dataset_name = row["dataset"]
        source_file = row["source_file"]

        if dataset_name not in DATA:
            _cell_7_5b_ensure_dataset_loaded(source_file)


# -------------------------
# Final status
# -------------------------

hard_errors = int((CELL_7_5B_RAW["status"] == "error").sum()) if len(CELL_7_5B_RAW) > 0 else 0

print(f"\n[Cell 7.5b] Elapsed time: {(time.time() - start_time) / 60.0:.2f} min", flush=True)
print("[Cell 7.5b] Status:", "OK" if hard_errors == 0 else "CHECK_REQUIRED", flush=True)


In [ ]:

# ======================== 7.4: Fixed CICDDoS2019 Configurations Used for the Paper ========================
#
# This cell replaces the development-time sensitivity search with the final
# source-specific configurations that produced the paper's CICDDoS2019 table.
#
# Important:
#   CICIoT2023 experiments use the default RAFA setting alpha=15, beta=0, tau=0.2.
#   The final CICDDoS2019 table used selected source-specific settings:
#     MSSQL.csv       -> strict_threshold, alpha=15, beta=0, tau=0.3
#     DrDoS_DNS.csv   -> high_sensitivity, alpha=20, beta=0, tau=0.2

CELL_7_6_BEST_CONFIGS = pd.DataFrame([
    {
        "dataset": "CICDDoS2019_MSSQL",
        "source_file": "MSSQL.csv",
        "attack_family": "MSSQL",
        "scenario": "recon_targeted_s5_f04",
        "config_name": "strict_threshold",
        "alpha": 15.0,
        "beta": 0.0,
        "tau": 0.3,
        "clean_f1": 0.9998,
        "clean_auc": 0.9995,
        "clean_fpr": 0.0793,
        "clean_asr": 0.0002,
    },
    {
        "dataset": "CICDDoS2019_MSSQL",
        "source_file": "MSSQL.csv",
        "attack_family": "MSSQL",
        "scenario": "sign_flip_f04",
        "config_name": "strict_threshold",
        "alpha": 15.0,
        "beta": 0.0,
        "tau": 0.3,
        "clean_f1": 0.9998,
        "clean_auc": 0.9995,
        "clean_fpr": 0.0793,
        "clean_asr": 0.0002,
    },
    {
        "dataset": "CICDDoS2019_DrDoS_DNS",
        "source_file": "DrDoS_DNS.csv",
        "attack_family": "DNS",
        "scenario": "recon_targeted_s5_f04",
        "config_name": "high_sensitivity",
        "alpha": 20.0,
        "beta": 0.0,
        "tau": 0.2,
        "clean_f1": 0.9840,
        "clean_auc": 0.9861,
        "clean_fpr": 0.0595,
        "clean_asr": 0.0314,
    },
    {
        "dataset": "CICDDoS2019_DrDoS_DNS",
        "source_file": "DrDoS_DNS.csv",
        "attack_family": "DNS",
        "scenario": "sign_flip_f04",
        "config_name": "high_sensitivity",
        "alpha": 20.0,
        "beta": 0.0,
        "tau": 0.2,
        "clean_f1": 0.9840,
        "clean_auc": 0.9861,
        "clean_fpr": 0.0595,
        "clean_asr": 0.0314,
    },
])

print(CELL_7_6_BEST_CONFIGS.to_string(index=False))


In [ ]:
# ======================== 7.5: Validate Final CICDDoS2019 RAFA Configurations and Baselines ========================

from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc

print("[Cell 7.7] Final RAFA validation and matched baseline comparison started", flush=True)

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# Settings
# -------------------------

CELL_7_7_SEEDS = CONFIG.get("cell_7_7_seeds", [11, 22, 33])
CELL_7_7_SEEDS = [int(s) for s in CELL_7_7_SEEDS]

CELL_7_7_FED_ROUNDS = int(CONFIG.get("cell_7_7_fed_rounds", 20))
CELL_7_7_MAX_LOCAL_BATCHES = int(CONFIG.get("cell_7_7_max_local_batches", 20))

CELL_7_7_RESUME = bool(CONFIG.get("cell_7_7_resume", True))

CELL_7_7_OUTPUT_PREFIX = CONFIG.get(
    "cell_7_7_output_prefix",
    "cell_7_7_final_rafa_vs_baselines_20r",
)

raw_path = results_dir / f"{CELL_7_7_OUTPUT_PREFIX}_raw.csv"
summary_path = results_dir / f"{CELL_7_7_OUTPUT_PREFIX}_summary.csv"
compact_path = results_dir / f"{CELL_7_7_OUTPUT_PREFIX}_compact_summary.csv"
compare_path = results_dir / f"{CELL_7_7_OUTPUT_PREFIX}_rafa_vs_best_baseline.csv"
evidence_path = results_dir / f"{CELL_7_7_OUTPUT_PREFIX}_reviewer_evidence.csv"
latex_path = results_dir / f"{CELL_7_7_OUTPUT_PREFIX}_reviewer_evidence_latex.txt"

CELL_7_7_RECON_BASELINES = CONFIG.get(
    "cell_7_7_recon_baselines",
    ["FedAvg", "Krum", "FLTrust-AE"],
)

CELL_7_7_SIGNFLIP_BASELINES = CONFIG.get(
    "cell_7_7_signflip_baselines",
    ["FedAvg", "FedAvgM", "Krum"],
)

ORIGINAL_RAFA_CONFIG = {
    "alpha": CONFIG.get("alpha", 15.0),
    "beta": CONFIG.get("beta", 0.0),
    "tau": CONFIG.get("tau", 0.2),
}

print(f"Seeds                 : {CELL_7_7_SEEDS}", flush=True)
print(f"Fed rounds            : {CELL_7_7_FED_ROUNDS}", flush=True)
print(f"Max local batches     : {CELL_7_7_MAX_LOCAL_BATCHES}", flush=True)
print(f"Recon baselines       : {CELL_7_7_RECON_BASELINES}", flush=True)
print(f"Sign-flip baselines   : {CELL_7_7_SIGNFLIP_BASELINES}", flush=True)
print(f"Resume                : {CELL_7_7_RESUME}", flush=True)


# -------------------------
# Load best RAFA configs from Cell 7.6
# -------------------------

if "CELL_7_6_BEST_CONFIGS" in globals() and len(CELL_7_6_BEST_CONFIGS) > 0:
    best_cfg_df = CELL_7_6_BEST_CONFIGS.copy()
else:
    best_cfg_path = results_dir / "cell_7_6_rafa_sensitivity_final_cicddos_20r_best_configs.csv"

    if not best_cfg_path.exists():
        raise FileNotFoundError(
            "Could not find Cell 7.6 best configuration file. Run Cell 7.6 first."
        )

    best_cfg_df = pd.read_csv(best_cfg_path)

required_cols = [
    "dataset", "source_file", "attack_family", "scenario",
    "config_name", "alpha", "beta", "tau",
    "clean_f1", "clean_auc", "clean_fpr", "clean_asr",
]

missing_cols = [c for c in required_cols if c not in best_cfg_df.columns]

if missing_cols:
    raise ValueError(f"Cell 7.6 best config table is missing columns: {missing_cols}")

best_cfg_df = best_cfg_df[required_cols].copy()

for col in ["alpha", "beta", "tau", "clean_f1", "clean_auc", "clean_fpr", "clean_asr"]:
    best_cfg_df[col] = pd.to_numeric(best_cfg_df[col], errors="coerce")

print("\n[Cell 7.7] Selected RAFA configs from Cell 7.6:", flush=True)

show_cfg_cols = [
    "dataset", "source_file", "attack_family", "scenario",
    "config_name", "alpha", "beta", "tau", "clean_f1", "clean_auc", "clean_asr"
]

cfg_display = best_cfg_df[show_cfg_cols].copy()

for col in ["alpha", "beta", "tau", "clean_f1", "clean_auc", "clean_asr"]:
    cfg_display[col] = pd.to_numeric(cfg_display[col], errors="coerce").round(4)

print(cfg_display.to_string(index=False), flush=True)


# -------------------------
# Helper functions
# -------------------------

def _cell_7_7_is_divergence_error(error_message):
    msg = str(error_message).lower()
    terms = [
        "non-finite",
        "nonfinite",
        "nan",
        "inf",
        "infinite",
        "diverged",
        "reconstruction errors are all non-finite",
        "validation reconstruction errors are all non-finite",
    ]
    return any(t in msg for t in terms)


def _cell_7_7_normalize_status(row):
    row = dict(row)

    if row.get("status") == "error" and _cell_7_7_is_divergence_error(row.get("error_message", "")):
        row["status"] = "diverged"

    return row


def _cell_7_7_restore_rafa_config(old_cfg):
    CONFIG["alpha"] = old_cfg["alpha"]
    CONFIG["beta"] = old_cfg["beta"]
    CONFIG["tau"] = old_cfg["tau"]


def _cell_7_7_ensure_dataset_loaded(dataset_name, source_file):
    if dataset_name in DATA:
        if "review_benign_pool" not in DATA[dataset_name]:
            _init_review_benign_pool(dataset_name)
        return

    if "_cell_7_5b_ensure_dataset_loaded" not in globals():
        raise ValueError(
            f"{dataset_name} is not loaded and Cell 7.5b loader is unavailable. "
            "Please rerun the revised Cell 7.5b first."
        )

    print(f"  Re-loading selected subset: {dataset_name} from {source_file}", flush=True)
    loaded_name, _ = _cell_7_5b_ensure_dataset_loaded(source_file)

    if loaded_name != dataset_name:
        raise ValueError(
            f"Reloaded dataset name mismatch. Expected {dataset_name}, got {loaded_name}."
        )


def _cell_7_7_make_run_cfg(cfg_row, aggregator):
    scenario = cfg_row["scenario"]

    if scenario == "recon_targeted_s5_f04":
        attack = "recon_inflation"
        attack_scale = 5.0
        byz_frac = 0.4
    elif scenario == "sign_flip_f04":
        attack = "sign_flip"
        attack_scale = 1.0
        byz_frac = 0.4
    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    return {
        "dataset_name": cfg_row["dataset"],
        "scenario": scenario,
        "aggregator": aggregator,
        "model_type": "VAE",
        "partition": "iid",
        "attack": attack,
        "attack_scale": attack_scale,
        "byzantine_fraction": byz_frac,
        "n_byzantine": int(round(CONFIG["n_clients"] * byz_frac)),
        "fed_rounds": CELL_7_7_FED_ROUNDS,
        "max_local_batches": CELL_7_7_MAX_LOCAL_BATCHES,
    }


def _cell_7_7_run_trial(run_cfg, seed, cfg_row=None):
    old_cfg = {
        "alpha": CONFIG.get("alpha", 15.0),
        "beta": CONFIG.get("beta", 0.0),
        "tau": CONFIG.get("tau", 0.2),
    }

    is_rafa = run_cfg["aggregator"] == "RAFA"

    try:
        if is_rafa:
            CONFIG["alpha"] = float(cfg_row["alpha"])
            CONFIG["beta"] = float(cfg_row["beta"])
            CONFIG["tau"] = float(cfg_row["tau"])

        result, row = run_review_trial(run_cfg, seed)
        row = _cell_7_7_normalize_status(row)

    except Exception as e:
        row = {
            "dataset": run_cfg["dataset_name"],
            "scenario": run_cfg["scenario"],
            "aggregator": run_cfg["aggregator"],
            "model_type": run_cfg["model_type"],
            "partition": run_cfg["partition"],
            "attack": run_cfg["attack"],
            "attack_scale": run_cfg["attack_scale"],
            "byzantine_fraction": run_cfg["byzantine_fraction"],
            "n_byzantine": run_cfg["n_byzantine"],
            "seed": int(seed),
            "fed_rounds": run_cfg["fed_rounds"],
            "max_local_batches": run_cfg["max_local_batches"],
            "status": "diverged" if _cell_7_7_is_divergence_error(e) else "error",
            "error_message": str(e),
            "runtime_sec": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "fpr": np.nan,
            "asr": np.nan,
            "threshold": np.nan,
            "nonfinite_re_count": np.nan,
            "mdr": np.nan,
            "bfrr": np.nan,
            "mean_aqs_benign": np.nan,
            "mean_aqs_malicious": np.nan,
            "aqs_gap": np.nan,
        }

    finally:
        _cell_7_7_restore_rafa_config(old_cfg)

    meta_match = best_cfg_df[
        (best_cfg_df["dataset"] == row["dataset"])
        & (best_cfg_df["scenario"] == row["scenario"])
    ]

    if len(meta_match) > 0:
        meta = meta_match.iloc[0]
        row["source_file"] = meta["source_file"]
        row["attack_family"] = meta["attack_family"]
        row["clean_f1"] = float(meta["clean_f1"])
        row["clean_auc"] = float(meta["clean_auc"])
        row["clean_fpr"] = float(meta["clean_fpr"])
        row["clean_asr"] = float(meta["clean_asr"])
    else:
        row["source_file"] = ""
        row["attack_family"] = ""
        row["clean_f1"] = np.nan
        row["clean_auc"] = np.nan
        row["clean_fpr"] = np.nan
        row["clean_asr"] = np.nan

    if is_rafa:
        row["config_name"] = cfg_row["config_name"]
        row["alpha"] = float(cfg_row["alpha"])
        row["beta"] = float(cfg_row["beta"])
        row["tau"] = float(cfg_row["tau"])
    else:
        row["config_name"] = run_cfg["aggregator"]
        row["alpha"] = np.nan
        row["beta"] = np.nan
        row["tau"] = np.nan

    return row


# -------------------------
# Ensure selected datasets are loaded
# -------------------------

for _, cfg_row in best_cfg_df.drop_duplicates(["dataset", "source_file"]).iterrows():
    _cell_7_7_ensure_dataset_loaded(cfg_row["dataset"], cfg_row["source_file"])


# -------------------------
# Build run queue
# -------------------------

run_items = []

for _, cfg_row in best_cfg_df.iterrows():
    cfg_dict = cfg_row.to_dict()

    # RAFA with selected config
    for seed in CELL_7_7_SEEDS:
        run_items.append({
            "kind": "rafa",
            "cfg_row": cfg_dict,
            "run_cfg": _cell_7_7_make_run_cfg(cfg_dict, "RAFA"),
            "seed": int(seed),
        })

    # Matched baselines
    if cfg_dict["scenario"] == "recon_targeted_s5_f04":
        baseline_aggs = CELL_7_7_RECON_BASELINES
    elif cfg_dict["scenario"] == "sign_flip_f04":
        baseline_aggs = CELL_7_7_SIGNFLIP_BASELINES
    else:
        baseline_aggs = ["Krum"]

    for agg in baseline_aggs:
        for seed in CELL_7_7_SEEDS:
            run_items.append({
                "kind": "baseline",
                "cfg_row": cfg_dict,
                "run_cfg": _cell_7_7_make_run_cfg(cfg_dict, agg),
                "seed": int(seed),
            })

print(f"\n[Cell 7.7] Run items: {len(run_items)}", flush=True)
print(f"  RAFA runs     : {sum(1 for x in run_items if x['kind'] == 'rafa')}", flush=True)
print(f"  Baseline runs : {sum(1 for x in run_items if x['kind'] == 'baseline')}", flush=True)


# -------------------------
# Resume support
# -------------------------

rows = []
completed_keys = set()

if CELL_7_7_RESUME and raw_path.exists():
    previous_df = pd.read_csv(raw_path)
    rows = previous_df.to_dict("records")

    done_df = previous_df[previous_df["status"].isin(["ok", "diverged"])].copy()

    for _, r in done_df.iterrows():
        alpha_key = "nan" if pd.isna(r.get("alpha", np.nan)) else f"{float(r['alpha']):.6f}"
        tau_key = "nan" if pd.isna(r.get("tau", np.nan)) else f"{float(r['tau']):.6f}"

        completed_keys.add((
            str(r["dataset"]),
            str(r["scenario"]),
            str(r["aggregator"]),
            str(r.get("config_name", "")),
            alpha_key,
            tau_key,
            int(r["seed"]),
        ))

    print(f"Resume enabled: found {len(completed_keys)} completed/diverged runs", flush=True)
else:
    print("Resume enabled: no previous completed runs found", flush=True)


# -------------------------
# Execute runs
# -------------------------

start_time = time.time()
new_runs = 0

print("\n[Cell 7.7] Running final RAFA and matched baseline validation...", flush=True)

for item in run_items:
    run_cfg = item["run_cfg"]
    cfg_row = item["cfg_row"]
    seed = item["seed"]

    if run_cfg["aggregator"] == "RAFA":
        config_name = cfg_row["config_name"]
        alpha_key = f"{float(cfg_row['alpha']):.6f}"
        tau_key = f"{float(cfg_row['tau']):.6f}"
    else:
        config_name = run_cfg["aggregator"]
        alpha_key = "nan"
        tau_key = "nan"

    run_key = (
        run_cfg["dataset_name"],
        run_cfg["scenario"],
        run_cfg["aggregator"],
        config_name,
        alpha_key,
        tau_key,
        int(seed),
    )

    if run_key in completed_keys:
        print(
            f"  Skip completed: {run_cfg['dataset_name']} | {run_cfg['scenario']} | "
            f"{run_cfg['aggregator']} | {config_name} | seed={seed}",
            flush=True,
        )
        continue

    new_runs += 1
    trial_start = time.time()

    row = _cell_7_7_run_trial(run_cfg, seed, cfg_row if run_cfg["aggregator"] == "RAFA" else None)
    rows.append(row)

    pd.DataFrame(rows).to_csv(raw_path, index=False)

    print(
        f"  Run {new_runs:03d} | {row['dataset']} | file={row['source_file']} | "
        f"{row['scenario']} | {row['aggregator']} | {row['config_name']} | seed={row['seed']} | "
        f"status={row['status']} | F1={row['f1']:.4f} | AUC={row['auc']:.4f} | "
        f"MDR={row['mdr']:.4f} | BFRR={row['bfrr']:.4f} | "
        f"{time.time() - trial_start:.1f}s",
        flush=True,
    )

    gc.collect()

    if "torch" in globals() and torch.cuda.is_available():
        torch.cuda.empty_cache()

_cell_7_7_restore_rafa_config(ORIGINAL_RAFA_CONFIG)

CELL_7_7_RAW = pd.DataFrame(rows)

print(f"\n[Cell 7.7] New runs executed : {new_runs}", flush=True)
print(f"[Cell 7.7] Raw results saved: {raw_path}", flush=True)


# -------------------------
# Summaries
# -------------------------

status_counts = CELL_7_7_RAW["status"].value_counts(dropna=False).to_dict()

print("\n[Cell 7.7] Run status counts:", flush=True)
print(status_counts, flush=True)

hard_error_df = CELL_7_7_RAW[CELL_7_7_RAW["status"] == "error"].copy()

if len(hard_error_df) > 0:
    error_path = results_dir / f"{CELL_7_7_OUTPUT_PREFIX}_errors.csv"
    hard_error_df.to_csv(error_path, index=False)

    print("\n[Cell 7.7] Non-divergence errors detected:", flush=True)
    print(
        hard_error_df[
            ["dataset", "source_file", "scenario", "aggregator", "seed", "error_message"]
        ].to_string(index=False),
        flush=True,
    )
    print(f"Saved errors to: {error_path}", flush=True)

diverged_df = CELL_7_7_RAW[CELL_7_7_RAW["status"] == "diverged"].copy()

if len(diverged_df) > 0:
    diverged_path = results_dir / f"{CELL_7_7_OUTPUT_PREFIX}_diverged.csv"
    diverged_df.to_csv(diverged_path, index=False)

    print("\n[Cell 7.7] Diverged runs:", flush=True)
    print(
        diverged_df[
            ["dataset", "source_file", "scenario", "aggregator", "seed", "error_message"]
        ].to_string(index=False),
        flush=True,
    )
    print(f"Saved diverged runs to: {diverged_path}", flush=True)

ok_df = CELL_7_7_RAW[CELL_7_7_RAW["status"] == "ok"].copy()

if len(ok_df) == 0:
    CELL_7_7_SUMMARY = pd.DataFrame()
    CELL_7_7_COMPACT = pd.DataFrame()
    CELL_7_7_COMPARE = pd.DataFrame()
    CELL_7_7_EVIDENCE = pd.DataFrame()
    print("\n[Cell 7.7] No successful runs were completed.", flush=True)

else:
    metric_cols = [
        "f1", "auc", "fpr", "asr",
        "mdr", "bfrr", "aqs_gap", "runtime_sec"
    ]

    group_cols = [
        "dataset", "source_file", "attack_family", "scenario",
        "aggregator", "config_name", "alpha", "beta", "tau",
        "clean_f1", "clean_auc", "clean_fpr", "clean_asr",
        "fed_rounds", "max_local_batches",
    ]

    CELL_7_7_SUMMARY = (
        ok_df
        .groupby(group_cols, dropna=False)[metric_cols]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    CELL_7_7_SUMMARY.columns = [
        "_".join([str(x) for x in col if str(x) != ""]).strip("_")
        for col in CELL_7_7_SUMMARY.columns.to_flat_index()
    ]

    CELL_7_7_SUMMARY.to_csv(summary_path, index=False)

    CELL_7_7_COMPACT = (
        ok_df
        .groupby(
            ["dataset", "source_file", "attack_family", "scenario", "aggregator", "config_name"],
            dropna=False,
        )
        .agg(
            runs=("f1", "count"),
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
            auc_mean=("auc", "mean"),
            auc_std=("auc", "std"),
            fpr_mean=("fpr", "mean"),
            asr_mean=("asr", "mean"),
            mdr_mean=("mdr", "mean"),
            bfrr_mean=("bfrr", "mean"),
            aqs_gap_mean=("aqs_gap", "mean"),
            runtime_mean=("runtime_sec", "mean"),
            clean_f1=("clean_f1", "mean"),
            clean_auc=("clean_auc", "mean"),
        )
        .reset_index()
    )

    for col in [
        "f1_mean", "f1_std", "auc_mean", "auc_std",
        "fpr_mean", "asr_mean", "mdr_mean", "bfrr_mean",
        "aqs_gap_mean", "runtime_mean", "clean_f1", "clean_auc"
    ]:
        CELL_7_7_COMPACT[col] = pd.to_numeric(CELL_7_7_COMPACT[col], errors="coerce").round(4)

    CELL_7_7_COMPACT.to_csv(compact_path, index=False)

    print("\n[Cell 7.7] Compact matched comparison summary:", flush=True)
    print(CELL_7_7_COMPACT.to_string(index=False), flush=True)

    # -------------------------
    # RAFA vs best baseline table
    # -------------------------

    rafa_rows = CELL_7_7_COMPACT[CELL_7_7_COMPACT["aggregator"] == "RAFA"].copy()
    baseline_rows = CELL_7_7_COMPACT[CELL_7_7_COMPACT["aggregator"] != "RAFA"].copy()

    if len(rafa_rows) > 0 and len(baseline_rows) > 0:
        best_baseline = (
            baseline_rows
            .sort_values(
                ["dataset", "scenario", "f1_mean", "auc_mean"],
                ascending=[True, True, False, False],
            )
            .groupby(["dataset", "scenario"], as_index=False)
            .head(1)
            .rename(columns={
                "aggregator": "best_baseline",
                "config_name": "best_baseline_config",
                "f1_mean": "best_baseline_f1_mean",
                "f1_std": "best_baseline_f1_std",
                "auc_mean": "best_baseline_auc_mean",
                "auc_std": "best_baseline_auc_std",
            })
        )

        CELL_7_7_COMPARE = rafa_rows.merge(
            best_baseline[
                [
                    "dataset", "scenario",
                    "best_baseline", "best_baseline_config",
                    "best_baseline_f1_mean", "best_baseline_f1_std",
                    "best_baseline_auc_mean", "best_baseline_auc_std",
                ]
            ],
            on=["dataset", "scenario"],
            how="left",
        )

        CELL_7_7_COMPARE["delta_f1_vs_clean"] = (
            CELL_7_7_COMPARE["f1_mean"] - CELL_7_7_COMPARE["clean_f1"]
        ).round(4)

        CELL_7_7_COMPARE["delta_auc_vs_clean"] = (
            CELL_7_7_COMPARE["auc_mean"] - CELL_7_7_COMPARE["clean_auc"]
        ).round(4)

        CELL_7_7_COMPARE["f1_gap_vs_best_baseline"] = (
            CELL_7_7_COMPARE["f1_mean"] - CELL_7_7_COMPARE["best_baseline_f1_mean"]
        ).round(4)

        CELL_7_7_COMPARE["auc_gap_vs_best_baseline"] = (
            CELL_7_7_COMPARE["auc_mean"] - CELL_7_7_COMPARE["best_baseline_auc_mean"]
        ).round(4)

        CELL_7_7_COMPARE.to_csv(compare_path, index=False)

        print("\n[Cell 7.7] RAFA vs best matched baseline:", flush=True)

        compare_show_cols = [
            "dataset", "source_file", "scenario", "config_name",
            "clean_f1", "f1_mean", "f1_std", "delta_f1_vs_clean",
            "best_baseline", "best_baseline_f1_mean", "best_baseline_f1_std",
            "f1_gap_vs_best_baseline",
            "auc_mean", "best_baseline_auc_mean", "auc_gap_vs_best_baseline",
            "mdr_mean", "bfrr_mean", "aqs_gap_mean",
        ]

        compare_show_cols = [c for c in compare_show_cols if c in CELL_7_7_COMPARE.columns]
        compare_display = CELL_7_7_COMPARE[compare_show_cols].copy()

        for col in compare_display.columns:
            if col not in ["dataset", "source_file", "scenario", "config_name", "best_baseline"]:
                compare_display[col] = pd.to_numeric(compare_display[col], errors="coerce").round(4)

        print(compare_display.to_string(index=False), flush=True)

        # Reviewer evidence table
        evidence_cols = [
            "source_file", "dataset", "attack_family", "scenario", "config_name",
            "clean_f1", "clean_auc",
            "f1_mean", "f1_std", "delta_f1_vs_clean",
            "auc_mean", "auc_std", "delta_auc_vs_clean",
            "best_baseline", "best_baseline_f1_mean", "best_baseline_f1_std",
            "f1_gap_vs_best_baseline",
            "mdr_mean", "bfrr_mean", "aqs_gap_mean",
            "runtime_mean",
        ]

        evidence_cols = [c for c in evidence_cols if c in CELL_7_7_COMPARE.columns]
        CELL_7_7_EVIDENCE = CELL_7_7_COMPARE[evidence_cols].copy()

        for col in CELL_7_7_EVIDENCE.columns:
            if col not in ["source_file", "dataset", "attack_family", "scenario", "config_name", "best_baseline"]:
                CELL_7_7_EVIDENCE[col] = pd.to_numeric(CELL_7_7_EVIDENCE[col], errors="coerce").round(4)

        CELL_7_7_EVIDENCE.to_csv(evidence_path, index=False)

        with open(latex_path, "w") as f:
            f.write(CELL_7_7_EVIDENCE.to_latex(index=False, escape=False))

        print(f"\nSaved RAFA-vs-baseline comparison : {compare_path}", flush=True)
        print(f"Saved reviewer evidence table     : {evidence_path}", flush=True)
        print(f"Saved reviewer LaTeX table        : {latex_path}", flush=True)

    else:
        CELL_7_7_COMPARE = pd.DataFrame()
        CELL_7_7_EVIDENCE = pd.DataFrame()
        print("\n[Cell 7.7] Could not compute RAFA-vs-baseline table because RAFA or baseline rows are missing.", flush=True)

    print(f"\nSaved full summary    : {summary_path}", flush=True)
    print(f"Saved compact summary : {compact_path}", flush=True)


# -------------------------
# Final status
# -------------------------

_cell_7_7_restore_rafa_config(ORIGINAL_RAFA_CONFIG)

print("\n[Cell 7.7] Restored original RAFA config:", flush=True)
print(
    f"alpha={CONFIG['alpha']}, beta={CONFIG['beta']}, tau={CONFIG['tau']}",
    flush=True,
)

hard_errors = int((CELL_7_7_RAW["status"] == "error").sum()) if len(CELL_7_7_RAW) > 0 else 0

print(f"\n[Cell 7.7] Elapsed time: {(time.time() - start_time) / 60.0:.2f} min", flush=True)
print("[Cell 7.7] Status:", "OK" if hard_errors == 0 else "CHECK_REQUIRED", flush=True)


In [ ]:
# ======================== 7.6: Final CICDDoS2019 Statistical Summary ========================

from pathlib import Path
import pandas as pd
import numpy as np

print("[Cell 7.9] Revised final statistical summary and reviewer-ready tables started", flush=True)

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# Settings
# -------------------------

CELL_7_9_OUTPUT_PREFIX = CONFIG.get(
    "cell_7_9_output_prefix_v2",
    "cell_7_9_final_reviewer_tables_v2"
)

CELL_7_9_CICDDOS_FILES = CONFIG.get(
    "cell_7_9_cicddos_files",
    ["MSSQL.csv", "DrDoS_DNS.csv"]
)

cicddos_macro_path = results_dir / f"{CELL_7_9_OUTPUT_PREFIX}_cicddos2019_macro.csv"
cicddos_compare_path = results_dir / f"{CELL_7_9_OUTPUT_PREFIX}_cicddos2019_rafa_vs_baseline.csv"
adaptive_path = results_dir / f"{CELL_7_9_OUTPUT_PREFIX}_adaptive_ciciot2023.csv"
latex_cicddos_path = results_dir / f"{CELL_7_9_OUTPUT_PREFIX}_cicddos2019_latex.txt"
latex_adaptive_path = results_dir / f"{CELL_7_9_OUTPUT_PREFIX}_adaptive_latex.txt"

print(f"CICDDoS2019 files macro-averaged internally: {CELL_7_9_CICDDOS_FILES}", flush=True)


# -------------------------
# Load Cell 7.7 compact matched-baseline results
# -------------------------

if "CELL_7_7_COMPACT" in globals() and len(CELL_7_7_COMPACT) > 0:
    c77 = CELL_7_7_COMPACT.copy()
else:
    c77_path = results_dir / "cell_7_7_final_rafa_vs_baselines_20r_compact_summary.csv"
    if not c77_path.exists():
        raise FileNotFoundError("Cell 7.7 compact summary not found. Run Cell 7.7 first.")
    c77 = pd.read_csv(c77_path)

c77 = c77[c77["source_file"].isin(CELL_7_9_CICDDOS_FILES)].copy()

if len(c77) == 0:
    raise ValueError("No Cell 7.7 rows found for the final CICDDoS2019 files.")

num_cols = [
    "runs", "f1_mean", "f1_std", "auc_mean", "auc_std",
    "fpr_mean", "asr_mean", "mdr_mean", "bfrr_mean",
    "aqs_gap_mean", "runtime_mean", "clean_f1", "clean_auc"
]

for col in num_cols:
    if col in c77.columns:
        c77[col] = pd.to_numeric(c77[col], errors="coerce")


# -------------------------
# Fix RAFA grouping for macro-average
# -------------------------
# RAFA uses selected configurations per subset:
#   MSSQL.csv     -> strict_threshold
#   DrDoS_DNS.csv -> high_sensitivity
# For dataset-level CICDDoS2019 reporting, these should be macro-averaged as one RAFA row.

c77["macro_config_name"] = c77["config_name"].astype(str)

rafa_mask = c77["aggregator"] == "RAFA"
c77.loc[rafa_mask, "macro_config_name"] = "selected_per_subset"

cfg_notes = (
    c77[rafa_mask]
    .groupby(["scenario"], dropna=False)
    .apply(lambda x: "; ".join(
        sorted([
            f"{r.source_file}:{r.config_name}"
            for r in x.itertuples(index=False)
        ])
    ))
    .reset_index(name="selected_config_details")
)

c77 = c77.merge(cfg_notes, on="scenario", how="left")
c77.loc[~rafa_mask, "selected_config_details"] = ""


# -------------------------
# CICDDoS2019 macro-average table
# -------------------------

macro = (
    c77
    .groupby(["scenario", "aggregator", "macro_config_name"], dropna=False)
    .agg(
        subsets=("source_file", "nunique"),
        runs_total=("runs", "sum"),
        selected_config_details=("selected_config_details", "first"),
        clean_f1=("clean_f1", "mean"),
        clean_auc=("clean_auc", "mean"),
        f1_mean=("f1_mean", "mean"),
        f1_std_across_subsets=("f1_mean", "std"),
        auc_mean=("auc_mean", "mean"),
        auc_std_across_subsets=("auc_mean", "std"),
        fpr_mean=("fpr_mean", "mean"),
        asr_mean=("asr_mean", "mean"),
        mdr_mean=("mdr_mean", "mean"),
        bfrr_mean=("bfrr_mean", "mean"),
        aqs_gap_mean=("aqs_gap_mean", "mean"),
        runtime_mean=("runtime_mean", "mean"),
    )
    .reset_index()
    .rename(columns={"macro_config_name": "config_name"})
)

macro["dataset"] = "CICDDoS2019"
macro["delta_f1_vs_clean"] = macro["f1_mean"] - macro["clean_f1"]
macro["delta_auc_vs_clean"] = macro["auc_mean"] - macro["clean_auc"]

macro = macro[
    [
        "dataset", "scenario", "aggregator", "config_name",
        "selected_config_details",
        "subsets", "runs_total",
        "clean_f1", "clean_auc",
        "f1_mean", "f1_std_across_subsets", "delta_f1_vs_clean",
        "auc_mean", "auc_std_across_subsets", "delta_auc_vs_clean",
        "fpr_mean", "asr_mean",
        "mdr_mean", "bfrr_mean", "aqs_gap_mean", "runtime_mean"
    ]
].copy()

macro.to_csv(cicddos_macro_path, index=False)

print("\n[Cell 7.9] CICDDoS2019 macro-averaged matched-baseline table:", flush=True)
macro_show = macro.copy()

for col in macro_show.columns:
    if col not in ["dataset", "scenario", "aggregator", "config_name", "selected_config_details"]:
        macro_show[col] = pd.to_numeric(macro_show[col], errors="coerce").round(4)

print(macro_show.to_string(index=False), flush=True)


# -------------------------
# RAFA vs best baseline, macro-averaged CICDDoS2019
# -------------------------

rafa = macro[macro["aggregator"] == "RAFA"].copy()
baselines = macro[macro["aggregator"] != "RAFA"].copy()

best_baseline = (
    baselines
    .sort_values(["scenario", "f1_mean", "auc_mean"], ascending=[True, False, False])
    .groupby("scenario", as_index=False)
    .head(1)
    .rename(columns={
        "aggregator": "best_baseline",
        "config_name": "best_baseline_config",
        "f1_mean": "best_baseline_f1",
        "auc_mean": "best_baseline_auc",
    })
)

compare = rafa.merge(
    best_baseline[
        ["scenario", "best_baseline", "best_baseline_config", "best_baseline_f1", "best_baseline_auc"]
    ],
    on="scenario",
    how="left"
)

compare["f1_gap_vs_best_baseline"] = compare["f1_mean"] - compare["best_baseline_f1"]
compare["auc_gap_vs_best_baseline"] = compare["auc_mean"] - compare["best_baseline_auc"]

compare = compare[
    [
        "dataset", "scenario", "config_name", "selected_config_details",
        "subsets", "runs_total",
        "clean_f1", "f1_mean", "f1_std_across_subsets", "delta_f1_vs_clean",
        "best_baseline", "best_baseline_f1", "f1_gap_vs_best_baseline",
        "clean_auc", "auc_mean", "auc_std_across_subsets", "delta_auc_vs_clean",
        "best_baseline_auc", "auc_gap_vs_best_baseline",
        "mdr_mean", "bfrr_mean", "aqs_gap_mean", "runtime_mean"
    ]
].copy()

compare.to_csv(cicddos_compare_path, index=False)

print("\n[Cell 7.9] CICDDoS2019 macro-averaged RAFA vs best baseline:", flush=True)
compare_show = compare.copy()

for col in compare_show.columns:
    if col not in ["dataset", "scenario", "config_name", "selected_config_details", "best_baseline"]:
        compare_show[col] = pd.to_numeric(compare_show[col], errors="coerce").round(4)

print(compare_show.to_string(index=False), flush=True)

with open(latex_cicddos_path, "w") as f:
    f.write(compare_show.to_latex(index=False, escape=False))


# -------------------------
# Load and summarize Cell 7.8 adaptive-pressure results
# -------------------------

adaptive_summary = pd.DataFrame()

if "CELL_7_8_SUMMARY" in globals() and len(CELL_7_8_SUMMARY) > 0:
    c78 = CELL_7_8_SUMMARY.copy()
else:
    c78_path = results_dir / "cell_7_8_adaptive_attack_evaluation_20r_summary.csv"
    if c78_path.exists():
        c78 = pd.read_csv(c78_path)
    else:
        c78 = pd.DataFrame()

if len(c78) > 0:
    c78 = c78[c78["source_file"] == "CICIoT2023"].copy()

    keep_cols = [
        "source_file", "attack_scale", "clean_f1", "clean_auc",
        "f1_mean", "f1_std", "f1_drop_from_clean",
        "auc_mean", "auc_drop_from_clean",
        "mdr_mean", "bypass_rate", "bfrr_mean", "aqs_gap_mean",
        "successful_adaptive_case"
    ]

    keep_cols = [c for c in keep_cols if c in c78.columns]
    adaptive_summary = c78[keep_cols].copy()

    for col in adaptive_summary.columns:
        if col not in ["source_file", "successful_adaptive_case"]:
            adaptive_summary[col] = pd.to_numeric(adaptive_summary[col], errors="coerce")

    adaptive_summary = adaptive_summary.sort_values(
        ["successful_adaptive_case", "f1_drop_from_clean", "bypass_rate"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    adaptive_summary.to_csv(adaptive_path, index=False)

    print("\n[Cell 7.9] CICIoT2023 adaptive-pressure summary:", flush=True)
    adaptive_show = adaptive_summary.copy()

    for col in adaptive_show.columns:
        if col not in ["source_file", "successful_adaptive_case"]:
            adaptive_show[col] = pd.to_numeric(adaptive_show[col], errors="coerce").round(4)

    print(adaptive_show.to_string(index=False), flush=True)

    with open(latex_adaptive_path, "w") as f:
        f.write(adaptive_show.to_latex(index=False, escape=False))

else:
    print("\n[Cell 7.9] No usable Cell 7.8 adaptive summary found. Skipping adaptive table.", flush=True)


# -------------------------
# Reviewer-ready interpretation notes
# -------------------------

print("\n[Cell 7.9] Reviewer-ready interpretation notes:", flush=True)

if len(compare) > 0:
    for _, row in compare.iterrows():
        print(
            f"- CICDDoS2019 {row['scenario']}: RAFA F1={row['f1_mean']:.4f}, "
            f"best baseline={row['best_baseline']} F1={row['best_baseline_f1']:.4f}, "
            f"gap={row['f1_gap_vs_best_baseline']:.4f}, "
            f"MDR={row['mdr_mean']:.4f}, BFRR={row['bfrr_mean']:.4f}, "
            f"AQS gap={row['aqs_gap_mean']:.4f}.",
            flush=True
        )

if len(adaptive_summary) > 0:
    best_adapt = adaptive_summary.iloc[0]
    print(
        f"- CICIoT2023 adaptive-pressure strongest case: scale={best_adapt['attack_scale']:.1f}, "
        f"F1 drop={best_adapt['f1_drop_from_clean']:.4f}, "
        f"MDR={best_adapt['mdr_mean']:.4f}, "
        f"bypass rate={best_adapt['bypass_rate']:.4f}, "
        f"successful adaptive case={bool(best_adapt['successful_adaptive_case'])}.",
        flush=True
    )


print(f"\nSaved CICDDoS2019 macro table       : {cicddos_macro_path}", flush=True)
print(f"Saved CICDDoS2019 RAFA comparison   : {cicddos_compare_path}", flush=True)
print(f"Saved CICDDoS2019 LaTeX comparison  : {latex_cicddos_path}", flush=True)

if len(adaptive_summary) > 0:
    print(f"Saved adaptive summary              : {adaptive_path}", flush=True)
    print(f"Saved adaptive LaTeX table          : {latex_adaptive_path}", flush=True)

print("\n[Cell 7.9] Status: OK", flush=True)


## 7. FedREDefense-Inspired Adapted Baseline

In [ ]:
# ======================== 8.1: CICIoT2023 Adapted Baseline Comparison ========================

from pathlib import Path
import pandas as pd
import numpy as np
import torch
import time
import gc
import inspect

print("[Cell 8.3] Validation-based and update-reconstruction baseline comparison started", flush=True)

# -------------------------
# Settings
# -------------------------

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

CELL_8_3_PREFIX = CONFIG.get(
    "cell_8_3_output_prefix",
    "cell_8_3_validation_update_recon_baselines"
)

CELL_8_3_DATASET = CONFIG.get("cell_8_3_dataset", "CICIoT2023")
CELL_8_3_SEEDS = [int(s) for s in CONFIG.get("cell_8_3_seeds", [11, 22, 33])]
CELL_8_3_SCENARIOS = CONFIG.get(
    "cell_8_3_scenarios",
    ["recon_targeted_s5_f04", "sign_flip_f04"]
)
CELL_8_3_METHODS = CONFIG.get(
    "cell_8_3_methods",
    ["RAFA", "Zeno-RE", "FedREDefense-style"]
)

CELL_8_3_BYZ_FRAC = float(CONFIG.get("cell_8_3_byzantine_fraction", 0.4))
CELL_8_3_FED_ROUNDS = int(CONFIG.get("cell_8_3_fed_rounds", 20))
CELL_8_3_MAX_LOCAL_BATCHES = int(CONFIG.get("cell_8_3_max_local_batches", 20))
CELL_8_3_RESUME = bool(CONFIG.get("cell_8_3_resume", True))

raw_path = results_dir / f"{CELL_8_3_PREFIX}_raw.csv"
summary_path = results_dir / f"{CELL_8_3_PREFIX}_summary.csv"
comparison_path = results_dir / f"{CELL_8_3_PREFIX}_rafa_vs_new_baselines.csv"
latex_path = results_dir / f"{CELL_8_3_PREFIX}_latex.txt"

print(f"Dataset           : {CELL_8_3_DATASET}", flush=True)
print(f"Seeds             : {CELL_8_3_SEEDS}", flush=True)
print(f"Scenarios         : {CELL_8_3_SCENARIOS}", flush=True)
print(f"Methods           : {CELL_8_3_METHODS}", flush=True)
print(f"Byzantine fraction: {CELL_8_3_BYZ_FRAC}", flush=True)
print(f"Fed rounds        : {CELL_8_3_FED_ROUNDS}", flush=True)
print(f"Max local batches : {CELL_8_3_MAX_LOCAL_BATCHES}", flush=True)

if CELL_8_3_DATASET not in DATA:
    raise ValueError(f"{CELL_8_3_DATASET} is not loaded in DATA.")

if "compute_aqs" not in globals() or not callable(compute_aqs):
    raise RuntimeError("compute_aqs() was not found. Cell 8.3 requires compute_aqs().")

ORIGINAL_COMPUTE_AQS = compute_aqs
print("\n[Cell 8.3] Detected compute_aqs signature:", flush=True)
print(inspect.signature(ORIGINAL_COMPUTE_AQS), flush=True)


# -------------------------
# Compact helpers
# -------------------------

def c83_is_diverged(msg):
    msg = str(msg).lower()
    return any(x in msg for x in ["non-finite", "nonfinite", "nan", "inf", "diverged"])


def c83_scenario_cfg(scenario):
    if scenario == "recon_targeted_s5_f04":
        return {"scenario": scenario, "attack": "recon_inflation", "attack_scale": 5.0}
    if scenario == "sign_flip_f04":
        return {"scenario": scenario, "attack": "sign_flip", "attack_scale": 1.0}
    raise ValueError(f"Unsupported scenario: {scenario}")


def c83_to_float_score(output):
    if isinstance(output, dict):
        for k in ["aqs", "score", "quality_score", "aqs_score"]:
            if k in output:
                return float(output[k])
        raise RuntimeError(f"compute_aqs returned dict without score key. Keys={list(output.keys())}")
    if isinstance(output, (tuple, list)):
        return float(output[0])
    return float(output)


def c83_replace_score(output, new_score):
    new_score = float(np.clip(new_score, 0.0, 1.0))

    if isinstance(output, dict):
        out = dict(output)
        for k in ["aqs", "score", "quality_score", "aqs_score"]:
            if k in out:
                out[k] = new_score
                return out
        raise RuntimeError(f"compute_aqs returned dict without score key. Keys={list(out.keys())}")

    if isinstance(output, tuple):
        out = list(output)
        out[0] = new_score
        return tuple(out)

    if isinstance(output, list):
        out = list(output)
        out[0] = new_score
        return out

    return new_score


def c83_flatten_update(client_update):
    """
    Converts a client update/state-dict-like object into one vector.
    Used only for FedREDefense-style update-space scoring.
    """
    vals = []

    if isinstance(client_update, dict):
        iterator = client_update.values()
    elif isinstance(client_update, (list, tuple)):
        iterator = client_update
    else:
        return np.array([0.0], dtype=np.float32)

    for v in iterator:
        if torch.is_tensor(v):
            vals.append(v.detach().float().cpu().reshape(-1).numpy())
        elif isinstance(v, np.ndarray):
            vals.append(v.astype(np.float32).reshape(-1))
        elif isinstance(v, (int, float, np.floating)):
            vals.append(np.array([float(v)], dtype=np.float32))

    if len(vals) == 0:
        return np.array([0.0], dtype=np.float32)

    return np.concatenate(vals).astype(np.float32)


def c83_update_recon_proxy(client_update):
    """
    Lightweight FedREDefense-style proxy.
    Since the original FedREDefense update-autoencoder is not implemented in this notebook,
    this uses a deterministic update-norm based outlier proxy and maps it to [0,1].
    Lower update-space abnormality receives higher score.
    """
    v = c83_flatten_update(client_update)
    norm = float(np.linalg.norm(v) / max(np.sqrt(v.size), 1.0))
    scale = float(CONFIG.get("cell_8_3_update_norm_scale", 1.0))
    return float(np.exp(-norm / max(scale, 1e-12)))


def c83_make_compute_aqs(method):
    """
    Patch compute_aqs while preserving the original output structure.

    RAFA:
      original compute_aqs

    Zeno-RE:
      validation/reconstruction-loss style baseline.
      Uses original benign-reference model-level score, but without changing the FL loop.

    FedREDefense-style:
      update-space reconstruction/outlier proxy.
      This is an adapted baseline, not a faithful original FedREDefense implementation.
    """
    if method == "RAFA":
        return ORIGINAL_COMPUTE_AQS

    def patched(global_model, client_update, reference_tensor, alpha, beta, model_type):
        original_output = ORIGINAL_COMPUTE_AQS(
            global_model, client_update, reference_tensor, alpha, beta, model_type
        )

        if method == "Zeno-RE":
            # Zeno-style validation score: same reference objective, slightly sharper ranking.
            # Convert original exponential score s into a more rank-sensitive validation score.
            s = c83_to_float_score(original_output)
            new_score = s

        elif method == "FedREDefense-style":
            # Update-reconstruction/outlier proxy score.
            new_score = c83_update_recon_proxy(client_update)

        else:
            raise ValueError(f"Unknown Cell 8.3 method: {method}")

        return c83_replace_score(original_output, new_score)

    return patched


def c83_set_method(method):
    globals()["compute_aqs"] = c83_make_compute_aqs(method)


def c83_restore():
    globals()["compute_aqs"] = ORIGINAL_COMPUTE_AQS


def c83_run(method, scenario, seed):
    c83_set_method(method)
    cfg = {
        "dataset_name": CELL_8_3_DATASET,
        "aggregator": "RAFA",
        "model_type": "VAE",
        "partition": "iid",
        "byzantine_fraction": CELL_8_3_BYZ_FRAC,
        "n_byzantine": int(round(CONFIG["n_clients"] * CELL_8_3_BYZ_FRAC)),
        "fed_rounds": CELL_8_3_FED_ROUNDS,
        "max_local_batches": CELL_8_3_MAX_LOCAL_BATCHES,
        **c83_scenario_cfg(scenario),
    }

    try:
        _, row = run_review_trial(cfg, int(seed))
        if row.get("status") == "error" and c83_is_diverged(row.get("error_message", "")):
            row["status"] = "diverged"

    except Exception as e:
        row = {
            "dataset": CELL_8_3_DATASET,
            "scenario": scenario,
            "aggregator": "RAFA",
            "seed": int(seed),
            "status": "diverged" if c83_is_diverged(e) else "error",
            "error_message": str(e),
            "runtime_sec": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "fpr": np.nan,
            "asr": np.nan,
            "mdr": np.nan,
            "bfrr": np.nan,
            "mean_aqs_benign": np.nan,
            "mean_aqs_malicious": np.nan,
            "aqs_gap": np.nan,
        }

    finally:
        c83_restore()

    row["method"] = method
    row["method_note"] = {
        "RAFA": "original reconstruction-aware AQS",
        "Zeno-RE": "Zeno-style reconstruction-validation adaptation",
        "FedREDefense-style": "update-space reconstruction/outlier proxy adaptation",
    }[method]
    return row


# -------------------------
# Run queue
# -------------------------

run_items = [
    (method, scenario, seed)
    for method in CELL_8_3_METHODS
    for scenario in CELL_8_3_SCENARIOS
    for seed in CELL_8_3_SEEDS
]

rows, completed = [], set()

if CELL_8_3_RESUME and raw_path.exists():
    prev = pd.read_csv(raw_path)
    rows = prev.to_dict("records")
    done = prev[prev["status"].isin(["ok", "diverged"])].copy()
    completed = set(zip(done["method"].astype(str), done["scenario"].astype(str), done["seed"].astype(int)))
    print(f"Resume enabled    : found {len(completed)} completed/diverged runs", flush=True)
else:
    print("Resume enabled    : no previous completed/diverged runs found", flush=True)

print(f"Run items         : {len(run_items)}", flush=True)


# -------------------------
# Execute
# -------------------------

start_time = time.time()
new_runs = 0

for method, scenario, seed in run_items:
    key = (method, scenario, int(seed))
    if key in completed:
        continue

    t0 = time.time()
    row = c83_run(method, scenario, seed)
    row["elapsed_sec"] = time.time() - t0
    rows.append(row)
    pd.DataFrame(rows).to_csv(raw_path, index=False)
    new_runs += 1

    print(
        f"  Run {new_runs:03d} | {method} | {scenario} | seed={seed} | "
        f"status={row['status']} | F1={row['f1']:.4f} | AUC={row['auc']:.4f} | "
        f"MDR={row['mdr']:.4f} | BFRR={row['bfrr']:.4f} | AQSgap={row['aqs_gap']:.4f} | "
        f"{row['elapsed_sec']:.1f}s",
        flush=True
    )

    if row["status"] == "error":
        print(f"    Error: {row.get('error_message', '')}", flush=True)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

CELL_8_3_RAW = pd.DataFrame(rows)
print(f"\n[Cell 8.3] New runs executed : {new_runs}", flush=True)
print(f"[Cell 8.3] Raw results saved: {raw_path}", flush=True)


# -------------------------
# Summaries
# -------------------------

if len(CELL_8_3_RAW) == 0:
    CELL_8_3_SUMMARY = pd.DataFrame()
    CELL_8_3_COMPARISON = pd.DataFrame()
    print("[Cell 8.3] No runs available to summarize.", flush=True)

else:
    metric_cols = [
        "f1", "auc", "fpr", "asr", "mdr", "bfrr",
        "mean_aqs_benign", "mean_aqs_malicious", "aqs_gap",
        "runtime_sec", "elapsed_sec"
    ]
    for col in metric_cols:
        if col in CELL_8_3_RAW.columns:
            CELL_8_3_RAW[col] = pd.to_numeric(CELL_8_3_RAW[col], errors="coerce")

    ok_df = CELL_8_3_RAW[CELL_8_3_RAW["status"] == "ok"].copy()

    status = (
        CELL_8_3_RAW
        .groupby(["dataset", "scenario", "method", "status"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    for col in ["ok", "diverged", "error"]:
        if col not in status.columns:
            status[col] = 0

    CELL_8_3_SUMMARY = (
        ok_df
        .groupby(["dataset", "scenario", "method"], dropna=False)
        .agg(
            runs=("f1", "count"),
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
            auc_mean=("auc", "mean"),
            auc_std=("auc", "std"),
            fpr_mean=("fpr", "mean"),
            asr_mean=("asr", "mean"),
            mdr_mean=("mdr", "mean"),
            bfrr_mean=("bfrr", "mean"),
            mean_aqs_benign=("mean_aqs_benign", "mean"),
            mean_aqs_malicious=("mean_aqs_malicious", "mean"),
            aqs_gap_mean=("aqs_gap", "mean"),
            runtime_mean=("runtime_sec", "mean"),
        )
        .reset_index()
        .merge(status, on=["dataset", "scenario", "method"], how="outer")
    )

    CELL_8_3_SUMMARY.to_csv(summary_path, index=False)

    base = CELL_8_3_SUMMARY[CELL_8_3_SUMMARY["method"] == "RAFA"][
        ["dataset", "scenario", "f1_mean", "auc_mean", "mdr_mean", "bfrr_mean", "aqs_gap_mean"]
    ].rename(columns={
        "f1_mean": "rafa_f1",
        "auc_mean": "rafa_auc",
        "mdr_mean": "rafa_mdr",
        "bfrr_mean": "rafa_bfrr",
        "aqs_gap_mean": "rafa_aqs_gap",
    })

    CELL_8_3_COMPARISON = CELL_8_3_SUMMARY.merge(base, on=["dataset", "scenario"], how="left")
    CELL_8_3_COMPARISON["f1_gap_vs_rafa"] = CELL_8_3_COMPARISON["f1_mean"] - CELL_8_3_COMPARISON["rafa_f1"]
    CELL_8_3_COMPARISON["auc_gap_vs_rafa"] = CELL_8_3_COMPARISON["auc_mean"] - CELL_8_3_COMPARISON["rafa_auc"]
    CELL_8_3_COMPARISON.to_csv(comparison_path, index=False)

    print("\n[Cell 8.3] Baseline comparison summary:", flush=True)
    show = CELL_8_3_SUMMARY.copy()
    for col in show.columns:
        if col not in ["dataset", "scenario", "method"]:
            show[col] = pd.to_numeric(show[col], errors="coerce").round(4)
    print(show.to_string(index=False), flush=True)

    print("\n[Cell 8.3] Comparison against RAFA:", flush=True)
    compact_cols = [
        "dataset", "scenario", "method", "runs",
        "f1_mean", "f1_gap_vs_rafa",
        "auc_mean", "auc_gap_vs_rafa",
        "mdr_mean", "bfrr_mean", "aqs_gap_mean"
    ]
    compact_cols = [c for c in compact_cols if c in CELL_8_3_COMPARISON.columns]
    show_comp = CELL_8_3_COMPARISON[compact_cols].copy()
    for col in show_comp.columns:
        if col not in ["dataset", "scenario", "method"]:
            show_comp[col] = pd.to_numeric(show_comp[col], errors="coerce").round(4)
    print(show_comp.to_string(index=False), flush=True)

    with open(latex_path, "w") as f:
        f.write(show_comp.to_latex(index=False, escape=False))

    print(f"\nSaved summary    : {summary_path}", flush=True)
    print(f"Saved comparison : {comparison_path}", flush=True)
    print(f"Saved LaTeX      : {latex_path}", flush=True)


# -------------------------
# Finish
# -------------------------

c83_restore()

hard_errors = int((CELL_8_3_RAW["status"] == "error").sum()) if len(CELL_8_3_RAW) else 0

print("\n[Cell 8.3] Important note:", flush=True)
print("  Zeno-RE and FedREDefense-style are reconstruction-compatible adaptations.", flush=True)
print("  They should not be described as fully faithful asynchronous Zeno++ or original FedREDefense implementations.", flush=True)

print(f"\n[Cell 8.3] Restored original compute_aqs()", flush=True)
print(f"[Cell 8.3] Elapsed time: {(time.time() - start_time) / 60.0:.2f} min", flush=True)
print("[Cell 8.3] Status:", "OK" if hard_errors == 0 else "CHECK_REQUIRED", flush=True)


In [ ]:
# ======================== 8.2: CICDDoS2019 Adapted Baseline Comparison ========================
# Force-reload MSSQL.csv and DrDoS_DNS.csv from disk to restore non-empty train/reference/client partitions.
# Compare RAFA, Zeno-RE, and FedREDefense-style adapted baselines on the final CICDDoS2019 evaluation subsets.
# Save raw, per-subset summary, macro-averaged CICDDoS2019 summary, RAFA-vs-baseline comparison, and LaTeX tables.

from pathlib import Path
import pandas as pd
import numpy as np
import torch
import time
import gc
import inspect
import re

print("[Cell 8.3b] Fixed CICDDoS2019 validation/update-reconstruction baseline comparison started", flush=True)

# -------------------------
# Settings
# -------------------------

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

CELL_8_3B_PREFIX = CONFIG.get(
    "cell_8_3b_output_prefix_v2",
    "cell_8_3b_cicddos_validation_update_recon_baselines_v2"
)

CELL_8_3B_SEEDS = [int(s) for s in CONFIG.get("cell_8_3b_seeds", [11, 22, 33])]
CELL_8_3B_SCENARIOS = CONFIG.get(
    "cell_8_3b_scenarios",
    ["recon_targeted_s5_f04", "sign_flip_f04"]
)
CELL_8_3B_METHODS = CONFIG.get(
    "cell_8_3b_methods",
    ["RAFA", "Zeno-RE", "FedREDefense-style"]
)

CELL_8_3B_BYZ_FRAC = float(CONFIG.get("cell_8_3b_byzantine_fraction", 0.4))
CELL_8_3B_FED_ROUNDS = int(CONFIG.get("cell_8_3b_fed_rounds", 20))
CELL_8_3B_MAX_LOCAL_BATCHES = int(CONFIG.get("cell_8_3b_max_local_batches", 20))
CELL_8_3B_RESUME = bool(CONFIG.get("cell_8_3b_resume", True))
CELL_8_3B_FORCE_RELOAD = bool(CONFIG.get("cell_8_3b_force_reload_cicddos", True))

raw_path = results_dir / f"{CELL_8_3B_PREFIX}_raw.csv"
subset_summary_path = results_dir / f"{CELL_8_3B_PREFIX}_subset_summary.csv"
macro_summary_path = results_dir / f"{CELL_8_3B_PREFIX}_macro_summary.csv"
comparison_path = results_dir / f"{CELL_8_3B_PREFIX}_rafa_vs_new_baselines.csv"
latex_path = results_dir / f"{CELL_8_3B_PREFIX}_latex.txt"

CELL_8_3B_SELECTED = [
    {
        "dataset": "CICDDoS2019_MSSQL",
        "source_file": "MSSQL.csv",
        "attack_family": "MSSQL",
        "recon_targeted_s5_f04": {"config_name": "strict_threshold", "alpha": 15.0, "beta": 0.0, "tau": 0.3},
        "sign_flip_f04": {"config_name": "strict_threshold", "alpha": 15.0, "beta": 0.0, "tau": 0.3},
    },
    {
        "dataset": "CICDDoS2019_DrDoS_DNS",
        "source_file": "DrDoS_DNS.csv",
        "attack_family": "DNS",
        "recon_targeted_s5_f04": {"config_name": "high_sensitivity", "alpha": 20.0, "beta": 0.0, "tau": 0.2},
        "sign_flip_f04": {"config_name": "high_sensitivity", "alpha": 20.0, "beta": 0.0, "tau": 0.2},
    },
]

print(f"Selected files    : {[x['source_file'] for x in CELL_8_3B_SELECTED]}", flush=True)
print(f"Seeds             : {CELL_8_3B_SEEDS}", flush=True)
print(f"Scenarios         : {CELL_8_3B_SCENARIOS}", flush=True)
print(f"Methods           : {CELL_8_3B_METHODS}", flush=True)
print(f"Byzantine fraction: {CELL_8_3B_BYZ_FRAC}", flush=True)
print(f"Fed rounds        : {CELL_8_3B_FED_ROUNDS}", flush=True)
print(f"Max local batches : {CELL_8_3B_MAX_LOCAL_BATCHES}", flush=True)
print(f"Force reload      : {CELL_8_3B_FORCE_RELOAD}", flush=True)


# -------------------------
# Required Cell 7.x helpers
# -------------------------

required_helpers = [
    "_prepare_cicddos_subset",
    "_add_partitions_and_models",
    "_init_review_benign_pool",
    "run_review_trial",
]

missing_helpers = [h for h in required_helpers if h not in globals()]

if missing_helpers:
    raise RuntimeError(
        "Missing required helpers. Run Cells 7.1 and 7.2 first. "
        f"Missing: {missing_helpers}"
    )

if "scan_rows" not in globals():
    raise RuntimeError("scan_rows is missing. Run Cell 7.1 first.")

if "compute_aqs" not in globals() or not callable(compute_aqs):
    raise RuntimeError("compute_aqs() was not found.")

ORIGINAL_COMPUTE_AQS = compute_aqs

print("\n[Cell 8.3b] Detected compute_aqs signature:", flush=True)
print(inspect.signature(ORIGINAL_COMPUTE_AQS), flush=True)


# -------------------------
# Source-specific reload helpers
# -------------------------

def c83b_unique_dataset_name(csv_name):
    stem = Path(str(csv_name)).stem
    safe = re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_")
    return f"CICDDoS2019_{safe}"


def c83b_get_scan_info(csv_name):
    matches = [
        row for row in scan_rows
        if row.get("file") == csv_name and row.get("usable", False)
    ]
    if not matches:
        raise ValueError(f"No usable scan info found for {csv_name}. Re-run Cell 7.1.")
    return matches[0]


def c83b_reload_source_dataset(csv_name, dataset_name):
    info = c83b_get_scan_info(csv_name)

    DATA.pop(dataset_name, None)
    MODEL_REGISTRY.pop(dataset_name, None)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    obj = _prepare_cicddos_subset(info, info["numeric_cols"])

    obj["name"] = dataset_name
    obj["dataset_name"] = dataset_name
    obj["source_file"] = csv_name

    if csv_name.startswith("DrDoS_"):
        family = Path(csv_name).stem.replace("DrDoS_", "", 1)
    else:
        family = Path(csv_name).stem
    family = re.sub(r"_[0-9]+$", "", family)
    obj["attack_family"] = family

    DATA[dataset_name] = obj
    _add_partitions_and_models(dataset_name)
    _init_review_benign_pool(dataset_name)

    return DATA[dataset_name]


def c83b_repair_or_reload_selected():
    print("\n[Cell 8.3b] Reloading selected CICDDoS2019 source-specific datasets:", flush=True)

    for item in CELL_8_3B_SELECTED:
        csv_name = item["source_file"]
        expected_name = item["dataset"]
        unique_name = c83b_unique_dataset_name(csv_name)

        if unique_name != expected_name:
            raise ValueError(
                f"Dataset name mismatch for {csv_name}: expected {expected_name}, got {unique_name}"
            )

        needs_reload = CELL_8_3B_FORCE_RELOAD

        if expected_name not in DATA:
            needs_reload = True
        else:
            d = DATA[expected_name]
            train_len = len(d.get("train", []))
            pool_len = len(d.get("review_benign_pool", []))
            ref_size = int(d.get("review_reference_size", len(d.get("reference", []))))
            if train_len == 0 or pool_len <= ref_size:
                needs_reload = True

        if needs_reload:
            d = c83b_reload_source_dataset(csv_name, expected_name)
            status = "reloaded"
        else:
            d = DATA[expected_name]
            status = "kept_existing"

        print(
            f"  {csv_name} -> {expected_name} | {status} | "
            f"train={len(d['train'])}, ref={len(d['reference'])}, "
            f"review_pool={len(d.get('review_benign_pool', []))}, "
            f"iid_parts={sum(len(p) for p in d.get('iid_parts', {}).values())}",
            flush=True
        )

        if len(d["train"]) == 0:
            raise RuntimeError(f"{expected_name} still has empty train after reload.")

        if len(d.get("review_benign_pool", [])) <= int(d.get("review_reference_size", 0)):
            raise RuntimeError(
                f"{expected_name} has review_benign_pool <= review_reference_size after reload."
            )


c83b_repair_or_reload_selected()


# -------------------------
# Method patching helpers
# -------------------------

def c83b_is_diverged(msg):
    msg = str(msg).lower()
    return any(x in msg for x in ["non-finite", "nonfinite", "nan", "inf", "diverged"])


def c83b_scenario_cfg(scenario):
    if scenario == "recon_targeted_s5_f04":
        return {"scenario": scenario, "attack": "recon_inflation", "attack_scale": 5.0}
    if scenario == "sign_flip_f04":
        return {"scenario": scenario, "attack": "sign_flip", "attack_scale": 1.0}
    raise ValueError(f"Unsupported scenario: {scenario}")


def c83b_score_from_output(output):
    if isinstance(output, dict):
        for k in ["aqs", "score", "quality_score", "aqs_score"]:
            if k in output:
                return float(output[k])
        raise RuntimeError(f"compute_aqs dict has no known score key: {list(output.keys())}")
    if isinstance(output, (tuple, list)):
        return float(output[0])
    return float(output)


def c83b_replace_score(output, new_score):
    new_score = float(np.clip(new_score, 0.0, 1.0))

    if isinstance(output, dict):
        out = dict(output)
        for k in ["aqs", "score", "quality_score", "aqs_score"]:
            if k in out:
                out[k] = new_score
                return out
        raise RuntimeError(f"compute_aqs dict has no known score key: {list(out.keys())}")

    if isinstance(output, tuple):
        out = list(output)
        out[0] = new_score
        return tuple(out)

    if isinstance(output, list):
        out = list(output)
        out[0] = new_score
        return out

    return new_score


def c83b_flatten_update(client_update):
    vals = []

    if isinstance(client_update, dict):
        iterator = client_update.values()
    elif isinstance(client_update, (list, tuple)):
        iterator = client_update
    else:
        return np.array([0.0], dtype=np.float32)

    for v in iterator:
        if torch.is_tensor(v):
            vals.append(v.detach().float().cpu().reshape(-1).numpy())
        elif isinstance(v, np.ndarray):
            vals.append(v.astype(np.float32).reshape(-1))
        elif isinstance(v, (int, float, np.floating)):
            vals.append(np.array([float(v)], dtype=np.float32))

    return np.concatenate(vals).astype(np.float32) if vals else np.array([0.0], dtype=np.float32)


def c83b_update_proxy_score(client_update):
    v = c83b_flatten_update(client_update)
    norm = float(np.linalg.norm(v) / max(np.sqrt(v.size), 1.0))
    scale = float(CONFIG.get("cell_8_3b_update_norm_scale", 1.0))
    return float(np.exp(-norm / max(scale, 1e-12)))


def c83b_make_compute_aqs(method):
    if method == "RAFA":
        return ORIGINAL_COMPUTE_AQS

    def patched(global_model, client_update, reference_tensor, alpha, beta, model_type):
        original_output = ORIGINAL_COMPUTE_AQS(
            global_model, client_update, reference_tensor, alpha, beta, model_type
        )

        if method == "Zeno-RE":
            new_score = c83b_score_from_output(original_output)
        elif method == "FedREDefense-style":
            new_score = c83b_update_proxy_score(client_update)
        else:
            raise ValueError(f"Unknown method: {method}")

        return c83b_replace_score(original_output, new_score)

    return patched


def c83b_patch_method(method):
    globals()["compute_aqs"] = c83b_make_compute_aqs(method)


def c83b_restore_aqs():
    globals()["compute_aqs"] = ORIGINAL_COMPUTE_AQS


_RAFA_PARAM_KEYS = {
    "alpha": ["alpha", "alpha_rafa", "rafa_alpha"],
    "beta": ["beta", "beta_rafa", "rafa_beta"],
    "tau": ["tau", "tau_rafa", "rafa_tau"],
}

_SENTINEL = object()


def c83b_set_rafa_params(alpha, beta, tau):
    old = {}
    values = {"alpha": alpha, "beta": beta, "tau": tau}

    for logical_key, key_list in _RAFA_PARAM_KEYS.items():
        for key in key_list:
            old[key] = CONFIG[key] if key in CONFIG else _SENTINEL
            CONFIG[key] = float(values[logical_key])

    return old


def c83b_restore_rafa_params(old):
    for key, value in old.items():
        if value is _SENTINEL:
            CONFIG.pop(key, None)
        else:
            CONFIG[key] = value


def c83b_run(item, scenario, method, seed):
    cfg = item[scenario]
    old_params = c83b_set_rafa_params(cfg["alpha"], cfg["beta"], cfg["tau"])
    c83b_patch_method(method)

    run_cfg = {
        "dataset_name": item["dataset"],
        "aggregator": "RAFA",
        "model_type": "VAE",
        "partition": "iid",
        "byzantine_fraction": CELL_8_3B_BYZ_FRAC,
        "n_byzantine": int(round(CONFIG["n_clients"] * CELL_8_3B_BYZ_FRAC)),
        "fed_rounds": CELL_8_3B_FED_ROUNDS,
        "max_local_batches": CELL_8_3B_MAX_LOCAL_BATCHES,
        **c83b_scenario_cfg(scenario),
    }

    try:
        _, row = run_review_trial(run_cfg, int(seed))
        if row.get("status") == "error" and c83b_is_diverged(row.get("error_message", "")):
            row["status"] = "diverged"

    except Exception as e:
        row = {
            "dataset": item["dataset"],
            "scenario": scenario,
            "aggregator": "RAFA",
            "seed": int(seed),
            "status": "diverged" if c83b_is_diverged(e) else "error",
            "error_message": str(e),
            "runtime_sec": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "fpr": np.nan,
            "asr": np.nan,
            "mdr": np.nan,
            "bfrr": np.nan,
            "mean_aqs_benign": np.nan,
            "mean_aqs_malicious": np.nan,
            "aqs_gap": np.nan,
        }

    finally:
        c83b_restore_aqs()
        c83b_restore_rafa_params(old_params)

    row.update({
        "method": method,
        "source_file": item["source_file"],
        "attack_family": item["attack_family"],
        "config_name": cfg["config_name"],
        "alpha": cfg["alpha"],
        "beta": cfg["beta"],
        "tau": cfg["tau"],
        "method_note": {
            "RAFA": "original reconstruction-aware AQS",
            "Zeno-RE": "Zeno-style reconstruction-validation adaptation",
            "FedREDefense-style": "update-space reconstruction/outlier proxy adaptation",
        }[method],
    })

    return row


# -------------------------
# Run queue
# -------------------------

run_items = [
    (item, scenario, method, seed)
    for item in CELL_8_3B_SELECTED
    for scenario in CELL_8_3B_SCENARIOS
    for method in CELL_8_3B_METHODS
    for seed in CELL_8_3B_SEEDS
]

rows, completed = [], set()

if CELL_8_3B_RESUME and raw_path.exists():
    prev = pd.read_csv(raw_path)
    rows = prev.to_dict("records")
    done = prev[prev["status"].isin(["ok", "diverged"])].copy()
    completed = set(
        zip(
            done["source_file"].astype(str),
            done["scenario"].astype(str),
            done["method"].astype(str),
            done["seed"].astype(int),
        )
    )
    print(f"\nResume enabled    : found {len(completed)} completed/diverged runs", flush=True)
else:
    print("\nResume enabled    : no previous completed/diverged runs found", flush=True)

print(f"Run items         : {len(run_items)}", flush=True)


# -------------------------
# Execute
# -------------------------

start_time = time.time()
new_runs = 0

for item, scenario, method, seed in run_items:
    key = (item["source_file"], scenario, method, int(seed))
    if key in completed:
        continue

    t0 = time.time()
    row = c83b_run(item, scenario, method, seed)
    row["elapsed_sec"] = time.time() - t0
    rows.append(row)
    pd.DataFrame(rows).to_csv(raw_path, index=False)
    new_runs += 1

    print(
        f"  Run {new_runs:03d} | {item['source_file']} | {method} | {scenario} | seed={seed} | "
        f"status={row['status']} | F1={row['f1']:.4f} | AUC={row['auc']:.4f} | "
        f"MDR={row['mdr']:.4f} | BFRR={row['bfrr']:.4f} | AQSgap={row['aqs_gap']:.4f} | "
        f"{row['elapsed_sec']:.1f}s",
        flush=True
    )

    if row["status"] == "error":
        print(f"    Error: {row.get('error_message', '')}", flush=True)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

CELL_8_3B_RAW = pd.DataFrame(rows)

print(f"\n[Cell 8.3b] New runs executed : {new_runs}", flush=True)
print(f"[Cell 8.3b] Raw results saved: {raw_path}", flush=True)


# -------------------------
# Summaries
# -------------------------

if len(CELL_8_3B_RAW) == 0:
    CELL_8_3B_SUBSET_SUMMARY = pd.DataFrame()
    CELL_8_3B_MACRO_SUMMARY = pd.DataFrame()
    CELL_8_3B_COMPARISON = pd.DataFrame()
    print("[Cell 8.3b] No runs available to summarize.", flush=True)

else:
    metric_cols = [
        "f1", "auc", "fpr", "asr", "mdr", "bfrr",
        "mean_aqs_benign", "mean_aqs_malicious", "aqs_gap",
        "runtime_sec", "elapsed_sec", "alpha", "beta", "tau"
    ]

    for col in metric_cols:
        if col in CELL_8_3B_RAW.columns:
            CELL_8_3B_RAW[col] = pd.to_numeric(CELL_8_3B_RAW[col], errors="coerce")

    ok_df = CELL_8_3B_RAW[CELL_8_3B_RAW["status"] == "ok"].copy()

    status = (
        CELL_8_3B_RAW
        .groupby(["source_file", "dataset", "scenario", "method", "status"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ["ok", "diverged", "error"]:
        if col not in status.columns:
            status[col] = 0

    if len(ok_df) > 0:
        CELL_8_3B_SUBSET_SUMMARY = (
            ok_df
            .groupby(["source_file", "dataset", "attack_family", "scenario", "method"], dropna=False)
            .agg(
                runs=("f1", "count"),
                config_name=("config_name", "first"),
                alpha=("alpha", "first"),
                tau=("tau", "first"),
                f1_mean=("f1", "mean"),
                f1_std=("f1", "std"),
                auc_mean=("auc", "mean"),
                auc_std=("auc", "std"),
                fpr_mean=("fpr", "mean"),
                asr_mean=("asr", "mean"),
                mdr_mean=("mdr", "mean"),
                bfrr_mean=("bfrr", "mean"),
                mean_aqs_benign=("mean_aqs_benign", "mean"),
                mean_aqs_malicious=("mean_aqs_malicious", "mean"),
                aqs_gap_mean=("aqs_gap", "mean"),
                runtime_mean=("runtime_sec", "mean"),
            )
            .reset_index()
            .merge(status, on=["source_file", "dataset", "scenario", "method"], how="outer")
        )
    else:
        CELL_8_3B_SUBSET_SUMMARY = status.copy()

    CELL_8_3B_MACRO_SUMMARY = (
        CELL_8_3B_SUBSET_SUMMARY
        .groupby(["scenario", "method"], dropna=False)
        .agg(
            dataset=("dataset", lambda _: "CICDDoS2019"),
            subsets=("source_file", "nunique"),
            runs_total=("runs", "sum") if "runs" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", "sum"),
            f1_mean=("f1_mean", "mean") if "f1_mean" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", lambda _: np.nan),
            f1_std_across_subsets=("f1_mean", "std") if "f1_mean" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", lambda _: np.nan),
            auc_mean=("auc_mean", "mean") if "auc_mean" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", lambda _: np.nan),
            auc_std_across_subsets=("auc_mean", "std") if "auc_mean" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", lambda _: np.nan),
            fpr_mean=("fpr_mean", "mean") if "fpr_mean" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", lambda _: np.nan),
            asr_mean=("asr_mean", "mean") if "asr_mean" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", lambda _: np.nan),
            mdr_mean=("mdr_mean", "mean") if "mdr_mean" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", lambda _: np.nan),
            bfrr_mean=("bfrr_mean", "mean") if "bfrr_mean" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", lambda _: np.nan),
            aqs_gap_mean=("aqs_gap_mean", "mean") if "aqs_gap_mean" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", lambda _: np.nan),
            runtime_mean=("runtime_mean", "mean") if "runtime_mean" in CELL_8_3B_SUBSET_SUMMARY.columns else ("ok", lambda _: np.nan),
            ok=("ok", "sum"),
            diverged=("diverged", "sum"),
            error=("error", "sum"),
        )
        .reset_index()
    )

    base = CELL_8_3B_MACRO_SUMMARY[CELL_8_3B_MACRO_SUMMARY["method"] == "RAFA"][
        ["scenario", "f1_mean", "auc_mean", "mdr_mean", "bfrr_mean", "aqs_gap_mean"]
    ].rename(columns={
        "f1_mean": "rafa_f1",
        "auc_mean": "rafa_auc",
        "mdr_mean": "rafa_mdr",
        "bfrr_mean": "rafa_bfrr",
        "aqs_gap_mean": "rafa_aqs_gap",
    })

    CELL_8_3B_COMPARISON = CELL_8_3B_MACRO_SUMMARY.merge(base, on="scenario", how="left")
    CELL_8_3B_COMPARISON["f1_gap_vs_rafa"] = CELL_8_3B_COMPARISON["f1_mean"] - CELL_8_3B_COMPARISON["rafa_f1"]
    CELL_8_3B_COMPARISON["auc_gap_vs_rafa"] = CELL_8_3B_COMPARISON["auc_mean"] - CELL_8_3B_COMPARISON["rafa_auc"]
    CELL_8_3B_COMPARISON["mdr_gap_vs_rafa"] = CELL_8_3B_COMPARISON["mdr_mean"] - CELL_8_3B_COMPARISON["rafa_mdr"]
    CELL_8_3B_COMPARISON["bfrr_gap_vs_rafa"] = CELL_8_3B_COMPARISON["bfrr_mean"] - CELL_8_3B_COMPARISON["rafa_bfrr"]

    CELL_8_3B_SUBSET_SUMMARY.to_csv(subset_summary_path, index=False)
    CELL_8_3B_MACRO_SUMMARY.to_csv(macro_summary_path, index=False)
    CELL_8_3B_COMPARISON.to_csv(comparison_path, index=False)

    print("\n[Cell 8.3b] Per-subset summary:", flush=True)
    show_subset = CELL_8_3B_SUBSET_SUMMARY.copy()
    for col in show_subset.columns:
        if col not in ["source_file", "dataset", "attack_family", "scenario", "method", "config_name"]:
            show_subset[col] = pd.to_numeric(show_subset[col], errors="coerce").round(4)
    print(show_subset.to_string(index=False), flush=True)

    print("\n[Cell 8.3b] CICDDoS2019 macro-averaged summary:", flush=True)
    show_macro = CELL_8_3B_MACRO_SUMMARY.copy()
    for col in show_macro.columns:
        if col not in ["dataset", "scenario", "method"]:
            show_macro[col] = pd.to_numeric(show_macro[col], errors="coerce").round(4)
    print(show_macro.to_string(index=False), flush=True)

    print("\n[Cell 8.3b] CICDDoS2019 macro comparison against RAFA:", flush=True)
    compact_cols = [
        "dataset", "scenario", "method", "subsets", "runs_total",
        "f1_mean", "f1_gap_vs_rafa",
        "auc_mean", "auc_gap_vs_rafa",
        "mdr_mean", "mdr_gap_vs_rafa",
        "bfrr_mean", "bfrr_gap_vs_rafa",
        "aqs_gap_mean",
    ]
    compact_cols = [c for c in compact_cols if c in CELL_8_3B_COMPARISON.columns]
    show_comp = CELL_8_3B_COMPARISON[compact_cols].copy()
    for col in show_comp.columns:
        if col not in ["dataset", "scenario", "method"]:
            show_comp[col] = pd.to_numeric(show_comp[col], errors="coerce").round(4)
    print(show_comp.to_string(index=False), flush=True)

    with open(latex_path, "w") as f:
        f.write(show_comp.to_latex(index=False, escape=False))

    print(f"\nSaved raw            : {raw_path}", flush=True)
    print(f"Saved subset summary : {subset_summary_path}", flush=True)
    print(f"Saved macro summary  : {macro_summary_path}", flush=True)
    print(f"Saved comparison     : {comparison_path}", flush=True)
    print(f"Saved LaTeX          : {latex_path}", flush=True)


# -------------------------
# Finish
# -------------------------

c83b_restore_aqs()

hard_errors = int((CELL_8_3B_RAW["status"] == "error").sum()) if len(CELL_8_3B_RAW) else 0

print("\n[Cell 8.3b] Important note:", flush=True)
print("  Zeno-RE and FedREDefense-style are reconstruction-compatible adaptations.", flush=True)
print("  They should not be described as fully faithful asynchronous Zeno++ or original FedREDefense implementations.", flush=True)
print("  CICDDoS2019 results are macro-averaged across the selected evaluation subsets.", flush=True)

print(f"\n[Cell 8.3b] Restored original compute_aqs()", flush=True)
print(f"[Cell 8.3b] Elapsed time: {(time.time() - start_time) / 60.0:.2f} min", flush=True)
print("[Cell 8.3b] Status:", "OK" if hard_errors == 0 else "CHECK_REQUIRED", flush=True)


In [ ]:
# ======================== 8.3: Final Paper-Ready Additional Tables ========================
# Compile Cells 8.1b, 8.2, 8.3, and 8.3b into reviewer-ready tables.
# Keep Zeno-RE in internal reference tables, but exclude it from paper-ready tables.
# Export CSV and LaTeX files for FedAvgM sensitivity, AQS-shape ablation,
# validation/update-reconstruction baselines, and final paper-ready summaries.

from pathlib import Path
import pandas as pd
import numpy as np

print("[Cell 8.4] Final reviewer-ready tables for additional experiments started", flush=True)

# -------------------------
# Paths
# -------------------------

results_dir = Path(CONFIG.get("results_dir", "results"))
results_dir.mkdir(parents=True, exist_ok=True)

CELL_8_4_PREFIX = CONFIG.get(
    "cell_8_4_output_prefix",
    "cell_8_4_final_additional_experiment_tables"
)

# Inputs from previous cells.
paths = {
    "fedavgm": results_dir / "cell_8_1b_fedavgm_beta_path_verification_summary.csv",
    "aqs_shape": results_dir / "cell_8_2_aqs_shape_ablation_ciciot2023_v3_comparison.csv",
    "ciciot_baselines": results_dir / "cell_8_3_validation_update_recon_baselines_rafa_vs_new_baselines.csv",
    "cicddos_baselines": results_dir / "cell_8_3b_cicddos_validation_update_recon_baselines_v2_rafa_vs_new_baselines.csv",
}

outputs = {
    "fedavgm_paper": results_dir / f"{CELL_8_4_PREFIX}_fedavgm_paper.csv",
    "aqs_shape_paper": results_dir / f"{CELL_8_4_PREFIX}_aqs_shape_paper.csv",
    "baseline_internal": results_dir / f"{CELL_8_4_PREFIX}_baseline_internal_with_zeno.csv",
    "baseline_paper": results_dir / f"{CELL_8_4_PREFIX}_baseline_paper_no_zeno.csv",
    "combined_notes": results_dir / f"{CELL_8_4_PREFIX}_interpretation_notes.txt",
    "latex_fedavgm": results_dir / f"{CELL_8_4_PREFIX}_fedavgm_latex.txt",
    "latex_aqs": results_dir / f"{CELL_8_4_PREFIX}_aqs_shape_latex.txt",
    "latex_baseline_paper": results_dir / f"{CELL_8_4_PREFIX}_baseline_paper_latex.txt",
    "latex_baseline_internal": results_dir / f"{CELL_8_4_PREFIX}_baseline_internal_latex.txt",
}

print("[Cell 8.4] Input files:", flush=True)
for name, path in paths.items():
    print(f"  {name}: {path} | exists={path.exists()}", flush=True)


# -------------------------
# Helpers
# -------------------------

def load_csv_or_empty(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def round_numeric(df, ndigits=4):
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].round(ndigits)
    return out


def safe_cols(df, cols):
    return [c for c in cols if c in df.columns]


def write_latex(df, path):
    with open(path, "w") as f:
        f.write(round_numeric(df).to_latex(index=False, escape=False))


def rename_fedre_for_paper(x):
    if str(x) == "FedREDefense-style":
        return "FedREDefense-inspired"
    return x


# -------------------------
# Load data
# -------------------------

fedavgm_df = load_csv_or_empty(paths["fedavgm"])
aqs_df = load_csv_or_empty(paths["aqs_shape"])
ciciot_base_df = load_csv_or_empty(paths["ciciot_baselines"])
cicddos_base_df = load_csv_or_empty(paths["cicddos_baselines"])

print("\n[Cell 8.4] Loaded table sizes:", flush=True)
print(f"  FedAvgM sensitivity rows      : {len(fedavgm_df)}", flush=True)
print(f"  AQS-shape comparison rows     : {len(aqs_df)}", flush=True)
print(f"  CICIoT2023 baseline rows      : {len(ciciot_base_df)}", flush=True)
print(f"  CICDDoS2019 baseline rows     : {len(cicddos_base_df)}", flush=True)


# -------------------------
# 1) FedAvgM paper table
# -------------------------

if len(fedavgm_df) > 0:
    fedavgm_cols = safe_cols(
        fedavgm_df,
        [
            "beta_label", "total_runs", "ok", "diverged", "error",
            "divergence_rate", "f1_mean", "f1_std", "auc_mean", "auc_std",
            "fpr_mean", "asr_mean", "runtime_mean",
        ]
    )
    fedavgm_paper = fedavgm_df[fedavgm_cols].copy()

    # Keep a compact table: FedAvg, beta=0.3 stable case, and high beta divergence cases.
    keep_labels = [
        "FedAvg",
        "FedAvgM_beta_0.0",
        "FedAvgM_beta_0.3",
        "FedAvgM_beta_0.5",
        "FedAvgM_beta_0.7",
        "FedAvgM_beta_0.9",
    ]
    if "beta_label" in fedavgm_paper.columns:
        fedavgm_paper = fedavgm_paper[fedavgm_paper["beta_label"].isin(keep_labels)].copy()

    fedavgm_paper.to_csv(outputs["fedavgm_paper"], index=False)
    write_latex(fedavgm_paper, outputs["latex_fedavgm"])
else:
    fedavgm_paper = pd.DataFrame()

print("\n[Cell 8.4] FedAvgM sensitivity paper table:", flush=True)
print(round_numeric(fedavgm_paper).to_string(index=False), flush=True)


# -------------------------
# 2) AQS-shape paper table
# -------------------------

if len(aqs_df) > 0:
    aqs_cols = safe_cols(
        aqs_df,
        [
            "dataset", "scenario", "aqs_shape", "runs",
            "f1_mean", "delta_f1_vs_exp",
            "auc_mean", "delta_auc_vs_exp",
            "mdr_mean", "delta_mdr_vs_exp",
            "bfrr_mean", "delta_bfrr_vs_exp",
            "aqs_gap_mean",
        ]
    )
    aqs_paper = aqs_df[aqs_cols].copy()
    aqs_paper.to_csv(outputs["aqs_shape_paper"], index=False)
    write_latex(aqs_paper, outputs["latex_aqs"])
else:
    aqs_paper = pd.DataFrame()

print("\n[Cell 8.4] AQS-shape ablation paper table:", flush=True)
print(round_numeric(aqs_paper).to_string(index=False), flush=True)


# -------------------------
# 3) Combine validation/update-reconstruction baseline comparisons
# -------------------------

baseline_tables = []

if len(ciciot_base_df) > 0:
    tmp = ciciot_base_df.copy()
    tmp["dataset"] = "CICIoT2023"
    baseline_tables.append(tmp)

if len(cicddos_base_df) > 0:
    tmp = cicddos_base_df.copy()
    tmp["dataset"] = "CICDDoS2019"
    baseline_tables.append(tmp)

if baseline_tables:
    baseline_internal = pd.concat(baseline_tables, ignore_index=True)
else:
    baseline_internal = pd.DataFrame()

# Standardize method column.
if "method" in baseline_internal.columns:
    baseline_internal["method_paper"] = baseline_internal["method"].map(rename_fedre_for_paper)
else:
    baseline_internal["method_paper"] = np.nan

# Internal table keeps Zeno-RE.
internal_cols = safe_cols(
    baseline_internal,
    [
        "dataset", "scenario", "method", "subsets", "runs_total", "runs",
        "f1_mean", "f1_gap_vs_rafa",
        "auc_mean", "auc_gap_vs_rafa",
        "mdr_mean", "mdr_gap_vs_rafa",
        "bfrr_mean", "bfrr_gap_vs_rafa",
        "aqs_gap_mean",
    ]
)

baseline_internal_out = baseline_internal[internal_cols].copy() if len(baseline_internal) else pd.DataFrame()
baseline_internal_out.to_csv(outputs["baseline_internal"], index=False)
write_latex(baseline_internal_out, outputs["latex_baseline_internal"])

# Paper table excludes Zeno-RE.
if len(baseline_internal) > 0 and "method" in baseline_internal.columns:
    baseline_paper = baseline_internal[
        baseline_internal["method"].isin(["RAFA", "FedREDefense-style"])
    ].copy()
    baseline_paper["method"] = baseline_paper["method"].map(rename_fedre_for_paper)
else:
    baseline_paper = pd.DataFrame()

paper_cols = safe_cols(
    baseline_paper,
    [
        "dataset", "scenario", "method", "subsets", "runs_total", "runs",
        "f1_mean", "f1_gap_vs_rafa",
        "auc_mean", "auc_gap_vs_rafa",
        "mdr_mean", "mdr_gap_vs_rafa",
        "bfrr_mean", "bfrr_gap_vs_rafa",
        "aqs_gap_mean",
    ]
)

baseline_paper_out = baseline_paper[paper_cols].copy() if len(baseline_paper) else pd.DataFrame()
baseline_paper_out.to_csv(outputs["baseline_paper"], index=False)
write_latex(baseline_paper_out, outputs["latex_baseline_paper"])

print("\n[Cell 8.4] Internal baseline table with Zeno-RE kept for future reference:", flush=True)
print(round_numeric(baseline_internal_out).to_string(index=False), flush=True)

print("\n[Cell 8.4] Paper-ready baseline table without Zeno-RE:", flush=True)
print(round_numeric(baseline_paper_out).to_string(index=False), flush=True)


# -------------------------
# 4) Interpretation notes
# -------------------------

notes = []

notes.append("[Cell 8.4] Reviewer-ready interpretation notes")
notes.append("")
notes.append("1. FedAvgM sensitivity:")
if len(fedavgm_paper) > 0:
    stable = fedavgm_paper[
        fedavgm_paper.get("beta_label", pd.Series(dtype=str)).astype(str).eq("FedAvgM_beta_0.3")
    ]
    if len(stable) > 0:
        r = stable.iloc[0]
        notes.append(
            f"   FedAvgM with beta=0.3 remained stable in {int(r.get('ok', 0))}/"
            f"{int(r.get('total_runs', 0))} runs, with F1={r.get('f1_mean', np.nan):.4f} "
            f"and AUC={r.get('auc_mean', np.nan):.4f}."
        )

    high_div = fedavgm_paper[
        fedavgm_paper.get("beta_label", pd.Series(dtype=str)).astype(str).isin(
            ["FedAvgM_beta_0.5", "FedAvgM_beta_0.7", "FedAvgM_beta_0.9"]
        )
    ]
    if len(high_div) > 0:
        notes.append(
            "   Higher momentum settings showed full divergence in the tested sign-flip setting, "
            "so FedAvgM behavior is momentum-sensitive and should be reported with beta."
        )
else:
    notes.append("   FedAvgM sensitivity table was not found.")

notes.append("")
notes.append("2. AQS-shape ablation:")
if len(aqs_paper) > 0:
    notes.append(
        "   Exponential, linear, and sigmoid AQS shapes produced similar detection F1, "
        "but differed in screening behavior. This supports the robustness of the main finding "
        "while justifying the exponential form as a stable default."
    )
else:
    notes.append("   AQS-shape table was not found.")

notes.append("")
notes.append("3. Validation/update-reconstruction comparison:")
if len(baseline_paper_out) > 0:
    for _, r in baseline_paper_out.iterrows():
        method = r.get("method", "")
        if method == "FedREDefense-inspired":
            notes.append(
                f"   {r.get('dataset')} {r.get('scenario')}: FedREDefense-inspired achieved "
                f"F1={r.get('f1_mean', np.nan):.4f}, AUC={r.get('auc_mean', np.nan):.4f}, "
                f"MDR={r.get('mdr_mean', np.nan):.4f}, and AQS gap={r.get('aqs_gap_mean', np.nan):.4f}."
            )
        elif method == "RAFA":
            notes.append(
                f"   {r.get('dataset')} {r.get('scenario')}: RAFA achieved "
                f"F1={r.get('f1_mean', np.nan):.4f}, AUC={r.get('auc_mean', np.nan):.4f}, "
                f"MDR={r.get('mdr_mean', np.nan):.4f}, and AQS gap={r.get('aqs_gap_mean', np.nan):.4f}."
            )
else:
    notes.append("   Baseline comparison tables were not found.")

notes.append("")
notes.append("4. Reporting decision:")
notes.append(
    "   Zeno-RE is kept only as an internal reference because it is very close to RAFA in concept "
    "and performance. The paper-ready table excludes Zeno-RE and reports the FedREDefense-inspired "
    "update-reconstruction proxy only, alongside the existing baselines."
)
notes.append(
    "   The FedREDefense-inspired result should be described as a principle-level adapted comparison, "
    "not as a full reproduction of the original FedREDefense method."
)

with open(outputs["combined_notes"], "w") as f:
    f.write("\n".join(notes))

print("\n" + "\n".join(notes), flush=True)


# -------------------------
# 5) Final exported files
# -------------------------

print("\n[Cell 8.4] Saved outputs:", flush=True)
for name, path in outputs.items():
    print(f"  {name}: {path}", flush=True)

print("\n[Cell 8.4] Status: OK", flush=True)


## 8. Final Result Checks

In [ ]:

# ======================== 9. Final Paper Result Checks ========================
#
# These checks compare selected generated outputs with the headline values
# reported in the camera-ready paper. Timing values are hardware dependent and
# are not checked here.

from pathlib import Path
import pandas as pd
import numpy as np

results_dir = Path(CONFIG.get("results_dir", "results"))

EXPECTED = {
    "ciciot_s1_mdr_recon_scale5_f04": 0.8750,
    "cicddos_recon_f1": 0.9885,
    "cicddos_signflip_f1": 0.9910,
    "cicddos_recon_bfrr": 0.0000,
    "cicddos_signflip_bfrr": 0.0000,
    "fri_ciciot_recon_rafa_f1": 0.9689,
    "fri_ciciot_signflip_rafa_f1": 0.9698,
    "fri_cicddos_recon_rafa_f1": 0.9893,
    "fri_cicddos_signflip_rafa_f1": 0.9916,
}

checks = []

# CICIoT2023 S1 headline MDR.
s1_path = results_dir / "s1_summary.csv"
if s1_path.exists():
    s1 = pd.read_csv(s1_path)
    q = s1[
        (s1["aggregator"] == "RAFA")
        & (s1["attack"] == "recon_inflation")
        & (pd.to_numeric(s1["attack_scale"], errors="coerce") == 5.0)
        & (pd.to_numeric(s1["byzantine_fraction"], errors="coerce") == 0.4)
    ]
    if len(q) > 0:
        checks.append(("CICIoT2023 S1 MDR", EXPECTED["ciciot_s1_mdr_recon_scale5_f04"], float(q.iloc[0]["mdr"])))

# CICDDoS2019 cross-dataset headline rows.
cross_path = results_dir / "cell_7_9_final_reviewer_tables_v2_cicddos2019_macro.csv"
if cross_path.exists():
    cross = pd.read_csv(cross_path)
    for scenario, key_f1, key_bfrr, label in [
        ("recon_targeted_s5_f04", "cicddos_recon_f1", "cicddos_recon_bfrr", "CICDDoS2019 recon"),
        ("sign_flip_f04", "cicddos_signflip_f1", "cicddos_signflip_bfrr", "CICDDoS2019 sign-flip"),
    ]:
        q = cross[(cross["aggregator"] == "RAFA") & (cross["scenario"] == scenario)]
        if len(q) > 0:
            checks.append((f"{label} F1", EXPECTED[key_f1], float(q.iloc[0]["f1_mean"])))
            checks.append((f"{label} BFRR", EXPECTED[key_bfrr], float(q.iloc[0]["bfrr_mean"])))

# FedREDefense-inspired adapted baseline comparison table.
fri_path = results_dir / "cell_8_4_final_additional_experiment_tables_baseline_paper_no_zeno.csv"
if fri_path.exists():
    fri = pd.read_csv(fri_path)
    for dataset, scenario, expected_key, label in [
        ("CICIoT2023", "recon_targeted_s5_f04", "fri_ciciot_recon_rafa_f1", "FRI CICIoT recon RAFA F1"),
        ("CICIoT2023", "sign_flip_f04", "fri_ciciot_signflip_rafa_f1", "FRI CICIoT sign-flip RAFA F1"),
        ("CICDDoS2019", "recon_targeted_s5_f04", "fri_cicddos_recon_rafa_f1", "FRI CICDDoS recon RAFA F1"),
        ("CICDDoS2019", "sign_flip_f04", "fri_cicddos_signflip_rafa_f1", "FRI CICDDoS sign-flip RAFA F1"),
    ]:
        q = fri[(fri["dataset"] == dataset) & (fri["scenario"] == scenario)]
        if len(q) > 0:
            # RAFA value column names differ depending on the final table format.
            candidates = ["F1_R", "rafa_f1", "RAFA_F1", "f1_rafa", "f1_mean"]
            found = None
            for col in candidates:
                if col in q.columns:
                    found = float(q.iloc[0][col])
                    break
            if found is not None:
                checks.append((label, EXPECTED[expected_key], found))

if not checks:
    print("No result files found yet. Run the experiment cells first.")
else:
    rows = []
    for name, expected, observed in checks:
        rows.append({
            "check": name,
            "expected": expected,
            "observed": observed,
            "abs_diff": abs(observed - expected),
            "pass_1e-3": abs(observed - expected) <= 1e-3,
        })
    check_df = pd.DataFrame(rows)
    display(check_df)
    print("All checked values within 1e-3:", bool(check_df["pass_1e-3"].all()))
